# Rigorous Branch 1 Processing-Only Pipeline

This notebook audits existing processed sessions, recovers bad or missing sessions from raw extracts, classifies raw material as gold-standard or nonstandard, then runs the same CPU/GPU preprocessing split without model training.

_Patched: (1) every stage now checks GCS and skips already-processed work instead of overwriting it (override with the `FORCE_REUPLOAD_*` flags); (2) batch IDs are 1-based and consistent across all three stages; (3) depth sessions are recognized as gold-standard — depth is frame-aligned to the color timebase, so no separate depth_timestamps.csv is required; (4) gold output group renamed to `dataset5_compressed` to match the training notebook._

In [ ]:
# 1. Update the gcloud CLI (Google Cloud SDK) to ensure the latest 'gcloud storage' features are available
!echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | sudo tee -a /etc/apt/sources.list.d/google-cloud-sdk.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key --keyring /usr/share/keyrings/cloud.google.gpg add -
!sudo apt-get update && sudo apt-get install -y google-cloud-cli

!gcloud --version

## Authenticate and configure GCS / Colab paths

In [ ]:
# @title
import os, subprocess, pathlib
from google.colab import auth
import os, subprocess

auth.authenticate_user()

# Core GCS / Colab paths
PROJECT = "fluent-webbing-496616-u8"
BUCKET = "miamioh-resa-data"
GCS_MOUNT_POINT = "/content/gcs"
WORK_ROOT = "/content/work"
CODE_ROOT = "/content/work/code"
MOUNT_ROOT = pathlib.Path(GCS_MOUNT_POINT)
MOUNT_ROOT.mkdir(parents=True, exist_ok=True)

GCLOUD_PROCESS_COUNT = 4
GCLOUD_THREAD_COUNT = 16


# Install gcsfuse if missing.
if subprocess.run(["bash", "-lc", "command -v gcsfuse"], capture_output=True).returncode != 0:
    subprocess.run(["bash", "-lc", "export GCSFUSE_REPO=gcsfuse-$(lsb_release -c -s); echo deb https://packages.cloud.google.com/apt $GCSFUSE_REPO main | tee /etc/apt/sources.list.d/gcsfuse.list"], check=True)
    subprocess.run(["bash", "-lc", "curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | apt-key add -"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gcsfuse"], check=True)
subprocess.run(["gcloud", "components", "update", "--quiet"], check=False)
subprocess.run(["gcloud", "config", "set", "project", PROJECT], check=True)
subprocess.run(["gcloud", "config", "set", "storage/process_count", str(GCLOUD_PROCESS_COUNT)], check=True)
subprocess.run(["gcloud", "config", "set", "storage/thread_count", str(GCLOUD_THREAD_COUNT)], check=True)

subprocess.run(["gcsfuse", "--implicit-dirs", BUCKET, str(MOUNT_ROOT)], check=False)
print("Mounted path candidate:", MOUNT_ROOT / "CapstoneData")
print("Configured project:", PROJECT)
print("Configured bucket:", BUCKET)

In [ ]:
# @title Sync repository code from GCS
from pathlib import Path
import subprocess, shlex, os, time

CODE_SRC_URI = f"gs://{BUCKET}/CapstoneData/code_v2/RESA_mmWave"
Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)
Path(CODE_ROOT).mkdir(parents=True, exist_ok=True)

cmd = [
    "gcloud", "--quiet", "--verbosity=error", "storage", "rsync",
    CODE_SRC_URI, CODE_ROOT, "--recursive"
]
print("$", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
print("Synced code to:", CODE_ROOT)

In [ ]:
# @title Write fast radar tensor exporter
%%writefile /content/export_radar_tensors_fast.py
#!/usr/bin/env python3
"""
Generate {session}_radar_tensors.npz for Branch 1 processing sessions.

Fast/A100-Colab oriented changes relative to the original exporter:
  * session-level parallelism via --workers
  * early skip when {session}_radar_tensors.npz already exists
  * optional uncompressed NPZ output via --npz-compression stored
  * strip rd_cube from hybrid_rd frame sidecars by rewriting sidecars once,
    instead of re-exporting every sidecar from the radar .bin a second time
  * compact per-session summaries for notebook logs

The script is still CPU/I/O bound. The A100 does not help much here; with
64 GB system RAM, use roughly --workers 4 first, then try 6 if stable.
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

# RESA_CODE_ROOT is exported into the subprocess env by the driver notebook
# (PYTHONPATH already includes it too, via ENV["PYTHONPATH"] in cell 7's resolver).
_CODE_ROOT_ENV = os.environ.get("RESA_CODE_ROOT")
if _CODE_ROOT_ENV and _CODE_ROOT_ENV not in sys.path:
    sys.path.insert(0, _CODE_ROOT_ENV)

import numpy as np

from branch1.models.hybrid_rd.hybrid_rd_runtime import export_session_sidecars, RuntimeRDPatchConfig

CFG_PATH = Path(_CODE_ROOT_ENV or ".") / "config" / "profile_objdet.cfg"
DATA_ROOT = Path(os.environ.get("RADAR_TENSOR_PROCESSING_ROOT", "."))
SESSION_PREFIX = os.environ.get("RADAR_TENSOR_SESSION_PREFIX", "session_2026-05-11_")
FRAME_NUMBER_OFFSET = 0


@dataclass
class SessionResult:
    session: str
    status: str
    message: str = ""
    frames: int = 0
    tensor_path: str = ""
    elapsed_sec: float = 0.0
    stripped_sidecars: int = 0
    skipped_sidecars: int = 0


def _has_data_rows(csv_path: Path) -> bool:
    try:
        with csv_path.open("r", encoding="utf-8", errors="replace") as fh:
            return sum(1 for _ in fh) > 1
    except OSError:
        return False


def _get_rd_cube_from_sidecar(sidecar_path: Path) -> np.ndarray:
    with np.load(sidecar_path, allow_pickle=False) as d:
        if "rd_cube" not in d:
            raise KeyError(f"rd_cube not in {sidecar_path}")
        # Copy while file is open, so returned array is independent of mmap/zip handle.
        return np.asarray(d["rd_cube"], dtype=np.complex64).copy()


def _save_npz(path: Path, payload: dict[str, Any], *, compression: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp.npz")
    if compression == "stored":
        np.savez(tmp, **payload)
    elif compression == "compressed":
        np.savez_compressed(tmp, **payload)
    else:
        raise ValueError(f"Unknown compression mode: {compression}")
    os.replace(tmp, path)


def _manifest_frame_paths(session_dir: Path) -> tuple[Path, list[Path], dict[str, Any]]:
    manifest_path = session_dir / "hybrid_rd" / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"No hybrid_rd/manifest.json in {session_dir}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    frames = []
    for frame_info in manifest.get("frames", []):
        rel = frame_info.get("path")
        if not rel:
            continue
        frames.append(session_dir / "hybrid_rd" / rel)
    return manifest_path, frames, manifest


def build_radar_tensors_from_sidecars(
    session_dir: Path,
    *,
    compression: str,
    overwrite: bool = False,
) -> tuple[Path, int]:
    """Write {session}_radar_tensors.npz by reading rd_cube from sidecar frames."""
    out_path = session_dir / f"{session_dir.name}_radar_tensors.npz"
    if out_path.exists() and not overwrite:
        return out_path, -1

    manifest_path, _, manifest = _manifest_frame_paths(session_dir)
    if not manifest.get("include_rd_cube", False):
        raise RuntimeError(f"{manifest_path} has include_rd_cube=False; re-export sidecars first")

    tensors: dict[str, np.ndarray] = {}
    for frame_info in manifest.get("frames", []):
        try:
            frame_index = int(frame_info["frame_index"])
            frame_path = session_dir / "hybrid_rd" / frame_info["path"]
            tensors[f"rd_{frame_index}"] = _get_rd_cube_from_sidecar(frame_path)
        except (KeyError, FileNotFoundError) as exc:
            print(f"  [WARN] {session_dir.name} frame={frame_info.get('frame_index', '?')}: {exc}", flush=True)
            continue

    if not tensors:
        raise RuntimeError(f"No rd_cube tensors extracted for {session_dir.name}")

    _save_npz(out_path, tensors, compression=compression)
    return out_path, len(tensors)


def strip_rd_cube_from_sidecars_rewrite(
    session_dir: Path,
    *,
    compression: str = "compressed",
) -> tuple[int, int]:
    """Remove rd_cube from existing sidecar .npz files without reparsing the radar .bin.

    This replaces the original second export_session_sidecars(...include_rd_cube=False)
    pass, which is much slower because it reprocesses the radar binary.
    """
    manifest_path, frame_paths, manifest = _manifest_frame_paths(session_dir)
    stripped = 0
    skipped = 0
    for frame_path in frame_paths:
        if not frame_path.exists():
            skipped += 1
            continue
        with np.load(frame_path, allow_pickle=False) as payload:
            files = list(payload.files)
            if "rd_cube" not in files:
                skipped += 1
                continue
            keep = {k: np.asarray(payload[k]).copy() for k in files if k != "rd_cube"}
        _save_npz(frame_path, keep, compression=compression)
        stripped += 1

    # Keep manifest consistent with stripped sidecars.
    manifest["include_rd_cube"] = False
    tmp_manifest = manifest_path.with_name(manifest_path.name + ".tmp")
    tmp_manifest.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(tmp_manifest, manifest_path)
    return stripped, skipped


def _process_one_session(
    session_dir_raw: str,
    cfg_path_raw: str,
    label_csv_name: str,
    include_unlabeled: bool,
    keep_rd_cube_sidecars: bool,
    strip_mode: str,
    tensor_compression: str,
    sidecar_strip_compression: str,
    overwrite: bool,
    quiet: bool,
) -> dict[str, Any]:
    start = time.time()
    session_dir = Path(session_dir_raw)
    cfg_path = Path(cfg_path_raw)
    out_path = session_dir / f"{session_dir.name}_radar_tensors.npz"

    def done(status: str, message: str = "", frames: int = 0, stripped: int = 0, skipped: int = 0) -> dict[str, Any]:
        return asdict(SessionResult(
            session=session_dir.name,
            status=status,
            message=message,
            frames=int(frames),
            tensor_path=str(out_path),
            elapsed_sec=float(time.time() - start),
            stripped_sidecars=int(stripped),
            skipped_sidecars=int(skipped),
        ))

    try:
        if not include_unlabeled and not _has_data_rows(session_dir / label_csv_name):
            return done("skip", f"missing/empty {label_csv_name}")

        bin_path = session_dir / f"{session_dir.name}.bin"
        if not bin_path.exists() or bin_path.stat().st_size <= 0:
            return done("skip", "missing/empty radar .bin")

        if out_path.exists() and not overwrite:
            return done("exists", "tensor NPZ already exists; skipped sidecar export", frames=-1)

        rd_config = RuntimeRDPatchConfig(
            frame_number_offset=FRAME_NUMBER_OFFSET,
            sidecar_subdir="hybrid_rd",
            clutter_mode="off",
            log_scale=True,
            include_rd_cube=True,
        )

        if not quiet:
            print(f"[{session_dir.name}] exporting sidecars with rd_cube=True", flush=True)
        export_session_sidecars(
            session_dir,
            cfg_path=cfg_path,
            config=rd_config,
            overwrite=True,
        )

        tensor_path, n_frames = build_radar_tensors_from_sidecars(
            session_dir,
            compression=tensor_compression,
            overwrite=True,
        )

        stripped = 0
        skipped = 0
        if keep_rd_cube_sidecars or strip_mode == "keep":
            pass
        elif strip_mode == "rewrite":
            stripped, skipped = strip_rd_cube_from_sidecars_rewrite(
                session_dir,
                compression=sidecar_strip_compression,
            )
        elif strip_mode == "reexport":
            export_session_sidecars(
                session_dir,
                cfg_path=cfg_path,
                config=RuntimeRDPatchConfig(
                    frame_number_offset=FRAME_NUMBER_OFFSET,
                    sidecar_subdir="hybrid_rd",
                    clutter_mode="off",
                    log_scale=True,
                    include_rd_cube=False,
                ),
                overwrite=True,
            )
        else:
            raise ValueError(f"Unknown strip mode: {strip_mode}")

        return done("ok", f"wrote {tensor_path.name}", frames=n_frames, stripped=stripped, skipped=skipped)
    except Exception as exc:
        return done("error", f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")


def _select_sessions(args: argparse.Namespace) -> list[Path]:
    data_root = args.data_root.resolve()
    sessions = sorted(p for p in data_root.glob(f"{args.session_prefix}*") if p.is_dir())
    if args.max_sessions is not None:
        sessions = sessions[: int(args.max_sessions)]
    if not sessions:
        raise SystemExit(f"No sessions found at {data_root} with prefix {args.session_prefix}")
    if not args.include_unlabeled:
        labeled = [s for s in sessions if _has_data_rows(s / args.label_csv_name)]
        if not labeled:
            raise SystemExit(f"No labeled sessions found at {data_root} with prefix {args.session_prefix}")
        sessions = labeled
    return sessions


def _print_summary(results: list[dict[str, Any]]) -> None:
    counts: dict[str, int] = {}
    for row in results:
        counts[str(row.get("status", "unknown"))] = counts.get(str(row.get("status", "unknown")), 0) + 1
    print("\nSummary:", counts, flush=True)
    for row in results:
        status = row.get("status")
        session = row.get("session")
        frames = row.get("frames")
        sec = float(row.get("elapsed_sec", 0.0) or 0.0)
        msg = str(row.get("message", "")).splitlines()[0]
        print(f"[{status}] {session}: frames={frames} sec={sec:.1f} {msg}", flush=True)


def main() -> None:
    ap = argparse.ArgumentParser(
        description=(
            "Generate <session>_radar_tensors.npz for Branch 1 sessions from local "
            "hybrid_rd sidecars. By default, only sessions with non-empty labels are processed."
        )
    )
    ap.add_argument("--data-root", type=Path, default=DATA_ROOT)
    ap.add_argument("--session-prefix", default=SESSION_PREFIX)
    ap.add_argument("--cfg-path", type=Path, default=CFG_PATH)
    ap.add_argument("--label-csv-name", default="labeled_radar_points_v4.csv")
    ap.add_argument("--include-unlabeled", action="store_true", help="Also process sessions without a non-empty label CSV.")
    ap.add_argument("--keep-rd-cube-sidecars", action="store_true", help="Keep rd_cube arrays in hybrid_rd frame sidecars after building the tensor NPZ.")
    ap.add_argument("--workers", type=int, default=1, help="Number of sessions to process concurrently. With 64 GB RAM, try 4 first, then 6.")
    ap.add_argument("--max-sessions", type=int, default=None)
    ap.add_argument("--overwrite", action="store_true", help="Rebuild tensor NPZ even if it already exists.")
    ap.add_argument("--npz-compression", choices=["compressed", "stored"], default="compressed", help="Compression for {session}_radar_tensors.npz. 'stored' is much faster but larger.")
    ap.add_argument("--sidecar-strip-mode", choices=["rewrite", "reexport", "keep"], default="rewrite", help="How to remove temporary rd_cube arrays from hybrid_rd sidecars.")
    ap.add_argument("--sidecar-strip-compression", choices=["compressed", "stored"], default="compressed", help="Compression used when --sidecar-strip-mode=rewrite.")
    ap.add_argument("--quiet", action="store_true", help="Reduce per-session chatter from this wrapper.")
    args = ap.parse_args()

    data_root = args.data_root.resolve()
    sessions = _select_sessions(args)
    workers = max(1, int(args.workers))
    print(f"Found {len(sessions)} sessions at {data_root} with prefix {args.session_prefix}", flush=True)
    print(
        f"workers={workers} npz_compression={args.npz_compression} "
        f"sidecar_strip_mode={args.sidecar_strip_mode} sidecar_strip_compression={args.sidecar_strip_compression}",
        flush=True,
    )
    if workers > 6:
        print("WARNING: --workers > 6 may exceed 64 GB CPU RAM on large sessions.", flush=True)

    t0 = time.time()
    results: list[dict[str, Any]] = []
    if workers == 1:
        for session_dir in sessions:
            row = _process_one_session(
                str(session_dir),
                str(args.cfg_path),
                args.label_csv_name,
                bool(args.include_unlabeled),
                bool(args.keep_rd_cube_sidecars),
                args.sidecar_strip_mode,
                args.npz_compression,
                args.sidecar_strip_compression,
                bool(args.overwrite),
                bool(args.quiet),
            )
            results.append(row)
            print(f"[{row['status']}] {row['session']}: frames={row['frames']} sec={row['elapsed_sec']:.1f} {str(row['message']).splitlines()[0]}", flush=True)
    else:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            futures = {
                pool.submit(
                    _process_one_session,
                    str(session_dir),
                    str(args.cfg_path),
                    args.label_csv_name,
                    bool(args.include_unlabeled),
                    bool(args.keep_rd_cube_sidecars),
                    args.sidecar_strip_mode,
                    args.npz_compression,
                    args.sidecar_strip_compression,
                    bool(args.overwrite),
                    bool(args.quiet),
                ): session_dir.name
                for session_dir in sessions
            }
            for fut in as_completed(futures):
                row = fut.result()
                results.append(row)
                print(f"[{row['status']}] {row['session']}: frames={row['frames']} sec={row['elapsed_sec']:.1f} {str(row['message']).splitlines()[0]}", flush=True)

    results = sorted(results, key=lambda r: str(r.get("session", "")))
    summary_path = data_root / "radar_tensor_export_summary.json"
    summary_path.write_text(json.dumps(results, indent=2, sort_keys=True), encoding="utf-8")
    _print_summary(results)
    errors = [r for r in results if r.get("status") == "error"]
    print(f"\nDone in {time.time() - t0:.1f}s. Summary: {summary_path}", flush=True)
    if errors:
        raise SystemExit(f"{len(errors)} session(s) failed during radar tensor export.")


if __name__ == "__main__":
    main()

## Define preprocessing engine

This cell is adapted from the existing Branch 1 preprocessing setup, but the record source is overridden by the audit manifest and no training is performed.

In [ ]:
# @title defs: processing engine: CPU/GPU split preprocessing helpers only
# Defines the reusable CPU/GPU preprocessing stages; this notebook overrides record selection from an audit manifest.
#
# What this does:
#   0. Skips corrupt/incomplete sessions by default and records why they were skipped.
#   1. Pulls expanded raw session folders from GCS raw_session_extracts in bounded batches.
#   2. Runs repo preprocessing hooks: prepare_autolabel, OneFormer labelmaps,
#      OneFormer radar autolabeling, depth correspondence, radar tensor export.
#   3. Runs ghost/directness + depth-correspondence fusion.
#   4. Builds points_with_rd_patch_index.csv with RD + true-RA patches.
#   5. Writes train/internal-val plus separate external-holdout datasets to GCS.
#
# The cell intentionally fails early if the synced repo lacks a required tool.
# In particular, true-RA training requires real range_bin/doppler_bin provenance
# and a complex rd_cube source, not reconstructed bins.
#
# This cell assumes GCS/gcloud has already been initialized once by the notebook setup
# cells. It does not call `gcloud config set` or mutate runtime gcloud settings.

from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Iterable

# Notebook subprocesses receive ENV["PYTHONPATH"] below, but this kernel needs
# the synced repository root on sys.path before importing shared helpers.
_selector_code_root = Path(str(globals().get("CODE_ROOT", "/content/work/code")))
if str(_selector_code_root) not in sys.path:
    sys.path.insert(0, str(_selector_code_root))
if not (_selector_code_root / "tools" / "session_selector.py").exists():
    raise FileNotFoundError(
        f"Missing {_selector_code_root / 'tools' / 'session_selector.py'}. "
        "Sync the current RESA_mmWave tree to CapstoneData/code_v2/RESA_mmWave before running NB1."
    )
from tools.session_selector import admit_session, load_catalog, select_sessions, sync_catalog_from_gcs, sync_catalog_to_gcs

# -----------------------------
# User-tunable configuration
# -----------------------------

# Raw-session intake is manifest-driven. Sessions remain discoverable only through the
# live catalog instead of notebook-local dataset group lists.
RAW_SESSION_PREFIX = str(globals().get("RAW_SESSION_PREFIX", "session_"))
RAW_LABEL_STATUS_INCLUDE = tuple(globals().get("RAW_LABEL_STATUS_INCLUDE", ("raw",)))
RAW_DATASET_IDS = tuple(globals().get("RAW_DATASET_IDS", ()))
# Optional exact allowlist for a bounded smoke test; combine with RAW_DATASET_IDS
# when session IDs might not be globally unique across legacy datasets.
RAW_SESSION_IDS = tuple(globals().get("RAW_SESSION_IDS", ()))
RAW_SESSIONS_SUBDIR = "raw/sessions"
INTERMEDIATE_ONEFORMER_INPUT_SUBDIR = "intermediate/oneformer/inputs"
INTERMEDIATE_ONEFORMER_OUTPUT_SUBDIR = "intermediate/oneformer/outputs"
INTERMEDIATE_AUDIT_SUBDIR = "intermediate/audits"
CURATED_SESSIONS_SUBDIR = "curated/sessions"
CURATED_MANIFEST_SUBDIR = "curated/manifests"
FULL_DATASET_SUBDIR = "colab_outputs/rigorous_preprocessing_unused_final_dataset/rdra_patch_dataset"
MIN_AUTOLABEL_COVERAGE = float(globals().get("MIN_AUTOLABEL_COVERAGE", 0.90))

# Local SSD controls.
RAW_PREPROCESS_BATCH_SIZE = 10      # number of raw session folders staged at a time
SESSION_DOWNLOAD_BATCH_SIZE = int(globals().get("SESSION_DOWNLOAD_BATCH_SIZE", 4))

# # OneFormer / autolabel controls.
# ONEFORMER_MODEL = globals().get("ONEFORMER_MODEL", "tiny")
# ONEFORMER_EVERY_N = int(globals().get("ONEFORMER_EVERY_N", 1))
# # Batch OneFormer export loads the model once per local raw-preprocess batch/group,
# # then writes the usual per-session seg/*.npy outputs.
# ONEFORMER_MAX_W = int(globals().get("ONEFORMER_MAX_W", 640))
# ONEFORMER_SAVE_PNG = bool(globals().get("ONEFORMER_SAVE_PNG", False))
ONEFORMER_MODEL = globals().get("ONEFORMER_MODEL", "large")
ONEFORMER_MAX_W = int(globals().get("ONEFORMER_MAX_W", 1280))
ONEFORMER_EVERY_N = int(globals().get("ONEFORMER_EVERY_N", 1))
ONEFORMER_SAVE_PNG = bool(globals().get("ONEFORMER_SAVE_PNG", False))
# A100: start at 4 for large/1280, try 8 if stable. Tiny can usually use 8-16.
ONEFORMER_FRAME_BATCH_SIZE = int(globals().get("ONEFORMER_FRAME_BATCH_SIZE", 4 if str(ONEFORMER_MODEL).lower() == "large" else 8))
ONEFORMER_USE_FP16 = bool(globals().get("ONEFORMER_USE_FP16", True))
ONEFORMER_TIMING = bool(globals().get("ONEFORMER_TIMING", True))
MAX_TIME_DIFF_MS = "35"
DEPTH_OCCLUSION_M = "0.20"
DEPTH_PATCH_R = "2"



# Dataset split controls.
# The canonical training CSV keeps only split={train,val}. The final chronological
# session(s) per raw group are written to a separate external holdout CSV and are
# excluded from training/early-stopping.
INTERNAL_VAL_SESSIONS_PER_GROUP = int(globals().get("INTERNAL_VAL_SESSIONS_PER_GROUP", globals().get("VAL_SESSIONS_PER_GROUP", 2)))
INTERNAL_VAL_FRACTION_IF_SMALL = float(globals().get("INTERNAL_VAL_FRACTION_IF_SMALL", globals().get("VAL_FRACTION_IF_SMALL", 0.15)))
EXTERNAL_HOLDOUT_SESSIONS_PER_GROUP = int(globals().get("EXTERNAL_HOLDOUT_SESSIONS_PER_GROUP", 1))
EXTERNAL_HOLDOUT_FRACTION_IF_SMALL = float(globals().get("EXTERNAL_HOLDOUT_FRACTION_IF_SMALL", 0.10))
MIN_TRAINVAL_SESSIONS_PER_GROUP_AFTER_EXTERNAL = int(globals().get("MIN_TRAINVAL_SESSIONS_PER_GROUP_AFTER_EXTERNAL", 2))
WRITE_ALL_WITH_EXTERNAL_HOLDOUT_CSV = bool(globals().get("WRITE_ALL_WITH_EXTERNAL_HOLDOUT_CSV", True))

# Output behavior.
FORCE_REBUILD_FULL_DATASET = False

# Idempotency / re-run controls (added).
# When False (default), each staged upload first checks GCS and SKIPS work whose
# outputs already exist, instead of overwriting previously processed content.
# Set the matching flag True to force a clean rebuild of that stage.
FORCE_REUPLOAD_ONEFORMER_INPUTS = bool(globals().get("FORCE_REUPLOAD_ONEFORMER_INPUTS", False))
FORCE_REUPLOAD_ONEFORMER_OUTPUTS = bool(globals().get("FORCE_REUPLOAD_ONEFORMER_OUTPUTS", False))
FORCE_REUPLOAD_PROCESSED_SESSIONS = bool(globals().get("FORCE_REUPLOAD_PROCESSED_SESSIONS", False))
# Set True once after ranking changes to force the audit to re-run;
# set back to False on subsequent runs to reuse the cached staged_sessions.csv.
FORCE_REAUDIT = bool(globals().get("FORCE_REAUDIT", False))
COPY_GHOST_ASSETS = False               # if ghost_teacher_fusion supports it; normally keep false
DRY_RUN = False
STRICT_SCHEMA_ALIGNMENT = True
# Corrupt/incomplete raw sessions are common in long DCA/RealSense captures.
# Keep this True for dataset builds; set False only when debugging a specific session.
SKIP_CORRUPT_SESSIONS = bool(globals().get("SKIP_CORRUPT_SESSIONS", True))
MIN_VALID_SESSIONS_PER_GROUP_AFTER_SKIP = int(globals().get("MIN_VALID_SESSIONS_PER_GROUP_AFTER_SKIP", 1))

# Keep the heavy/root-wide operations batchwise whenever the underlying tool supports it.
# Per-session loops remain only where the underlying API is inherently per-session
# (native ADC decode and corrupt-session filtering), but the model/tool-loading stages
# are batched at the group root level.
BATCHWISE_ONEFORMER = bool(globals().get("BATCHWISE_ONEFORMER", True))
BATCHWISE_V3_AUTOLABEL = bool(globals().get("BATCHWISE_V3_AUTOLABEL", True))
AUDIT_DIRECTNESS_AND_GHOST_FEATURES = bool(globals().get("AUDIT_DIRECTNESS_AND_GHOST_FEATURES", True))
AUDIT_RA_TENSORS_AND_PATCHES = bool(globals().get("AUDIT_RA_TENSORS_AND_PATCHES", True))
RA_TENSOR_AUDIT_MAX_ARRAYS = int(globals().get("RA_TENSOR_AUDIT_MAX_ARRAYS", 4))

# Radar tensor export can dominate CPU Stage 3. The fast exporter parallelizes
# across sessions and avoids a second full radar-bin reparse when stripping
# temporary rd_cube sidecars. With 64 GB CPU RAM, start at 4 workers and try 6
# if the session sizes are moderate.
RADAR_TENSOR_WORKERS = int(globals().get("RADAR_TENSOR_WORKERS", 4))
RADAR_TENSOR_NPZ_COMPRESSION = str(globals().get("RADAR_TENSOR_NPZ_COMPRESSION", "stored"))  # stored=fast/larger, compressed=smaller/slower
RADAR_TENSOR_SIDECAR_STRIP_MODE = str(globals().get("RADAR_TENSOR_SIDECAR_STRIP_MODE", "rewrite"))  # rewrite avoids second bin parse
RADAR_TENSOR_SIDECAR_STRIP_COMPRESSION = str(globals().get("RADAR_TENSOR_SIDECAR_STRIP_COMPRESSION", "compressed"))
RADAR_TENSOR_OVERWRITE = bool(globals().get("RADAR_TENSOR_OVERWRITE", False))
RADAR_TENSOR_QUIET = bool(globals().get("RADAR_TENSOR_QUIET", True))


# adc_to_pointcloud_v6 writes radar_frame_num = raw_sidecar_frame_index + 1 by default.
# dataset_tools.py build-patches expects this value as "CSV frame number minus sidecar frame_index".
# Keep this at 1 unless you explicitly generated CSVs with ADC_FRAME_NUMBER_OFFSET=0.
RD_PATCH_FRAME_NUMBER_OFFSET = int(globals().get("RD_PATCH_FRAME_NUMBER_OFFSET", 1))
ENRICH_LABEL_CSV_IN_PLACE = True

# Per-batch point-dataset builder needs at least one held-out session because
# the current dataset_tools.py implementation does not handle holdout_last=0
# correctly (sessions[:-0] is empty). Final train/val splits are reassigned
# after all batches are combined, so this per-batch holdout is temporary.
PER_BATCH_HOLDOUT_LAST = int(globals().get("PER_BATCH_HOLDOUT_LAST", 1))

# Set these only for controlled no-human datasets. They are passed directly to
# branch1_fourclass_tools.py and are never guessed from a date.
NO_HUMAN_SESSION_PREFIXES = list(globals().get("NO_HUMAN_SESSION_PREFIXES", []))
NO_HUMAN_VAL_SESSIONS = set(globals().get("NO_HUMAN_VAL_SESSIONS", []))

STRUCTURAL_TEMPORAL_RADIUS_FRAMES = 2
PERSIST_TEMPORAL_RADIUS_FRAMES = 5
PERSIST_RADIUS_M = 0.30
LOCAL_DENSITY_RADIUS_M = 0.50

# -----------------------------
# Environment/path resolution
# -----------------------------

def _global_first(*names, default=None):
    for name in names:
        if name in globals() and globals()[name] not in (None, ""):
            return globals()[name]
    return default

BUCKET_NAME = _global_first("BUCKET_NAME", "BUCKET", default="miamioh-resa-data")
GCS_MOUNT_POINT = str(_global_first("GCS_MOUNT_POINT", default="/content/gcs"))
WORK_ROOT = Path(str(_global_first("WORK_ROOT", default="/content/work")))
CODE_ROOT = Path(str(_global_first("CODE_ROOT", default=WORK_ROOT / "code")))
PY = Path(sys.executable)

GCS_ROOT_URI = f"gs://{BUCKET_NAME}/CapstoneData"
RAW_EXTRACTS_ROOT_URI = f"{GCS_ROOT_URI}/{RAW_SESSIONS_SUBDIR}"
GCS_CATALOG_URI = f"{GCS_ROOT_URI}/{CURATED_MANIFEST_SUBDIR}/session_catalog.csv"
CURATED_SESSION_ROOT_URI = f"{GCS_ROOT_URI}/{CURATED_SESSIONS_SUBDIR}"
FULL_DATASET_ROOT_URI = f"{GCS_ROOT_URI}/{FULL_DATASET_SUBDIR}"
CATALOG_LOCAL_PATH = WORK_ROOT / "_session_catalog" / "session_catalog.csv"

# Resolve code layout against the refactored RESA_mmWave tree synced from
# CapstoneData/code_v2/RESA_mmWave. CODE_ROOT is the repo root (branch1/, branch2/, branch3/,
# config/, sim_eval/ live directly under it -- no canon/ or LLM_ML/ nesting).
BRANCH1 = CODE_ROOT / "branch1"
BRANCH2 = CODE_ROOT / "branch2"
CONFIG_DIR = CODE_ROOT / "config"

ROOT = WORK_ROOT
STAGING_ROOT = WORK_ROOT / "_raw_branch1_preprocess_staging"
DATASET_WORK_ROOT = WORK_ROOT / "_raw_branch1_dataset_work"
FULL_DATASET_LOCAL_ROOT = WORK_ROOT / "_full_branch1_rdra_dataset_from_raw" / "rdra_patch_dataset"
BATCH_CSV_ROOT = WORK_ROOT / "_raw_branch1_batch_csvs"
FULL_DATASET_MOUNT_ROOT = Path(GCS_MOUNT_POINT) / "CapstoneData" / FULL_DATASET_SUBDIR

ENV = os.environ.copy()
ENV["RESA_CODE_ROOT"] = str(CODE_ROOT)
ENV["PYTHONPATH"] = ":".join([
    str(ROOT),
    str(CODE_ROOT),
] + ([ENV["PYTHONPATH"]] if ENV.get("PYTHONPATH") else []))

def first_existing(label: str, *paths: Path) -> Path:
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Missing required tool: {label}\nChecked:\n  " + "\n  ".join(str(Path(p)) for p in paths)
    )

def optional_existing(*paths: Path) -> Path | None:
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

ONEFORMER_EXPORT = first_existing(
    "OneFormer labelmap exporter",
    BRANCH1 / "processing" / "noah_scripts" / "OneFormer" / "noah_export_labelmaps_from_processing_v2.py",
)
ONEFORMER_AUTOLABEL = first_existing(
    "OneFormer radar autolabeler",
    # Preferred: depth-gated v4 autolabeler.
    BRANCH1 / "processing" / "noah_scripts" / "OneFormer" / "noah_autolabel_radar_using_synccsv_v4_depth_gated.py",
    # Fallback: v3 semantic autolabeler. The cell repairs/joins native radar bin
    # provenance afterward and then runs depth_correspondence_tools + ghost fusion.
    BRANCH1 / "processing" / "noah_scripts" / "OneFormer" / "noah_autolabel_radar_using_synccsv_v3.py",
)
ONEFORMER_AUTOLABEL_KIND = (
    "v4_depth_gated" if "v4_depth_gated" in ONEFORMER_AUTOLABEL.name else "v3_session"
)
PREPARE_AUTOLABEL = first_existing(
    "prepare_autolabel.py",
    BRANCH1 / "processing" / "prepare_autolabel.py",
)
RADAR_TENSOR_EXPORT = first_existing(
    "radar tensor / rd_cube exporter",
    # No refactored twin of export_radar_tensors_fast.py exists yet; the inline
    # %%writefile exporter above is the actual migration path for this stage.
    Path("/content/export_radar_tensors_fast.py"),
    BRANCH1 / "processing" / "export_radar_tensors.py",
)
DATASET_TOOLS = first_existing("dataset_tools.py", BRANCH1 / "tools" / "dataset_tools.py")
FOURCLASS_TOOLS = first_existing("branch1_fourclass_tools.py", BRANCH1 / "tools" / "branch1_fourclass_tools.py")
SPATIOTEMPORAL_TOOLS = first_existing("spatiotemporal_tools.py", BRANCH1 / "tools" / "spatiotemporal_tools.py")
DEPTH_CORRESPONDENCE_TOOLS = first_existing("depth_correspondence_tools.py", BRANCH1 / "tools" / "depth_correspondence_tools.py")
# Moved from tools/ to processing/ in the refactor.
GHOST_TEACHER_FUSION = first_existing("ghost_teacher_fusion.py", BRANCH1 / "processing" / "ghost_teacher_fusion.py")
EXTRINSICS_JSON = Path(_global_first("EXTRINSICS_JSON", default=CONFIG_DIR / "radar_camera_extrinsics.json"))
CFG_PATH = Path(_global_first("CFG_PATH", default=CONFIG_DIR / "profile_objdet.cfg"))

ADC_TO_POINTCLOUD = first_existing(
    "adc_to_pointcloud_v6.py",
    BRANCH1 / "processing" / "adc_to_pointcloud_v6.py",
)
CORRIDOR_SOFT_STRUCTURAL = first_existing(
    "corridor_soft_structural.py",
    BRANCH1 / "inputs" / "geometry" / "corridor_soft_structural.py",
)
DIRECTNESS_RUNTIME = first_existing(
    "directness_runtime.py",
    BRANCH2 / "directness_runtime.py",
)
DEPTH_CORRESPONDENCE_RUNTIME = first_existing(
    "depth_correspondence.py",
    BRANCH1 / "inputs" / "geometry" / "depth_correspondence.py",
)
CALIBRATION_RUNTIME = first_existing(
    "calibration.py",
    BRANCH1 / "calib" / "calibration.py",
)

# Ensure raw ADC conversion emits RD sidecars whenever the in-process converter is used.
os.environ.setdefault("ADC_WRITE_RD_SIDECARS", "1")
os.environ.setdefault("NAV_PERCEPTION_MODE", "pointcloud_rd_patch")
ENV.setdefault("ADC_WRITE_RD_SIDECARS", "1")
ENV.setdefault("NAV_PERCEPTION_MODE", "pointcloud_rd_patch")

if not EXTRINSICS_JSON.exists():
    raise FileNotFoundError(f"Missing extrinsics JSON required for depth-gated labels: {EXTRINSICS_JSON}")
if not CFG_PATH.exists():
    raise FileNotFoundError(f"Missing radar cfg required for RD/RA tensor export: {CFG_PATH}")

# -----------------------------
# Shell helpers
# -----------------------------

def run(cmd: Iterable[object], cwd: Path = ROOT, *, capture: bool = False, check: bool = True):
    argv = [str(x) for x in cmd]
    print("\n$ " + " ".join(shlex.quote(x) for x in argv))
    if DRY_RUN:
        return None
    if capture:
        r = subprocess.run(argv, cwd=str(cwd), env=ENV, text=True, capture_output=True)
        if check and r.returncode != 0:
            print(r.stdout or "")
            print(r.stderr or "")
            raise subprocess.CalledProcessError(r.returncode, argv, output=r.stdout, stderr=r.stderr)
        return r
    return subprocess.run(argv, cwd=str(cwd), env=ENV, text=True, check=check)

def run_quiet(cmd: Iterable[object], cwd: Path = ROOT, *, check: bool = True):
    argv = [str(x) for x in cmd]
    if DRY_RUN:
        print("\n$ " + " ".join(shlex.quote(x) for x in argv))
        return None
    return subprocess.run(argv, cwd=str(cwd), env=ENV, text=True, capture_output=True, check=check)

def command_help_text(cmd: Iterable[object]) -> str:
    r = run_quiet([*cmd, "--help"], check=False)
    if r is None:
        return ""
    return (r.stdout or "") + "\n" + (r.stderr or "")

def append_supported_flag(cmd: list[object], help_text: str, flag: str, *values: object) -> bool:
    if flag in help_text:
        cmd.append(flag)
        cmd.extend(values)
        return True
    return False

def gcloud_storage_ls(pattern: str, *, allow_empty=True) -> list[str]:
    r = run(["gcloud", "storage", "ls", pattern], capture=True, check=False)
    if r is None:
        return []
    if r.returncode != 0:
        if allow_empty:
            msg = (r.stderr or "").strip()
            if msg:
                print(msg)
            return []
        raise subprocess.CalledProcessError(r.returncode, r.args, output=r.stdout, stderr=r.stderr)
    return [line.strip() for line in (r.stdout or "").splitlines() if line.strip()]

def _remote_prefix_has_objects(uri_prefix: str) -> bool:
    """True when the GCS prefix already contains at least one real object.

    Used by the idempotency checks so a stage can skip (batch, group) work whose
    outputs were already uploaded by a previous run, instead of overwriting them.
    """
    listing = gcloud_storage_ls(f"{uri_prefix.rstrip('/')}/**", allow_empty=True)
    return any(line.strip() and not line.strip().endswith("/") for line in listing)

def _filter_gcloud_noise(text: str) -> str:
    keep = []
    noisy_prefixes = (
        "Copying ",
        "Removing ",
        "WARNING: Component check failed. Could not verify SDK install path.",
        "Average throughput:",
    )
    for raw in (text or "").splitlines():
        line = raw.rstrip()
        stripped = line.strip()
        if not stripped:
            continue
        if stripped and set(stripped) <= {"."}:
            continue
        if any(stripped.startswith(prefix) for prefix in noisy_prefixes):
            continue
        keep.append(line)
    return "\n".join(keep)


def run_gcloud_quiet(cmd, *, label="gcloud transfer"):
    argv = [str(x) for x in cmd]
    compact = " ".join(shlex.quote(x) for x in argv[:4])
    print(f"\n$ {label}: {compact} ...")
    if DRY_RUN:
        return None
    proc = subprocess.run(
        argv,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if proc.returncode != 0:
        filtered = _filter_gcloud_noise((proc.stdout or "") + "\n" + (proc.stderr or ""))
        if filtered.strip():
            print(filtered, file=sys.stderr)
        else:
            print("gcloud command failed, but emitted only suppressed progress output.", file=sys.stderr)
        raise subprocess.CalledProcessError(proc.returncode, argv, output=proc.stdout, stderr=proc.stderr)
    return proc


def gcloud_storage_cp(sources, dst, *, recursive=False):
    if isinstance(sources, (str, Path)):
        source_args = [str(sources)]
    else:
        source_args = [str(x) for x in sources]
    if not source_args:
        return
    cmd = ["gcloud", "--quiet", "--verbosity=error", "storage", "cp"]
    if recursive:
        cmd.append("--recursive")
    cmd.extend(source_args)
    cmd.append(str(dst))
    run_gcloud_quiet(cmd, label=f"gcloud cp {len(source_args)} source(s) -> {dst}")


def gcloud_storage_rsync(src, dst, *, delete_unmatched=False, exclude=None):
    cmd = ["gcloud", "--quiet", "--verbosity=error", "storage", "rsync", str(src), str(dst), "--recursive"]
    if delete_unmatched:
        cmd.append("--delete-unmatched-destination-objects")
    if exclude:
        cmd.extend(["--exclude", exclude])
    run_gcloud_quiet(cmd, label=f"gcloud rsync {src} -> {dst}")


def chunked(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]

def has_data_rows(csv_path: Path) -> bool:
    try:
        with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
            return sum(1 for _ in f) > 1
    except OSError:
        return False

SKIPPED_SESSION_LOG: list[dict[str, str]] = []

def active_session_dirs(root: Path, prefix: str) -> list[Path]:
    return sorted(p for p in Path(root).glob(f"{prefix}*") if p.is_dir())

def _csv_header_columns(path: Path) -> list[str]:
    try:
        with Path(path).open("r", encoding="utf-8", errors="replace", newline="") as fh:
            first = fh.readline().strip()
        return [c.strip().lower() for c in first.split(",") if c.strip()]
    except OSError:
        return []


def _looks_like_native_radar_csv(path: Path) -> bool:
    cols = set(_csv_header_columns(path))
    radar_markers = {"x", "y", "z", "range_m", "doppler", "doppler_mps", "snr", "azimuth_deg"}
    return len(cols & radar_markers) >= 3


def _has_color_timestamp_columns(path: Path) -> bool:
    """True when the CSV header contains at least one color-timestamp column.

    Some captures store radar detections and color frame timestamps in a single
    combined {session}.csv.  This predicate detects the timestamp side so that
    ensure_color_timestamp_aliases() can create an alias even when the file also
    looks like a native radar CSV.  _csv_header_columns() returns lower-cased names.
    """
    cols = set(_csv_header_columns(path))
    explicit = {"timestamp_us", "video_timestamp_us", "color_timestamp_us",
                "frame_index", "video_frame_index", "t_us", "ts_us",
                "timestamp", "frame_num", "color_ts"}
    if cols & explicit:
        return True
    # Also catch any column whose name contains these substrings
    return any("timestamp" in c or "frame_index" in c for c in cols)


def ensure_color_timestamp_aliases(root: Path, prefix: str) -> None:
    """Normalize raw sessions whose color timestamp CSV is named <session>.csv.

    Some captures use <session>.csv as the color timestamp file instead of
    <session>_color_timestamps.csv.  The downstream OneFormer exporter looks
    for a color timestamp CSV, so create a small alias when the session-named CSV
    exists and does not look like the native radar point CSV.
    """
    for session_dir in active_session_dirs(root, prefix):
        standard = session_dir / f"{session_dir.name}_color_timestamps.csv"
        if standard.exists():
            continue
        candidates = [
            session_dir / f"{session_dir.name}.csv",
            session_dir / "color_timestamps.csv",
        ]
        source = next((p for p in candidates if p.exists() and p.stat().st_size > 0), None)
        if source is None:
            continue
        if source.name == f"{session_dir.name}.csv" and _looks_like_native_radar_csv(source):
            if not _has_color_timestamp_columns(source):
                # Pure radar CSV with no timing columns: aliasing would produce a
                # broken color-timestamps file, so skip.
                print(f"  [COLOR TS] not aliasing {source.name}: pure native radar CSV "
                      "(no color-timestamp columns found)", flush=True)
                continue
            # Combined radar + color-timestamp CSV (common in depth captures where
            # {session}.csv holds both point cloud data and frame timing).  Create the
            # alias so the downstream OneFormer exporter can find the timestamps.
            print(f"  [COLOR TS] combined radar+timestamp CSV: aliasing "
                  f"{source.name} -> {standard.name}", flush=True)
        try:
            shutil.copy2(source, standard)
            print(f"  [COLOR TS] aliased {source.name} -> {standard.name}", flush=True)
        except Exception as exc:
            _record_skipped_session(session_dir, "color-timestamp-alias-failed", repr(exc))

def _record_skipped_session(session_dir: Path, stage: str, reason: str) -> None:
    session_dir = Path(session_dir)
    payload = {
        "session": session_dir.name,
        "stage": str(stage),
        "reason": str(reason),
        "path": str(session_dir),
    }
    SKIPPED_SESSION_LOG.append(payload)
    print(f"  [SKIP CORRUPT] {session_dir.name}: {stage}: {reason}")

def _drop_session_dir(session_dir: Path, stage: str, reason: str) -> None:
    _record_skipped_session(session_dir, stage, reason)
    if not DRY_RUN:
        shutil.rmtree(session_dir, ignore_errors=True)

def _handle_stage_problems(root: Path, prefix: str, stage: str, problems: list[tuple[str, str]]) -> bool:
    """Delete bad session dirs when SKIP_CORRUPT_SESSIONS=True; otherwise raise.

    Returns True if at least one session remains after skipping, False otherwise.
    """
    if not problems:
        return bool(active_session_dirs(root, prefix))
    sample = "\n".join(f"  {s}: {why}" for s, why in problems[:20])
    if not SKIP_CORRUPT_SESSIONS:
        raise RuntimeError(
            f"{stage} failed for one or more sessions:\n"
            + sample
            + ("\n  ..." if len(problems) > 20 else "")
        )
    print(f"\n[{stage}] skipping {len(problems)} corrupt/incomplete session(s):")
    print(sample + ("\n  ..." if len(problems) > 20 else ""))
    seen: set[str] = set()
    for sess_name, why in problems:
        if sess_name in seen:
            continue
        seen.add(sess_name)
        _drop_session_dir(Path(root) / sess_name, stage, why)
    remaining = active_session_dirs(root, prefix)
    print(f"[{stage}] remaining usable session folder(s): {len(remaining)}")
    return bool(remaining)

def _write_skip_log(out_dir: Path | None = None) -> Path | None:
    if not SKIPPED_SESSION_LOG:
        return None
    out_dir = Path(out_dir or BATCH_CSV_ROOT)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "skipped_sessions.csv"
    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=["session", "stage", "reason", "path"])
        writer.writeheader()
        writer.writerows(SKIPPED_SESSION_LOG)
    return path

# -----------------------------
# GCS/session discovery
# -----------------------------

def _normalize_csv_values(values: Iterable[str] | None) -> tuple[str, ...]:
    if values is None:
        return ()
    return tuple(str(v).strip() for v in values if str(v).strip())


def load_live_catalog() -> pd.DataFrame:
    sync_catalog_from_gcs(CATALOG_LOCAL_PATH, GCS_CATALOG_URI)
    return load_catalog(CATALOG_LOCAL_PATH)


def raw_catalog_rows(catalog: pd.DataFrame | None = None) -> pd.DataFrame:
    if catalog is None:
        catalog = load_live_catalog()
    out = select_sessions(
        catalog,
        quality_tier=(),
        label_status=RAW_LABEL_STATUS_INCLUDE,
        dataset_ids=RAW_DATASET_IDS or None,
        require_processed_path=False,
    )
    raw_session_ids = set(_normalize_csv_values(RAW_SESSION_IDS))
    if raw_session_ids:
        out = out[out["session_id"].astype(str).isin(raw_session_ids)]
    return out[out["raw_path"].fillna("").astype(str).str.len() > 0].reset_index(drop=True)


def session_ref_from_gcs_uri(uri: str, group: str, prefix: str) -> tuple[str, str] | None:
    uri = str(uri).rstrip("/")
    name = uri.split("/")[-1]
    if not name.startswith(prefix) or name.endswith((".tgz", ".tar.gz", ".tar", ".gz")):
        return None
    return name, uri


def list_raw_sessions_for_group(group: str, prefix: str) -> list[dict]:
    catalog = raw_catalog_rows()
    group_rows = catalog[catalog["dataset_id"].astype(str) == str(group)].copy()
    sessions = [
        {"name": str(row.session_id), "uri": str(row.raw_path).rstrip("/")}
        for row in group_rows.itertuples(index=False)
        if str(row.session_id).startswith(prefix)
    ]
    print(f"{group}: found {len(sessions)} raw session folder(s) from session_catalog.csv")
    if sessions:
        print("  first:", [s["name"] for s in sessions[:3]])
        print("  last :", [s["name"] for s in sessions[-3:]])
    return sessions


def raw_session_records() -> list[dict]:
    rows = raw_catalog_rows()
    records = []
    for row in rows.itertuples(index=False):
        dataset_id = str(row.dataset_id)
        session_id = str(row.session_id)
        raw_uri = str(row.raw_path).rstrip("/")
        if not session_id.startswith(RAW_SESSION_PREFIX):
            continue
        records.append({
            "tag": dataset_id,
            "group": dataset_id,
            "dataset_id": dataset_id,
            "legacy_dataset_name": str(row.legacy_dataset_name),
            "prefix": RAW_SESSION_PREFIX,
            "name": session_id,
            "session": session_id,
            "uri": raw_uri,
        })
    if not records:
        raise RuntimeError(
            f"No raw sessions matched the live catalog filters under {RAW_EXTRACTS_ROOT_URI}. "
            f"dataset_ids={list(RAW_DATASET_IDS) or '<all>'} session_ids={list(RAW_SESSION_IDS) or '<all>'} "
            f"label_statuses={list(RAW_LABEL_STATUS_INCLUDE)}"
        )
    return sorted(records, key=lambda r: (r["group"], r["name"]))

# -----------------------------
# Preprocessing stages
# -----------------------------


def _load_prepare_autolabel_module():
    """
    Import prepare_autolabel.py as a library, but do not call its main().

    Why: the script's main() assumes seg/*.npy already exists and tries to load
    ADE20K metadata before OneFormer export. In this raw-session build, we need
    radar CSV + metadata first, then OneFormer, then synchronized CSV.
    """
    import importlib.util
    module_name = "_branch1_prepare_autolabel_lib"
    if module_name in sys.modules:
        return sys.modules[module_name]
    spec = importlib.util.spec_from_file_location(module_name, str(PREPARE_AUTOLABEL))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not import prepare_autolabel.py from {PREPARE_AUTOLABEL}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

def _find_session_radar_csv(session_dir: Path) -> Path | None:
    preferred = session_dir / f"{session_dir.name}.csv"
    if preferred.exists():
        return preferred
    excluded = {
        "labeled_radar_points_v4.csv",
        "labeled_radar_points_v4_fused.csv",
        "batch_autolabel_summary_v3.csv",
        "batch_autolabel_summary_v4.csv",
    }
    candidates = []
    for p in sorted(session_dir.glob("*.csv")):
        if p.name in excluded or p.name.startswith("synchronized_"):
            continue
        # Prefer true point-cloud CSVs with bin provenance.
        try:
            with p.open("r", encoding="utf-8", errors="replace", newline="") as fh:
                header = fh.readline().strip().split(",")
            h = set(header)
            score = (
                10 * int({"x", "y", "z"}.issubset(h))
                + 5 * int({"range_bin", "doppler_bin"}.issubset(h))
                + 2 * int("timestamp_us" in h)
                + 1 * int(("frame_num" in h) or ("radar_frame_num" in h))
            )
            candidates.append((score, p))
        except OSError:
            continue
    candidates = [(s, p) for s, p in candidates if s >= 10]
    if not candidates:
        return None
    return sorted(candidates, key=lambda x: (-x[0], x[1].name))[0][1]

def _assert_native_bin_provenance(csv_path: Path, session_name: str):
    import pandas as pd

    df_head = pd.read_csv(csv_path, nrows=2048, low_memory=False)
    missing = sorted({"range_bin", "doppler_bin", "radar_frame_num"} - set(df_head.columns))
    if missing:
        raise RuntimeError(
            f"{session_name}: radar CSV exists but is missing native bin/frame provenance {missing}: {csv_path}\n"
            "Do not recover these from range_m/doppler for RD/true-RA training; regenerate with adc_to_pointcloud_v6."
        )
    for col in ("range_bin", "doppler_bin", "radar_frame_num"):
        vals = pd.to_numeric(df_head[col], errors="coerce")
        if vals.isna().any():
            raise RuntimeError(
                f"{session_name}: {col} contains NaN/non-numeric values in {csv_path}; "
                "regenerate with adc_to_pointcloud_v6."
            )
    if len(df_head) and not pd.to_numeric(df_head["range_bin"], errors="coerce").between(0, 255).all():
        raise RuntimeError(f"{session_name}: range_bin outside [0,255] in {csv_path}")
    if len(df_head) and not pd.to_numeric(df_head["doppler_bin"], errors="coerce").between(0, 31).all():
        raise RuntimeError(f"{session_name}: doppler_bin outside [0,31] in {csv_path}")

def _has_rd_sidecar_manifest(session_dir: Path) -> bool:
    manifest = session_dir / "hybrid_rd" / "manifest.json"
    frames_dir = session_dir / "hybrid_rd" / "frames"
    return bool(manifest.exists() and any(frames_dir.glob("rd_frame_*.npz")))

def _has_session_tensor(session_dir: Path) -> bool:
    return bool((session_dir / f"{session_dir.name}_radar_tensors.npz").exists())

def _needs_native_radar_reprocess(session_dir: Path, csv_path: Path | None) -> tuple[bool, str]:
    if csv_path is None:
        return True, "missing radar CSV"
    try:
        _assert_native_bin_provenance(csv_path, session_dir.name)
    except Exception as exc:
        return True, f"bad radar CSV provenance: {exc}"
    if not _has_rd_sidecar_manifest(session_dir):
        return True, "missing hybrid_rd/manifest.json or RD frame sidecars"
    return False, "ok"

def _import_file_module(module_name: str, path: Path):
    path = Path(path)
    cache_key = f"_branch1_{module_name}_{abs(hash(str(path)))}"
    if cache_key in sys.modules:
        return sys.modules[cache_key]
    spec = importlib.util.spec_from_file_location(cache_key, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not import {module_name} from {path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[cache_key] = mod
    spec.loader.exec_module(mod)
    return mod

def _build_radar_processor():
    prep = _load_prepare_autolabel_module()
    if hasattr(prep, "_add_adc_module_paths"):
        prep._add_adc_module_paths()
    for candidate in [
        CODE_ROOT,
        BRANCH1 / "processing",
        BRANCH1 / "inputs" / "geometry",
        BRANCH1 / "calib",
    ]:
        if candidate.exists():
            sys.path.insert(0, str(candidate))
    v6 = _import_file_module("adc_to_pointcloud_v6", ADC_TO_POINTCLOUD)
    if not CFG_PATH.exists():
        raise FileNotFoundError(f"Radar cfg not found: {CFG_PATH}")
    print(f"Building radar processor from {CFG_PATH}")
    # Force native RD sidecars here; true RA still requires the later rd_cube export stage.
    return v6, v6.build_processor(CFG_PATH, write_rd_sidecars=True)

def run_radar_csv_and_meta_stage(root: Path, prefix: str) -> bool:
    """
    Stage A: generate/reuse the native radar point-cloud CSV and write meta_data.json.

    Corrupt sessions are removed from the staging root when SKIP_CORRUPT_SESSIONS=True.
    Typical corrupt cases: missing .bin, unreadable raw file, broken CSV provenance, or
    ADC sidecar generation failure.
    """
    prep = _load_prepare_autolabel_module()
    sessions = active_session_dirs(root, prefix)
    if not sessions:
        print(f"[RADAR/META] no sessions under {root} matching {prefix}")
        return False

    reprocess_reasons: dict[str, str] = {}
    for sdir in sessions:
        csv_path = _find_session_radar_csv(sdir)
        needs, reason = _needs_native_radar_reprocess(sdir, csv_path)
        if needs:
            reprocess_reasons[sdir.name] = reason

    v6 = processor = None
    if reprocess_reasons:
        v6, processor = _build_radar_processor()

    failed: list[tuple[str, str]] = []
    for idx, session_dir in enumerate(sessions, start=1):
        print(f"\n[RADAR/META {idx}/{len(sessions)}] {session_dir.name}")
        csv_path = _find_session_radar_csv(session_dir)
        if session_dir.name in reprocess_reasons:
            print(f"  [CSV] regenerating radar point cloud ({reprocess_reasons[session_dir.name]}) …")
            try:
                ok, n_det, out_csv = v6.process_session_fast(processor, session_dir)
            except Exception as exc:
                print(f"  [CSV] ERROR: {exc}")
                failed.append((session_dir.name, f"radar CSV generation exception: {exc}"))
                continue
            if not ok:
                failed.append((session_dir.name, f"radar CSV generation failed: {out_csv}"))
                continue
            csv_path = Path(out_csv)
            print(f"  [CSV] wrote {n_det} detections -> {csv_path.name}")
        else:
            print(f"  [CSV] using existing {csv_path.name} with RD sidecars")

        try:
            _assert_native_bin_provenance(csv_path, session_dir.name)
        except Exception as exc:
            failed.append((session_dir.name, str(exc)))
            continue
        if not _has_rd_sidecar_manifest(session_dir):
            failed.append((session_dir.name, "adc_to_pointcloud_v6 did not produce hybrid_rd/manifest.json + RD sidecars"))
            continue

        try:
            prep.write_meta_data(session_dir, dry_run=False)
        except Exception as exc:
            failed.append((session_dir.name, f"meta_data.json write failed: {exc}"))

    return _handle_stage_problems(root, prefix, "radar/meta", failed)

def _load_ade20k_id2label(root: Path) -> dict[str, str]:
    """
    Load id2label after OneFormer export. Prefer session seg_meta; then HF config
    from cache; then transformers AutoConfig if installed/available.
    """
    # 1) Existing session metadata.
    for seg_meta in sorted(root.glob(f"session_*/seg_meta.json")) + sorted(root.glob("*/seg_meta.json")):
        try:
            meta = json.loads(seg_meta.read_text(encoding="utf-8"))
            id2label = meta.get("id2label")
            if id2label:
                print(f"Loading ADE20K id2label from {seg_meta}")
                return {str(k): str(v) for k, v in id2label.items()}
        except Exception:
            pass

    # 2) HuggingFace cache/config locations.
    candidates = [
        Path("/root/.cache/huggingface/hub/models--shi-labs--oneformer_ade20k_swin_tiny"),
        Path("/content/.cache/huggingface/hub/models--shi-labs--oneformer_ade20k_swin_tiny"),
        Path("/home/hullumdr/.cache/huggingface/hub/models--shi-labs--oneformer_ade20k_swin_tiny"),
    ]
    for base in candidates:
        for cfg in sorted(base.glob("snapshots/*/config.json")):
            try:
                data = json.loads(cfg.read_text(encoding="utf-8"))
                id2label = data.get("id2label")
                if id2label:
                    print(f"Loading ADE20K id2label from HF cache: {cfg}")
                    return {str(k): str(v) for k, v in id2label.items()}
            except Exception:
                pass

    # 3) transformers local/remote config fallback.
    try:
        from transformers import AutoConfig
        try:
            cfg = AutoConfig.from_pretrained("shi-labs/oneformer_ade20k_swin_tiny", local_files_only=True)
        except Exception:
            cfg = AutoConfig.from_pretrained("shi-labs/oneformer_ade20k_swin_tiny")
        id2label = getattr(cfg, "id2label", None)
        if id2label:
            print("Loading ADE20K id2label from transformers AutoConfig")
            return {str(k): str(v) for k, v in id2label.items()}
    except Exception as exc:
        print("Could not load ADE20K id2label via transformers:", exc)

    raise FileNotFoundError(
        "Could not find ADE20K id2label after OneFormer export. "
        "Expected a session seg_meta.json or a cached/downloadable OneFormer config."
    )

def run_sync_and_seg_meta_stage(root: Path, prefix: str) -> bool:
    """Stage C: after OneFormer has produced seg/*.npy, build sync CSVs and seg_meta.json."""
    prep = _load_prepare_autolabel_module()
    id2label = _load_ade20k_id2label(root)
    sessions = active_session_dirs(root, prefix)
    failed: list[tuple[str, str]] = []

    for idx, session_dir in enumerate(sessions, start=1):
        print(f"\n[SYNC/META {idx}/{len(sessions)}] {session_dir.name}")
        seg_dir = session_dir / "seg"
        if not seg_dir.exists() or not any(seg_dir.glob("*.npy")):
            failed.append((session_dir.name, "missing seg/*.npy after OneFormer export"))
            continue
        csv_path = _find_session_radar_csv(session_dir)
        if csv_path is None:
            failed.append((session_dir.name, "missing radar CSV"))
            continue
        try:
            radar_ts = prep.radar_timestamps_from_manifest(session_dir)
            if radar_ts is None:
                print("  [SYNC] no manifest — reading timestamps from CSV …")
                radar_ts = prep.radar_timestamps_from_csv(csv_path)
            if not radar_ts:
                failed.append((session_dir.name, "empty radar timestamps"))
                continue
            prep.build_sync_csv(session_dir, radar_ts, dry_run=False)
            prep.write_seg_meta(session_dir, id2label, dry_run=False)
        except Exception as exc:
            failed.append((session_dir.name, str(exc)))

    return _handle_stage_problems(root, prefix, "sync/seg-meta", failed)

def run_prepare_for_root(root: Path, prefix: str):
    return run_radar_csv_and_meta_stage(root, prefix)

def _load_oneformer_export_module():
    """Import the session-level exporter so we can reuse its helpers in batch mode."""
    spec = importlib.util.spec_from_file_location("oneformer_export_batch", str(ONEFORMER_EXPORT))
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


def _write_oneformer_session_maps(
    export_mod,
    *,
    session_dir: Path,
    processor,
    model,
    device: str,
    model_name: str,
    every_n: int,
    max_w: int,
    save_png: bool,
    model_size: str,
    frame_batch_size: int = 1,
) -> None:
    """Run OneFormer for one session using a model already loaded by the batch stage.

    This version micro-batches frames through the HF processor/model to keep A100-class
    GPUs fed. Outputs remain per-session/per-frame because downstream autolabel tools
    expect session_dir/seg/<frame_index>.npy plus seg_timestamps.csv.
    """
    import cv2
    import json
    import numpy as np
    import torch
    import time
    from collections import defaultdict
    from PIL import Image
    from tqdm import tqdm
    import pandas as pd

    session_dir = Path(session_dir)
    frame_batch_size = max(1, int(frame_batch_size))
    color_mp4 = export_mod.find_color_mp4(session_dir)
    ts_csv = export_mod.find_color_timestamps_csv(session_dir)
    frame_indices, ts_map, cols = export_mod.load_frame_indices(ts_csv)

    print(f"  Session   : {session_dir.name}")
    print(f"  Video     : {color_mp4.name}")
    print(f"  Timestamps: {ts_csv.name} ({len(frame_indices)} entries, cols={list(cols)})")
    print(f"  Frame batch size: {frame_batch_size}")

    seg_dir = session_dir / "seg"
    seg_dir.mkdir(parents=True, exist_ok=True)
    if save_png:
        (session_dir / "seg_vis").mkdir(parents=True, exist_ok=True)

    id2label = {str(k): v for k, v in (getattr(model.config, "id2label", {}) or {}).items()}
    meta = {
        "model_name": str(model_name),
        "model_size": str(model_size),
        "id2label": id2label,
        "every_n": int(every_n),
        "max_w": int(max_w),
        "frame_batch_size": int(frame_batch_size),
        "use_fp16": bool(ONEFORMER_USE_FP16),
        "color_video": color_mp4.name,
        "timestamps_csv": ts_csv.name,
        "export_mode": "batch_inprocess_frame_microbatched",
    }
    (session_dir / "seg_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    cap = cv2.VideoCapture(str(color_mp4))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {color_mp4}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    processed_rows: list[dict[str, int]] = []
    n_skipped = 0
    timing = defaultdict(float)
    n_batches = 0
    n_model_frames = 0
    pending: list[dict] = []

    def _sync_cuda():
        if device == "cuda" and torch.cuda.is_available():
            torch.cuda.synchronize()

    def _flush_pending() -> None:
        nonlocal pending, n_batches, n_model_frames
        if not pending:
            return
        batch = pending
        pending = []
        images = [item["pil_img"] for item in batch]
        target_sizes = [item["target_size"] for item in batch]

        t0 = time.perf_counter()
        task_inputs = ["semantic"] * len(images)
        inputs = processor(images=images, task_inputs=task_inputs, return_tensors="pt")
        timing["processor"] += time.perf_counter() - t0

        t0 = time.perf_counter()
        moved = {}
        for k, v in inputs.items():
            if torch.is_tensor(v):
                moved[k] = v.to(device, non_blocking=True)
            else:
                moved[k] = v
        if device == "cuda" and ONEFORMER_USE_FP16 and "pixel_values" in moved:
            moved["pixel_values"] = moved["pixel_values"].half()
        timing["to_device"] += time.perf_counter() - t0

        _sync_cuda()
        t0 = time.perf_counter()
        outputs = model(**moved)
        _sync_cuda()
        timing["gpu_model"] += time.perf_counter() - t0

        t0 = time.perf_counter()
        segs = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
        timing["postprocess"] += time.perf_counter() - t0

        t0 = time.perf_counter()
        for item, seg in zip(batch, segs):
            seg_np = seg.detach().cpu().numpy().astype(np.int32)
            np.save(item["out_npy"], seg_np)
            if save_png:
                vis = export_mod.colorize_label_map(seg_np)
                cv2.imwrite(str(session_dir / "seg_vis" / f"{item['frame_index']:06d}.png"), vis)
            processed_rows.append({
                "frame_index": int(item["frame_index"]),
                "timestamp_ms": int(ts_map.get(int(item["frame_index"]), -1)),
            })
        timing["save_npy"] += time.perf_counter() - t0
        n_batches += 1
        n_model_frames += len(batch)

    try:
        with torch.inference_mode():
            pbar = tqdm(
                total=min(len(frame_indices), total_frames) if total_frames > 0 else len(frame_indices),
                desc=f"OneFormer {session_dir.name}",
            )
            for _, frame_index in enumerate(frame_indices):
                t0 = time.perf_counter()
                ok, frame_bgr = cap.read()
                timing["decode"] += time.perf_counter() - t0
                if not ok:
                    break

                frame_index = int(frame_index)
                if frame_index % int(every_n) != 0:
                    n_skipped += 1
                    pbar.update(1)
                    continue

                out_npy = seg_dir / f"{frame_index:06d}.npy"
                if out_npy.exists():
                    processed_rows.append({
                        "frame_index": frame_index,
                        "timestamp_ms": int(ts_map.get(frame_index, -1)),
                    })
                    pbar.update(1)
                    continue

                t0 = time.perf_counter()
                h0, w0 = frame_bgr.shape[:2]
                if max_w and max_w > 0 and w0 > max_w:
                    scale = float(max_w) / float(w0)
                    w1, h1 = int(w0 * scale), int(h0 * scale)
                    frame_small = cv2.resize(frame_bgr, (w1, h1), interpolation=cv2.INTER_AREA)
                else:
                    frame_small = frame_bgr
                pil_img = Image.fromarray(cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB))
                timing["resize_convert"] += time.perf_counter() - t0

                pending.append({
                    "frame_index": frame_index,
                    "out_npy": out_npy,
                    "pil_img": pil_img,
                    "target_size": (h0, w0),
                })
                if len(pending) >= frame_batch_size:
                    _flush_pending()
                pbar.update(1)
            _flush_pending()
            pbar.close()
    finally:
        cap.release()

    pd.DataFrame(processed_rows).to_csv(session_dir / "seg_timestamps.csv", index=False)
    if not processed_rows:
        raise RuntimeError("OneFormer produced zero processed frames")
    if not any(seg_dir.glob("*.npy")):
        raise RuntimeError("OneFormer produced no seg/*.npy label maps")

    print(
        f"  [DONE] {len(processed_rows)} maps/timestamps -> {seg_dir}; "
        f"skipped_by_every_n={n_skipped}; model_batches={n_batches}; model_frames={n_model_frames}"
    )
    if ONEFORMER_TIMING:
        total_t = sum(float(v) for v in timing.values()) or 1.0
        print("  [TIMING] seconds / share:")
        for key in ["decode", "resize_convert", "processor", "to_device", "gpu_model", "postprocess", "save_npy"]:
            val = float(timing.get(key, 0.0))
            print(f"    {key:14s}: {val:8.2f}s  {100.0 * val / total_t:5.1f}%")



def _run_oneformer_batch_inprocess(root: Path, prefix: str, sessions: list[Path]) -> list[tuple[str, str]]:
    """Export OneFormer maps for all active sessions in one Python/model process."""
    import torch
    from transformers import OneFormerForUniversalSegmentation, OneFormerProcessor

    export_mod = _load_oneformer_export_module()
    here = Path(ONEFORMER_EXPORT).resolve().parent
    local_name = "oneformer_tiny" if ONEFORMER_MODEL == "tiny" else "oneformer_large"
    local_path = here / local_name
    hf_name = (
        "shi-labs/oneformer_ade20k_swin_tiny"
        if ONEFORMER_MODEL == "tiny"
        else "shi-labs/oneformer_ade20k_swin_large"
    )
    model_name = str(local_path) if local_path.is_dir() else hf_name
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n[ONEFORMER BATCH]")
    print(f"  root      : {root}")
    print(f"  prefix    : {prefix}")
    print(f"  sessions  : {len(sessions)}")
    print(f"  model     : {model_name} [{ONEFORMER_MODEL}]")
    print(f"  device    : {device}")
    print(f"  every_n   : {ONEFORMER_EVERY_N}")
    print(f"  max_w     : {ONEFORMER_MAX_W or 'disabled'}")
    print(f"  frame_bs  : {ONEFORMER_FRAME_BATCH_SIZE}")
    print(f"  fp16      : {ONEFORMER_USE_FP16}")

    if device == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision("high")

    use_local = local_path.is_dir()
    processor = OneFormerProcessor.from_pretrained(model_name, local_files_only=use_local)
    use_safetensors = True
    if local_path.is_dir():
        st_files = list(local_path.glob("*.safetensors")) + list(local_path.glob("model-*.safetensors"))
        use_safetensors = bool(st_files)
    model = OneFormerForUniversalSegmentation.from_pretrained(
        model_name,
        local_files_only=use_local,
        use_safetensors=use_safetensors,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    model.eval()
    print("  Model loaded once for this batch.")

    failed: list[tuple[str, str]] = []
    for idx, session_dir in enumerate(sessions, start=1):
        if (session_dir / "seg_timestamps.csv").exists() and any((session_dir / "seg").glob("*.npy")):
            print(f"\n[ONEFORMER BATCH {idx}/{len(sessions)}] skip existing: {session_dir.name}")
            continue
        try:
            print(f"\n[ONEFORMER BATCH {idx}/{len(sessions)}] {session_dir.name}")
            _write_oneformer_session_maps(
                export_mod,
                session_dir=session_dir,
                processor=processor,
                model=model,
                device=device,
                model_name=model_name,
                every_n=ONEFORMER_EVERY_N,
                max_w=ONEFORMER_MAX_W,
                save_png=ONEFORMER_SAVE_PNG,
                model_size=ONEFORMER_MODEL,
                frame_batch_size=ONEFORMER_FRAME_BATCH_SIZE,
            )
        except Exception as exc:
            failed.append((session_dir.name, str(exc)))
    # Release GPU memory before later stages.
    try:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return failed


def run_oneformer_for_root(root: Path, prefix: str) -> bool:
    """Batch-export OneFormer maps for the currently staged group.

    The output format stays per-session because downstream autolabel/depth tools
    expect session_dir/seg/*.npy, session_dir/seg_meta.json, and
    session_dir/seg_timestamps.csv. The important change is execution: the model
    is loaded once per batch/group instead of once per session.
    """
    sessions = active_session_dirs(root, prefix)
    if not sessions:
        print(f"[OneFormer] no sessions under {root} matching {prefix}")
        return False
    failed = _run_oneformer_batch_inprocess(root, prefix, sessions)
    # Validate every remaining session, including skipped-existing ones.
    for session_dir in active_session_dirs(root, prefix):
        if not ((session_dir / "seg_timestamps.csv").exists() and any((session_dir / "seg").glob("*.npy"))):
            failed.append((session_dir.name, "missing seg_timestamps.csv + seg/*.npy after batch OneFormer export"))
    return _handle_stage_problems(root, prefix, "oneformer-batch", failed)

# -----------------------------
# Explicit per-column feature enrichment / schema contract
# -----------------------------

ACTIVE_SCALAR_FEATURE_COLS = [
    "x", "y", "z", "range_m", "azimuth_deg", "elevation_deg", "doppler", "snr",
    "local_density", "persist_score", "z_norm_range",
    "frame_doppler_abs", "frame_doppler_std",
    "z_above_floor", "corridor_margin", "wall_anomaly",
    "floor_ang", "wall_ang", "has_floor", "has_wall",
    "ego_doppler_residual_mps", "ego_residual_abs_z", "ego_inlier_flag",
    "p_ego", "p_dir_doppler",
    "rd_entropy", "rd_doppler_spread", "rd_anisotropy", "rd_peak_ratio",
]

NATIVE_RADAR_REQUIRED_COLS = {
    "x", "y", "z", "range_m", "doppler", "snr", "range_bin", "doppler_bin", "radar_frame_num"
}
# Columns that must come from the native ADC/autolabel join and cannot be
# reconstructed from scalar geometry. `range_m`, `azimuth_deg`, and
# `elevation_deg` are safe to compute from XYZ when an autolabeler omits them.
AUTOLABEL_HARD_PROVENANCE_COLS = {
    "x", "y", "z", "bucket", "range_bin", "doppler_bin", "radar_frame_num"
}
COMPUTED_PREPATCH_REQUIRED_COLS = {
    "azimuth_deg", "elevation_deg", "local_density", "persist_score", "z_norm_range",
    "frame_doppler_abs", "frame_doppler_std",
    "z_above_floor", "corridor_margin", "wall_anomaly",
    "floor_ang", "wall_ang", "has_floor", "has_wall",
    "ego_doppler_residual_mps", "ego_residual_abs_z", "ego_inlier_flag", "p_ego", "p_dir_doppler",
}
DEPTH_TEACHER_EXPECTED_COLS = {
    "depth_corr_available", "depth_corr_in_fov", "depth_corr_valid_px", "depth_corr_patch_px",
    "depth_corr_radar_z_m", "depth_corr_min_m", "depth_corr_median_m", "depth_corr_selected_m",
    "depth_corr_residual_m", "depth_corr_abs_residual_m", "depth_corr_sigma_m", "depth_corr_quality",
    "p_depth_match", "depth_foreground_occluded", "depth_missing_reason",
    "matched_video_frame_index", "matched_depth_frame_index", "cam_u_depth", "cam_v_depth",
}
DIRECTNESS_MODEL_FEATURE_COLS = {
    "ego_doppler_residual_mps", "ego_residual_abs_z", "ego_inlier_flag", "p_ego", "p_dir_doppler",
}
SPATIOTEMPORAL_GHOST_FEATURE_COLS = {
    "ego_expected_doppler_mps", "motion_label", "spatiotemporal_weight",
    "temporal_persistence_frames", "ghost_score", "static_confidence",
    "final_train_weight", "ego_available",
}
GHOST_TEACHER_FEATURE_COLS = {
    "ghost_score_combined", "accumulatable_label", "return_type_label",
    "depth_teacher_eligible", "depth_teacher_acc_label", "depth_teacher_return_type",
    "depth_teacher_weight", "depth_teacher_reason", "depth_teacher_source",
}
FINAL_PATCH_REQUIRED_COLS = {
    "session", "radar_frame_num", "x", "y", "z", "range_m", "doppler", "snr",
    "range_bin", "doppler_bin", "rd_patch_shard", "rd_patch_index", "rd_patch_valid",
    "ra_patch_shard", "ra_patch_index", "ra_patch_valid", "bucket_4class",
}


def _to_numeric_series(series, default=0.0):
    import pandas as pd
    return pd.to_numeric(series, errors="coerce").fillna(float(default))


def _normalize_radar_columns(df):
    df = df.copy()
    if "radar_frame_num" not in df.columns and "frame_num" in df.columns:
        df["radar_frame_num"] = df["frame_num"]
    if "frame_num" not in df.columns and "radar_frame_num" in df.columns:
        df["frame_num"] = df["radar_frame_num"]
    if "doppler" not in df.columns:
        for alt in ("doppler_mps", "v"):
            if alt in df.columns:
                df["doppler"] = df[alt]
                break
    if "snr" not in df.columns and "power_snr" in df.columns:
        df["snr"] = df["power_snr"]
    if "range_m" not in df.columns and {"x", "y", "z"}.issubset(df.columns):
        import numpy as np
        xyz = df[["x", "y", "z"]].apply(_to_numeric_series).to_numpy(dtype=float)
        df["range_m"] = np.linalg.norm(xyz, axis=1)
    return df


def _compute_basic_geometry_features(df):
    import numpy as np
    df = _normalize_radar_columns(df)
    for col in ["x", "y", "z", "range_m", "doppler", "snr"]:
        if col in df.columns:
            df[col] = _to_numeric_series(df[col])
    x = df["x"].to_numpy(float)
    y = df["y"].to_numpy(float)
    z = df["z"].to_numpy(float)
    xy = np.hypot(x, y)
    # Some OneFormer autolabelers preserve XYZ/bins but omit range_m.
    # This is safe to compute from the same native radar XYZ and should not
    # trigger the expensive native-row join used for missing bin provenance.
    if "range_m" not in df.columns:
        df["range_m"] = np.sqrt(x * x + y * y + z * z)
    else:
        r_existing = _to_numeric_series(df["range_m"], default=np.nan)
        computed_r = np.sqrt(x * x + y * y + z * z)
        df["range_m"] = r_existing.where(np.isfinite(r_existing) & (r_existing > 0.0), computed_r)
    r = df["range_m"].to_numpy(float)
    safe_r = np.maximum(r, 1e-6)
    if "azimuth_deg" not in df.columns or df["azimuth_deg"].isna().any():
        df["azimuth_deg"] = np.degrees(np.arctan2(x, y))
    if "elevation_deg" not in df.columns or df["elevation_deg"].isna().any():
        df["elevation_deg"] = np.degrees(np.arctan2(z, np.maximum(xy, 1e-6)))
    df["z_norm_range"] = z / safe_r
    return df


def _compute_local_density_for_xyz(xyz, radius_m: float):
    import numpy as np
    n = int(len(xyz))
    if n == 0:
        return np.zeros(0, dtype=np.float32)
    if n == 1:
        return np.zeros(1, dtype=np.float32)
    try:
        from scipy.spatial import cKDTree
        tree = cKDTree(xyz.astype(float))
        counts = np.asarray([len(ix) - 1 for ix in tree.query_ball_point(xyz, r=radius_m)], dtype=np.float32)
        return counts
    except Exception:
        diff = xyz[:, None, :] - xyz[None, :, :]
        d = np.linalg.norm(diff, axis=2)
        return ((d <= radius_m).sum(axis=1) - 1).astype(np.float32)


def _compute_persist_scores(df, radius_m: float = PERSIST_RADIUS_M, frame_window: int = PERSIST_TEMPORAL_RADIUS_FRAMES):
    import numpy as np
    out = np.zeros(len(df), dtype=np.float32)
    if len(df) == 0 or "radar_frame_num" not in df.columns:
        return out
    frame_vals = _to_numeric_series(df["radar_frame_num"]).astype(int).to_numpy()
    xyz_all = df[["x", "y", "z"]].to_numpy(dtype=float)
    unique_frames = sorted(set(int(x) for x in frame_vals))
    by_frame = {fn: np.where(frame_vals == fn)[0] for fn in unique_frames}
    try:
        from scipy.spatial import cKDTree
    except Exception:
        cKDTree = None
    for fn in unique_frames:
        idx = by_frame[fn]
        if len(idx) == 0:
            continue
        neighbor_frames = [g for g in unique_frames if g != fn and abs(g - fn) <= int(frame_window)]
        denom = max(len(neighbor_frames), 1)
        if not neighbor_frames:
            continue
        hits = np.zeros(len(idx), dtype=np.float32)
        pts = xyz_all[idx]
        for gf in neighbor_frames:
            jdx = by_frame.get(gf, [])
            if len(jdx) == 0:
                continue
            other = xyz_all[jdx]
            if cKDTree is not None:
                tree = cKDTree(other)
                d, _ = tree.query(pts, k=1, distance_upper_bound=radius_m)
                hits += np.isfinite(d).astype(np.float32)
            else:
                d = np.linalg.norm(pts[:, None, :] - other[None, :, :], axis=2)
                hits += (np.min(d, axis=1) <= radius_m).astype(np.float32)
        out[idx] = np.clip(hits / float(denom), 0.0, 1.0)
    return out


def _apply_frame_level_features(df):
    import numpy as np
    df = df.copy()
    xyz = df[["x", "y", "z"]].to_numpy(dtype=float)
    frames = _to_numeric_series(df["radar_frame_num"]).astype(int).to_numpy()
    df["local_density"] = 0.0
    for fn in sorted(set(frames)):
        mask = frames == fn
        df.loc[mask, "local_density"] = _compute_local_density_for_xyz(xyz[mask], LOCAL_DENSITY_RADIUS_M)
    df["persist_score"] = _compute_persist_scores(df)
    dop = _to_numeric_series(df["doppler"]).to_numpy(dtype=float)
    df["frame_doppler_abs"] = 0.0
    df["frame_doppler_std"] = 0.0
    for fn in sorted(set(frames)):
        mask = frames == fn
        vals = dop[mask]
        df.loc[mask, "frame_doppler_abs"] = float(np.mean(np.abs(vals))) if len(vals) else 0.0
        df.loc[mask, "frame_doppler_std"] = float(np.std(vals)) if len(vals) else 0.0
    return df


def _apply_soft_structural_features(df):
    import numpy as np
    css = _import_file_module("corridor_soft_structural", CORRIDOR_SOFT_STRUCTURAL)
    df = df.copy()
    frames = _to_numeric_series(df["radar_frame_num"]).astype(int).to_numpy()
    xyz = df[["x", "y", "z"]].to_numpy(dtype=np.float32)
    for col in getattr(css, "SOFT_STRUCTURAL_FEATURE_COLS", []):
        if col not in df.columns:
            df[col] = 0.0
    for fn in sorted(set(frames)):
        mask = frames == fn
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue
        agg_mask = np.abs(frames - fn) <= int(STRUCTURAL_TEMPORAL_RADIUS_FRAMES)
        anchor = xyz[idx]
        aggregated = xyz[agg_mask]
        try:
            _, feats = css.process_frame_group(anchor, aggregated, compute_normals=True)
        except Exception as exc:
            print(f"  [STRUCT] frame {fn}: fallback due to {exc}")
            feats = {
                "z_above_floor": anchor[:, 2].astype(np.float32),
                "corridor_margin": np.zeros(len(anchor), dtype=np.float32),
                "wall_anomaly": np.zeros(len(anchor), dtype=np.float32),
                "floor_ang": np.full(len(anchor), math.pi / 2, dtype=np.float32),
                "wall_ang": np.full(len(anchor), math.pi / 2, dtype=np.float32),
                "has_floor": np.zeros(len(anchor), dtype=np.float32),
                "has_wall": np.zeros(len(anchor), dtype=np.float32),
            }
        for col, vals in feats.items():
            if col not in df.columns:
                df[col] = 0.0
            df.loc[df.index[idx], col] = np.asarray(vals, dtype=np.float32)
    # Active contract requires these seven; make fallback semantics explicit.
    for col, default in {
        "z_above_floor": 0.0,
        "corridor_margin": 0.0,
        "wall_anomaly": 0.0,
        "floor_ang": math.pi / 2,
        "wall_ang": math.pi / 2,
        "has_floor": 0.0,
        "has_wall": 0.0,
    }.items():
        if col not in df.columns:
            df[col] = default
        df[col] = _to_numeric_series(df[col], default=default)
    return df


def _apply_directness_features(df):
    import numpy as np
    dr = _import_file_module("directness_runtime", DIRECTNESS_RUNTIME)
    df = df.copy()
    frames = _to_numeric_series(df["radar_frame_num"]).astype(int).to_numpy()
    for col in [
        "ego_expected_doppler_mps", "ego_doppler_residual_mps", "ego_residual_abs_z",
        "ego_inlier_flag", "ego_reliable", "p_ego", "p_dir_doppler",
    ]:
        if col not in df.columns:
            df[col] = 0.0
    for fn in sorted(set(frames)):
        idx = np.where(frames == fn)[0]
        if len(idx) == 0:
            continue
        records = []
        for _, row in df.iloc[idx].iterrows():
            records.append({
                "x": float(row.get("x", 0.0)),
                "y": float(row.get("y", 0.0)),
                "z": float(row.get("z", 0.0)),
                "doppler": float(row.get("doppler", 0.0)),
                "snr": float(row.get("snr", 1.0)),
            })
        ego = dr.estimate_ego_velocity_no_class(records, dr.DirectnessConfig())
        annotated = dr.annotate_directness(records, ego, dr.DirectnessConfig())
        for col in [
            "ego_expected_doppler_mps", "ego_doppler_residual_mps", "ego_residual_abs_z",
            "ego_inlier_flag", "ego_reliable", "p_ego", "p_dir_doppler",
        ]:
            vals = [a.get(col, 0.0) for a in annotated]
            df.loc[df.index[idx], col] = vals
    df["ego_inlier_flag"] = df["ego_inlier_flag"].astype(str).str.lower().isin(["1", "true", "yes", "y"]).astype(float)
    return df


def enrich_label_csv_features(csv_path: Path, *, context: str):
    import pandas as pd
    csv_path = Path(csv_path)
    if not has_data_rows(csv_path):
        raise RuntimeError(f"{context}: missing/empty label CSV: {csv_path}")
    df = pd.read_csv(csv_path, low_memory=False)
    before_cols = set(df.columns)
    df = _compute_basic_geometry_features(df)
    missing_native = sorted(NATIVE_RADAR_REQUIRED_COLS - set(df.columns))
    if missing_native:
        raise RuntimeError(f"{context}: cannot enrich; missing native radar columns {missing_native} in {csv_path}")
    for col in ["range_bin", "doppler_bin", "radar_frame_num"]:
        if df[col].isna().any() or (df[col].astype(str).str.strip() == "").any():
            raise RuntimeError(f"{context}: {col} contains blanks/NaNs; regenerate native radar CSV, do not impute bins")
    df = _apply_frame_level_features(df)
    df = _apply_soft_structural_features(df)
    df = _apply_directness_features(df)
    missing_computed = sorted(COMPUTED_PREPATCH_REQUIRED_COLS - set(df.columns))
    if missing_computed:
        raise RuntimeError(f"{context}: enrichment failed to produce {missing_computed}")
    # Make non-supervised depth defaults explicit only when these columns are absent entirely.
    # Real depth evidence is produced by depth-gated autolabeling and depth_correspondence_tools later.
    for col in ["depth_teacher_eligible", "branch1_depth_teacher_matched"]:
        if col not in df.columns:
            df[col] = 0
    if "depth_teacher_reason" not in df.columns:
        df["depth_teacher_reason"] = "not_evaluated_before_depth_correspondence"
    if "depth_teacher_source" not in df.columns:
        df["depth_teacher_source"] = "none"
    added = sorted(set(df.columns) - before_cols)
    print(f"  [ENRICH] {context}: rows={len(df)} added/filled columns: {added[:20]}{' ...' if len(added) > 20 else ''}")
    if ENRICH_LABEL_CSV_IN_PLACE:
        df.to_csv(csv_path, index=False)
    return df


def enrich_autolabel_outputs_for_root(root: Path, prefix: str, label_csv_name: str = "labeled_radar_points_v4.csv"):
    sessions = sorted(p for p in Path(root).glob(f"{prefix}*") if p.is_dir())
    problems = []
    for session_dir in sessions:
        csv_path = session_dir / label_csv_name
        try:
            enrich_label_csv_features(csv_path, context=session_dir.name)
        except Exception as exc:
            problems.append((session_dir.name, str(exc)))
    if problems:
        sample = "\n".join(f"  {s}: {why}" for s, why in problems[:20])
        raise RuntimeError("Feature enrichment failed:\n" + sample + ("\n  ..." if len(problems) > 20 else ""))


def assert_csv_has_columns(csv_path: Path, required: set[str], *, context: str):
    import pandas as pd
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"{context}: missing {csv_path}")
    df_head = pd.read_csv(csv_path, nrows=5, low_memory=False)
    missing = sorted(required - set(df_head.columns))
    if missing:
        raise RuntimeError(f"{context}: missing required columns {missing} in {csv_path}")


def assert_root_label_schema(root: Path, prefix: str, label_csv_name: str, required: set[str], *, context: str):
    problems = []
    for session_dir in sorted(Path(root).glob(f"{prefix}*")):
        csv_path = session_dir / label_csv_name
        if not has_data_rows(csv_path):
            problems.append((session_dir.name, f"missing/empty {label_csv_name}"))
            continue
        try:
            assert_csv_has_columns(csv_path, required, context=f"{context}/{session_dir.name}")
        except Exception as exc:
            problems.append((session_dir.name, str(exc)))
    if problems:
        sample = "\n".join(f"  {s}: {why}" for s, why in problems[:20])
        raise RuntimeError(f"{context}: schema check failed:\n" + sample + ("\n  ..." if len(problems) > 20 else ""))


def _cheap_fill_autolabel_computable_columns(session_dir: Path, label_csv: Path):
    """Fill columns that are deterministic from autolabel XYZ without native-row joining.

    This is specifically for v4 depth-gated autolabel outputs that already carry
    true native frame/bin provenance but omit scalar convenience columns such as
    `range_m`. We should not reject those sessions or try to match by rounded XYZ.
    """
    import numpy as np
    import pandas as pd

    label_csv = Path(label_csv)
    if not has_data_rows(label_csv):
        return None, False
    df = pd.read_csv(label_csv, low_memory=False)
    before_cols = set(df.columns)
    df = _normalize_radar_columns(df)
    changed = set(df.columns) != before_cols

    # v3 uses `v` for image-row coordinate; preserve it as camera-v if present.
    if "v" in df.columns and "cam_v_px" not in df.columns:
        df = df.rename(columns={"v": "cam_v_px"})
        changed = True

    if {"x", "y", "z"}.issubset(df.columns):
        for col in ("x", "y", "z"):
            df[col] = _to_numeric_series(df[col], default=np.nan)
        x = df["x"].to_numpy(float)
        y = df["y"].to_numpy(float)
        z = df["z"].to_numpy(float)
        computed_r = np.sqrt(x * x + y * y + z * z)
        if "range_m" not in df.columns:
            df["range_m"] = computed_r
            changed = True
        else:
            old = _to_numeric_series(df["range_m"], default=np.nan)
            new = old.where(np.isfinite(old) & (old > 0.0), computed_r)
            if not new.equals(old):
                df["range_m"] = new
                changed = True
        xy = np.hypot(x, y)
        if "azimuth_deg" not in df.columns:
            df["azimuth_deg"] = np.degrees(np.arctan2(x, y))
            changed = True
        if "elevation_deg" not in df.columns:
            df["elevation_deg"] = np.degrees(np.arctan2(z, np.maximum(xy, 1e-6)))
            changed = True
        if "z_norm_range" not in df.columns:
            df["z_norm_range"] = z / np.maximum(computed_r, 1e-6)
            changed = True

    if "session" not in df.columns or df["session"].isna().any():
        df["session"] = session_dir.name
        changed = True

    if "bucket_3class" not in df.columns and "bucket" in df.columns:
        remap = {
            "wall": "structure", "door": "structure", "pillar": "structure",
            "box_like": "structure", "floor": "floor", "human": "human",
            "structure": "structure", "person": "human",
        }
        df["bucket_3class"] = df["bucket"].astype(str).str.strip().str.lower().map(remap)
        changed = True

    if changed:
        df.to_csv(label_csv, index=False)
        filled = sorted(set(df.columns) - before_cols)
        print(f"  [AUTOLABEL-FILL] {session_dir.name}: filled computable columns {filled}")
    return df, changed


def _join_autolabel_with_native_radar(session_dir: Path, label_csv: Path, out_csv: Path) -> None:
    """
    Repair semantic autolabel outputs that do not preserve native radar provenance.

    noah_autolabel_radar_using_synccsv_v3.py emits projected semantic rows but not
    range_bin/doppler_bin/radar_frame_num. Branch 1 RD/true-RA training cannot use
    reconstructed bins, so we join each accepted semantic row back to the native
    adc_to_pointcloud_v6 CSV by (radar_frame_num, rounded x/y/z, duplicate ordinal).
    """
    import numpy as np
    import pandas as pd

    raw_csv = _find_session_radar_csv(session_dir)
    if raw_csv is None:
        raise FileNotFoundError(f"{session_dir.name}: no native radar CSV found for autolabel repair")
    labels = pd.read_csv(label_csv, low_memory=False)
    if len(labels) == 0:
        labels.to_csv(out_csv, index=False)
        return
    raw = pd.read_csv(raw_csv, low_memory=False)
    raw = _normalize_radar_columns(raw)
    labels = _normalize_radar_columns(labels)

    # v3 uses `v` for the image row; dataset_tools.py later renames it to cam_v_px.
    # Preserve it now so it is not confused with raw Doppler/velocity `v`.
    if "v" in labels.columns and "cam_v_px" not in labels.columns:
        labels = labels.rename(columns={"v": "cam_v_px"})

    required_raw = {"radar_frame_num", "x", "y", "z", "range_bin", "doppler_bin"}
    missing_raw = sorted(required_raw - set(raw.columns))
    if missing_raw:
        raise RuntimeError(f"{session_dir.name}: native radar CSV missing columns for label repair: {missing_raw}")
    required_label = {"radar_frame_num", "x", "y", "z", "bucket"}
    missing_label = sorted(required_label - set(labels.columns))
    if missing_label:
        raise RuntimeError(f"{session_dir.name}: autolabel CSV missing required semantic columns: {missing_label}")

    native_cols = [
        "radar_frame_num", "frame_num", "timestamp_us", "x", "y", "z",
        "range_m", "azimuth_deg", "elevation_deg", "doppler", "snr",
        "noise", "power_snr", "power", "quality", "range_bin", "doppler_bin",
        "chunk_index", "file_offset", "session", "clipped_ratio",
    ]
    native_cols = [c for c in native_cols if c in raw.columns]
    raw_keep = raw[native_cols].copy()
    lbl = labels.copy()

    for df in (raw_keep, lbl):
        df["radar_frame_num"] = pd.to_numeric(df["radar_frame_num"], errors="coerce").astype("Int64")
        for col in ("x", "y", "z"):
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[f"_{col}j"] = df[col].round(6)

    join_keys = ["radar_frame_num", "_xj", "_yj", "_zj"]
    raw_keep = raw_keep.dropna(subset=["radar_frame_num", "_xj", "_yj", "_zj"]).copy()
    lbl = lbl.dropna(subset=["radar_frame_num", "_xj", "_yj", "_zj"]).copy()
    raw_keep["_join_ord"] = raw_keep.groupby(join_keys, dropna=False).cumcount()
    lbl["_join_ord"] = lbl.groupby(join_keys, dropna=False).cumcount()

    raw_suffix_cols = [c for c in raw_keep.columns if c not in set(join_keys + ["_join_ord"])]
    raw_renamed = raw_keep.rename(columns={c: f"_raw_{c}" for c in raw_suffix_cols})
    merged = lbl.merge(raw_renamed, on=join_keys + ["_join_ord"], how="left", validate="one_to_one")

    matched = int(merged["_raw_range_bin"].notna().sum()) if "_raw_range_bin" in merged.columns else 0
    match_frac = matched / max(len(merged), 1)
    if match_frac < 0.95:
        raise RuntimeError(
            f"{session_dir.name}: only {matched}/{len(merged)} autolabeled rows matched native radar rows "
            f"({match_frac:.1%}). Check coordinate rounding, frame offsets, and autolabel/radar CSV pairing."
        )

    # Fill only provenance/measurement columns from the native radar CSV.
    for col in raw_suffix_cols:
        raw_col = f"_raw_{col}"
        if raw_col not in merged.columns:
            continue
        if col not in merged.columns:
            merged[col] = merged[raw_col]
        else:
            # Avoid overwriting semantic projection x/y/z unless missing; bins/timestamps always come from raw.
            if col in {"range_bin", "doppler_bin", "frame_num", "timestamp_us", "range_m", "azimuth_deg", "elevation_deg", "noise", "power_snr", "power", "quality", "chunk_index", "file_offset", "clipped_ratio"}:
                merged[col] = merged[raw_col]
            else:
                merged[col] = merged[col].where(merged[col].notna(), merged[raw_col])

    if "session" not in merged.columns or merged["session"].isna().any():
        merged["session"] = session_dir.name
    # Normalize common output names.
    merged = _normalize_radar_columns(merged)
    if "bucket_3class" not in merged.columns:
        remap = {
            "wall": "structure", "door": "structure", "pillar": "structure", "box_like": "structure",
            "floor": "floor", "human": "human", "structure": "structure", "person": "human",
        }
        merged["bucket_3class"] = merged["bucket"].astype(str).str.strip().str.lower().map(remap)
    for col in ("range_bin", "doppler_bin", "radar_frame_num"):
        merged[col] = pd.to_numeric(merged[col], errors="coerce")
    if merged[["range_bin", "doppler_bin", "radar_frame_num"]].isna().any().any():
        raise RuntimeError(f"{session_dir.name}: repaired autolabel output still has missing bin/frame provenance")

    drop_cols = [c for c in merged.columns if c.startswith("_raw_") or c in {"_xj", "_yj", "_zj", "_join_ord"}]
    merged = merged.drop(columns=drop_cols, errors="ignore")
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_csv, index=False)
    print(f"  [AUTOLABEL-REPAIR] {session_dir.name}: {len(merged)} rows, native join matched {matched}/{len(labels)}")


def _ensure_autolabel_native_provenance_for_root(root: Path, prefix: str, label_csv_name: str = "labeled_radar_points_v4.csv") -> bool:
    import pandas as pd
    problems: list[tuple[str, str]] = []
    for session_dir in active_session_dirs(root, prefix):
        label_csv = session_dir / label_csv_name
        if not has_data_rows(label_csv):
            problems.append((session_dir.name, f"missing/empty {label_csv_name}"))
            continue
        try:
            filled_df, _ = _cheap_fill_autolabel_computable_columns(session_dir, label_csv)
            head = filled_df.head(100) if filled_df is not None else pd.read_csv(label_csv, nrows=100, low_memory=False)

            # Only missing/bad hard provenance requires native-row repair.
            # `range_m` is deterministic from XYZ and is filled above; treating it
            # as a join trigger caused valid v4 depth-gated rows to be skipped.
            missing_hard = sorted(AUTOLABEL_HARD_PROVENANCE_COLS - set(head.columns))
            bad_values = []
            for col in sorted({"range_bin", "doppler_bin", "radar_frame_num"} & set(head.columns)):
                vals = pd.to_numeric(head[col], errors="coerce")
                if vals.isna().any():
                    bad_values.append(col)

            # If doppler/snr are absent, try native repair too because those are
            # raw measurement columns used by the scalar feature contract.
            missing_measurements = sorted({"doppler", "snr"} - set(head.columns))
            if missing_hard or bad_values or missing_measurements:
                print(
                    f"  [AUTOLABEL-REPAIR] {session_dir.name}: "
                    f"hard_missing={missing_hard} measurement_missing={missing_measurements} bad={bad_values}"
                )
                _join_autolabel_with_native_radar(session_dir, label_csv, label_csv)
        except Exception as exc:
            problems.append((session_dir.name, str(exc)))
    return _handle_stage_problems(root, prefix, "autolabel-provenance", problems)



def _audit_numeric_feature_ranges(df, cols: set[str], *, context: str, low_high: dict[str, tuple[float, float]] | None = None) -> list[str]:
    import numpy as np
    import pandas as pd
    problems: list[str] = []
    low_high = low_high or {}
    for col in sorted(cols):
        if col not in df.columns:
            problems.append(f"missing column {col}")
            continue
        vals = pd.to_numeric(df[col], errors="coerce")
        finite = vals.replace([np.inf, -np.inf], np.nan).notna()
        if not finite.any():
            problems.append(f"column {col} has no finite values")
            continue
        if col in low_high:
            lo, hi = low_high[col]
            bad = vals[finite].lt(lo).sum() + vals[finite].gt(hi).sum()
            if int(bad) > 0:
                problems.append(f"column {col} has {int(bad)} value(s) outside [{lo}, {hi}]")
    return problems


def audit_directness_and_ghost_features_for_root(
    root: Path,
    prefix: str,
    label_csv_name: str,
    *,
    context: str,
    require_spatiotemporal: bool = False,
    require_teacher: bool = False,
) -> bool:
    """Validate the directness-aware ghost-return feature path without inventing missing columns."""
    if not AUDIT_DIRECTNESS_AND_GHOST_FEATURES:
        return True
    import numpy as np
    import pandas as pd

    problems: list[tuple[str, str]] = []
    aggregate_rows = 0
    summaries: list[str] = []
    for session_dir in active_session_dirs(root, prefix):
        csv_path = session_dir / label_csv_name
        if not has_data_rows(csv_path):
            problems.append((session_dir.name, f"missing/empty {label_csv_name}"))
            continue
        try:
            df = pd.read_csv(csv_path, low_memory=False)
            required = set(DIRECTNESS_MODEL_FEATURE_COLS)
            if require_spatiotemporal:
                required |= set(SPATIOTEMPORAL_GHOST_FEATURE_COLS)
            if require_teacher:
                required |= set(GHOST_TEACHER_FEATURE_COLS)
            missing = sorted(required - set(df.columns))
            if missing:
                raise RuntimeError(f"missing required directness/ghost columns: {missing}")
            numeric_cols = set(DIRECTNESS_MODEL_FEATURE_COLS)
            if require_spatiotemporal:
                numeric_cols |= {"spatiotemporal_weight", "temporal_persistence_frames", "ghost_score", "static_confidence", "final_train_weight"}
            if require_teacher:
                numeric_cols |= {"ghost_score_combined", "depth_teacher_weight"}
            range_checks = {
                "p_ego": (0.0, 1.0),
                "p_dir_doppler": (0.0, 1.0),
                "ego_inlier_flag": (0.0, 1.0),
                "spatiotemporal_weight": (0.0, 1.0),
                "ghost_score": (0.0, 1.0),
                "static_confidence": (0.0, 1.0),
                "final_train_weight": (0.0, 1.0),
                "ghost_score_combined": (0.0, 1.0),
                "depth_teacher_weight": (0.0, 1.0),
            }
            subproblems = _audit_numeric_feature_ranges(df, numeric_cols, context=f"{context}/{session_dir.name}", low_high=range_checks)
            if subproblems:
                raise RuntimeError("; ".join(subproblems))

            aggregate_rows += len(df)
            if len(df):
                p_dir = pd.to_numeric(df["p_dir_doppler"], errors="coerce")
                p_ego = pd.to_numeric(df["p_ego"], errors="coerce")
                ghost = pd.to_numeric(df["ghost_score"], errors="coerce") if "ghost_score" in df.columns else None
                msg = (
                    f"{session_dir.name}: rows={len(df)} "
                    f"p_dir(mean={float(p_dir.mean(skipna=True)):.3f}, min={float(p_dir.min(skipna=True)):.3f}, max={float(p_dir.max(skipna=True)):.3f}) "
                    f"p_ego(mean={float(p_ego.mean(skipna=True)):.3f})"
                )
                if ghost is not None:
                    msg += f" ghost_score(mean={float(ghost.mean(skipna=True)):.3f}, max={float(ghost.max(skipna=True)):.3f})"
                summaries.append(msg)
        except Exception as exc:
            problems.append((session_dir.name, str(exc)))

    if summaries:
        print(f"\n[{context}] directness/ghost feature audit ({aggregate_rows} rows):")
        for line in summaries[:12]:
            print("  ", line)
        if len(summaries) > 12:
            print(f"  ... {len(summaries) - 12} more session summaries omitted")
    return _handle_stage_problems(root, prefix, context, problems)


def _assert_radar_tensor_npz_has_complex_rd(session_dir: Path) -> tuple[bool, str]:
    import numpy as np
    tensor_path = session_dir / f"{session_dir.name}_radar_tensors.npz"
    if not tensor_path.exists():
        return False, f"missing {tensor_path.name}"
    try:
        with np.load(tensor_path) as payload:
            keys = [k for k in payload.files if k.startswith("rd_")]
            if not keys:
                return False, f"{tensor_path.name} has no rd_* arrays"
            checked = 0
            shapes = []
            for key in keys[:max(1, RA_TENSOR_AUDIT_MAX_ARRAYS)]:
                arr = np.asarray(payload[key])
                shapes.append((key, arr.shape, str(arr.dtype)))
                if arr.size == 0:
                    return False, f"{tensor_path.name}:{key} is empty"
                if not np.iscomplexobj(arr):
                    return False, f"{tensor_path.name}:{key} is {arr.dtype}, expected complex rd_cube tensor for true RA"
                if arr.ndim < 3:
                    return False, f"{tensor_path.name}:{key} shape={arr.shape}, expected at least 3-D rd_cube tensor"
                checked += 1
        return True, f"ok arrays={len(keys)} checked={checked} sample_shapes={shapes[:2]}"
    except Exception as exc:
        return False, f"could not read {tensor_path.name}: {exc}"


def audit_radar_tensors_for_root(root: Path, prefix: str) -> bool:
    if not AUDIT_RA_TENSORS_AND_PATCHES:
        return True
    problems: list[tuple[str, str]] = []
    ok_lines: list[str] = []
    for session_dir in active_session_dirs(root, prefix):
        if not has_data_rows(session_dir / "labeled_radar_points_v4.csv"):
            continue
        ok, msg = _assert_radar_tensor_npz_has_complex_rd(session_dir)
        if not ok:
            problems.append((session_dir.name, msg))
        else:
            ok_lines.append(f"{session_dir.name}: {msg}")
    if ok_lines:
        print("\n[true-RA tensor audit]")
        for line in ok_lines[:10]:
            print("  ", line)
        if len(ok_lines) > 10:
            print(f"  ... {len(ok_lines) - 10} more tensor summaries omitted")
    return _handle_stage_problems(root, prefix, "true-ra-tensor-audit", problems)


def audit_ra_patch_shards(dataset_root: Path, csv_path: Path, *, context: str) -> None:
    if not AUDIT_RA_TENSORS_AND_PATCHES:
        return
    import numpy as np
    import pandas as pd
    dataset_root = Path(dataset_root)
    df = pd.read_csv(csv_path, low_memory=False)
    if "ra_patch_valid" not in df.columns or "ra_patch_shard" not in df.columns:
        raise RuntimeError(f"{context}: missing RA patch columns")
    valid = df[df["ra_patch_valid"].astype(str).str.lower().isin({"1", "true", "t", "yes", "y"})]
    if len(valid) == 0:
        raise RuntimeError(f"{context}: no valid RA patches")
    shard_rel = str(valid["ra_patch_shard"].iloc[0])
    shard_path = dataset_root / shard_rel
    if not shard_path.exists():
        raise RuntimeError(f"{context}: first RA shard not found: {shard_path}")
    with np.load(shard_path) as payload:
        if "ra_patches" not in payload.files:
            raise RuntimeError(f"{context}: {shard_path.name} is missing ra_patches")
        ra = np.asarray(payload["ra_patches"])
        if ra.size == 0 or ra.ndim < 3:
            raise RuntimeError(f"{context}: invalid ra_patches shape={ra.shape}")
        finite_rate = float(np.isfinite(ra).mean()) if ra.size else 0.0
        if finite_rate < 1.0:
            raise RuntimeError(f"{context}: ra_patches finite_rate={finite_rate:.6f}")
    print(f"[{context}] RA patch audit: valid_rows={len(valid)}/{len(df)} first_shard={shard_rel} ra_shape={ra.shape}")


def _run_v3_autolabel_for_root(root: Path, prefix: str) -> bool:
    """Fallback v3 autolabel path. Runs in-process per group when possible.

    v3 itself is a session-oriented script, but this wrapper imports the module once
    and loops through the staged sessions without spawning a Python process or
    reinitializing module state for every session.
    """
    sessions = active_session_dirs(root, prefix)
    problems: list[tuple[str, str]] = []
    if not sessions:
        return False
    if not BATCHWISE_V3_AUTOLABEL:
        # Debug-only legacy mode: keep the old subprocess behavior.
        for idx, session_dir in enumerate(sessions, start=1):
            print(f"\n[AUTOLABEL v3 subprocess {idx}/{len(sessions)}] {session_dir.name}")
            try:
                radar_csv = _find_session_radar_csv(session_dir)
                if radar_csv is None:
                    raise FileNotFoundError("missing native radar CSV")
                sync_csv = session_dir / f"synchronized_{session_dir.name}.csv"
                if not sync_csv.exists():
                    raise FileNotFoundError(f"missing {sync_csv.name}")
                meta_json = session_dir / "meta_data.json"
                if not meta_json.exists():
                    raise FileNotFoundError("missing meta_data.json")
                tmp_csv = session_dir / "labeled_radar_points_v3.csv"
                run([
                    PY, ONEFORMER_AUTOLABEL,
                    "--processing_session_dir", session_dir,
                    "--sync_csv", sync_csv,
                    "--radar_csv", radar_csv,
                    "--meta_json", meta_json,
                    "--extrinsics_json", EXTRINSICS_JSON,
                    "--out_csv", tmp_csv,
                ])
                _join_autolabel_with_native_radar(session_dir, tmp_csv, session_dir / "labeled_radar_points_v4.csv")
            except Exception as exc:
                problems.append((session_dir.name, str(exc)))
        return _handle_stage_problems(root, prefix, "autolabel-v3", problems)

    print(f"\n[AUTOLABEL v3 BATCH] root={root} sessions={len(sessions)}")
    try:
        import pandas as pd
        import numpy as np
        from tqdm import tqdm
        v3 = _import_file_module("noah_autolabel_radar_using_synccsv_v3", ONEFORMER_AUTOLABEL)
        K_cache: dict[Path, object] = {}
        T_cr = v3.load_Tcr_from_extrinsics(EXTRINSICS_JSON)
    except Exception as exc:
        raise RuntimeError(f"Could not initialize v3 batch autolabeler: {exc}") from exc

    for idx, session_dir in enumerate(sessions, start=1):
        print(f"\n[AUTOLABEL v3 BATCH {idx}/{len(sessions)}] {session_dir.name}")
        try:
            radar_csv = _find_session_radar_csv(session_dir)
            if radar_csv is None:
                raise FileNotFoundError("missing native radar CSV")
            sync_csv = session_dir / f"synchronized_{session_dir.name}.csv"
            if not sync_csv.exists():
                raise FileNotFoundError(f"missing {sync_csv.name}")
            meta_json = session_dir / "meta_data.json"
            if not meta_json.exists():
                raise FileNotFoundError("missing meta_data.json")
            seg_dir = session_dir / "seg"
            if not seg_dir.exists():
                raise FileNotFoundError("missing seg directory")
            id2label = v3.load_seg_meta(session_dir)
            K = K_cache.get(meta_json)
            if K is None:
                K = v3.load_intrinsics_from_meta(meta_json)
                K_cache[meta_json] = K
            sync = pd.read_csv(sync_csv)
            radar = pd.read_csv(radar_csv)
            rcol = v3.pick_radar_columns(radar)
            radar[rcol["frame"]] = radar[rcol["frame"]].astype(int)
            radar_by_frame = {k: v for k, v in radar.groupby(rcol["frame"])}
            out_rows = []
            for _, srow in tqdm(sync.iterrows(), total=len(sync), desc=f"v3 autolabel {session_dir.name}"):
                if "status" in sync.columns and str(srow.get("status", "")).strip() not in {"", "matched"}:
                    continue
                vfi = int(srow["video_frame_index"])
                seg_path = seg_dir / f"{vfi:06d}.npy"
                if not seg_path.exists():
                    continue
                label_map = np.load(seg_path).astype(np.int32)
                H, W = label_map.shape
                roi_v_min = int(H * 0.35)
                radar_frames = v3.parse_semicolon_int_list(srow["radar_frame_nums"])
                for rf in radar_frames:
                    grp = radar_by_frame.get(rf)
                    if grp is None or len(grp) == 0:
                        continue
                    pts = grp[[rcol["x"], rcol["y"], rcol["z"]]].to_numpy(dtype=np.float32)
                    uv, zc = v3.project_points(pts, K, T_cr)
                    for i in range(len(pts)):
                        u_f, v_f = uv[i]
                        if not (np.isfinite(u_f) and np.isfinite(v_f)):
                            continue
                        ui, vi = int(round(u_f)), int(round(v_f))
                        if not (0 <= ui < W and 0 <= vi < H):
                            continue
                        lid, maj = v3.majority_label(label_map, ui, vi, r=1)
                        if maj < 0.60:
                            continue
                        ade_name = id2label.get(str(lid), str(lid))
                        if v3.should_ignore_ade(ade_name):
                            continue
                        if vi < roi_v_min and not v3.is_person_label(ade_name):
                            continue
                        rec = {
                            "video_frame_index": vfi,
                            "radar_frame_num": int(rf),
                            "u": ui,
                            "v": vi,
                            "ade_id": int(lid),
                            "ade_name": ade_name,
                            "bucket": v3.map_bucket(ade_name),
                            "maj_frac": round(float(maj), 4),
                            "x": round(float(pts[i, 0]), 6),
                            "y": round(float(pts[i, 1]), 6),
                            "z": round(float(pts[i, 2]), 6),
                            "cam_z": round(float(zc[i]), 5),
                        }
                        if rcol["doppler"] is not None:
                            rec["doppler"] = float(grp.iloc[i][rcol["doppler"]])
                        if rcol["snr"] is not None:
                            rec["snr"] = float(grp.iloc[i][rcol["snr"]])
                        out_rows.append(rec)
            tmp_csv = session_dir / "labeled_radar_points_v3.csv"
            pd.DataFrame(out_rows).to_csv(tmp_csv, index=False)
            if not has_data_rows(tmp_csv):
                raise RuntimeError("v3 autolabel produced no labeled rows")
            _join_autolabel_with_native_radar(session_dir, tmp_csv, session_dir / "labeled_radar_points_v4.csv")
        except Exception as exc:
            problems.append((session_dir.name, str(exc)))
    return _handle_stage_problems(root, prefix, "autolabel-v3-batch", problems)

def run_depth_gated_autolabel_for_root(root: Path, prefix: str) -> bool:
    if ONEFORMER_AUTOLABEL_KIND == "v4_depth_gated":
        try:
            run([
                PY, ONEFORMER_AUTOLABEL,
                "--processing_root_dir", root,
                "--synchronized_root_dir", root,
                "--extrinsics_json", EXTRINSICS_JSON,
                "--session_glob", f"{prefix}*",
                "--assign_mode", "best_per_radar",
                "--max_time_diff_ms", MAX_TIME_DIFF_MS,
                "--depth_occlusion_m", DEPTH_OCCLUSION_M,
                "--depth_patch_r", DEPTH_PATCH_R,
            ])
        except Exception as exc:
            # Root-level v4 tools sometimes abort on a single bad session without exposing
            # a clean per-session status. At this point earlier stages have already removed
            # obvious corrupt sessions, so keep the failure explicit.
            if SKIP_CORRUPT_SESSIONS:
                print("[AUTOLABEL v4] root command failed. Falling back to post-run provenance scan; sessions without usable labels will be skipped.")
                print("  error:", exc)
            else:
                raise
    else:
        print("Using v3 semantic autolabel fallback; depth evidence will be added by depth_correspondence_tools + ghost_teacher_fusion.")
        if not _run_v3_autolabel_for_root(root, prefix):
            return False
    return _ensure_autolabel_native_provenance_for_root(root, prefix, "labeled_radar_points_v4.csv")

def ensure_radar_tensors(root: Path, prefix: str) -> bool:
    missing = []
    for session_dir in active_session_dirs(root, prefix):
        if not has_data_rows(session_dir / "labeled_radar_points_v4.csv"):
            continue
        tensor_path = session_dir / f"{session_dir.name}_radar_tensors.npz"
        if not tensor_path.exists():
            missing.append(session_dir.name)
    if missing:
        print(f"Exporting radar tensors / RD cube sidecars for {len(missing)} session(s).")
        try:
            cmd: list[object] = [
                PY, RADAR_TENSOR_EXPORT,
                "--data-root", root,
                "--session-prefix", prefix,
                "--cfg-path", CFG_PATH,
            ]
            help_text = command_help_text([PY, RADAR_TENSOR_EXPORT])
            append_supported_flag(cmd, help_text, "--workers", str(int(RADAR_TENSOR_WORKERS)))
            append_supported_flag(cmd, help_text, "--npz-compression", RADAR_TENSOR_NPZ_COMPRESSION)
            append_supported_flag(cmd, help_text, "--sidecar-strip-mode", RADAR_TENSOR_SIDECAR_STRIP_MODE)
            append_supported_flag(cmd, help_text, "--sidecar-strip-compression", RADAR_TENSOR_SIDECAR_STRIP_COMPRESSION)
            if RADAR_TENSOR_OVERWRITE and "--overwrite" in help_text:
                cmd.append("--overwrite")
            if RADAR_TENSOR_QUIET and "--quiet" in help_text:
                cmd.append("--quiet")
            run(cmd)
        except Exception as exc:
            print("[RADAR TENSOR] export command failed; sessions still missing tensors will be skipped.")
            print("  error:", exc)
    still_missing: list[tuple[str, str]] = []
    for session_dir in active_session_dirs(root, prefix):
        if not has_data_rows(session_dir / "labeled_radar_points_v4.csv"):
            still_missing.append((session_dir.name, "missing/empty labeled_radar_points_v4.csv before tensor export"))
            continue
        if not (session_dir / f"{session_dir.name}_radar_tensors.npz").exists():
            still_missing.append((session_dir.name, f"missing {session_dir.name}_radar_tensors.npz required for true RA"))
    if not _handle_stage_problems(root, prefix, "radar-tensor-export", still_missing):
        return False
    return audit_radar_tensors_for_root(root, prefix)

def assert_preprocessed_sessions(root: Path, prefix: str) -> bool:
    problems: list[tuple[str, str]] = []
    for session_dir in active_session_dirs(root, prefix):
        if not has_data_rows(session_dir / "labeled_radar_points_v4.csv"):
            problems.append((session_dir.name, "missing/empty labeled_radar_points_v4.csv"))
        if not (session_dir / "seg_timestamps.csv").exists():
            problems.append((session_dir.name, "missing seg_timestamps.csv from OneFormer export"))
        if not (session_dir / f"synchronized_{session_dir.name}.csv").exists():
            problems.append((session_dir.name, "missing synchronized_{session}.csv"))
        tensor_path = session_dir / f"{session_dir.name}_radar_tensors.npz"
        if not _has_rd_sidecar_manifest(session_dir):
            problems.append((session_dir.name, "missing hybrid_rd/manifest.json or RD sidecar frames"))
        if not tensor_path.exists():
            problems.append((session_dir.name, f"missing {session_dir.name}_radar_tensors.npz required for true RA"))
    return _handle_stage_problems(root, prefix, "final-preprocess-contract", problems)

def preprocess_group_root(root: Path, prefix: str) -> bool:
    # Correct raw-session ordering:
    #   1. radar CSV + camera metadata, independent of segmentation
    #   2. OneFormer segmentation export
    #   3. synchronized_{session}.csv + seg_meta.json
    #   4. depth-gated radar autolabeling
    #   5. rd_cube / true-RA sidecars
    stages = [
        ("radar/meta", lambda: run_radar_csv_and_meta_stage(root, prefix)),
        ("oneformer", lambda: run_oneformer_for_root(root, prefix)),
        ("sync/seg-meta", lambda: run_sync_and_seg_meta_stage(root, prefix)),
        ("autolabel", lambda: run_depth_gated_autolabel_for_root(root, prefix)),
    ]
    for stage_name, fn in stages:
        if not active_session_dirs(root, prefix):
            print(f"[{stage_name}] no sessions remain; skipping group {root.name}")
            return False
        ok = fn()
        if not ok or not active_session_dirs(root, prefix):
            print(f"[{stage_name}] no usable sessions remain in {root.name}; skipping this group.")
            return False

    try:
        enrich_autolabel_outputs_for_root(root, prefix, "labeled_radar_points_v4.csv")
        assert_root_label_schema(
            root, prefix, "labeled_radar_points_v4.csv",
            NATIVE_RADAR_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS,
            context="post-autolabel feature contract",
        )
        if not audit_directness_and_ghost_features_for_root(
            root, prefix, "labeled_radar_points_v4.csv",
            context="post-autolabel-directness-audit",
            require_spatiotemporal=False,
            require_teacher=False,
        ):
            return False
    except Exception as exc:
        # This is a root-wide schema check. Determine bad sessions by checking each label CSV.
        problems = []
        for session_dir in active_session_dirs(root, prefix):
            try:
                assert_csv_has_columns(
                    session_dir / "labeled_radar_points_v4.csv",
                    NATIVE_RADAR_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS,
                    context=f"{session_dir.name} post-autolabel feature contract",
                )
            except Exception as sub_exc:
                problems.append((session_dir.name, str(sub_exc)))
        if not problems:
            if SKIP_CORRUPT_SESSIONS:
                print("[post-autolabel feature contract] root check failed but per-session checks passed; continuing.")
                print("  root error:", exc)
            else:
                raise
        elif not _handle_stage_problems(root, prefix, "post-autolabel-feature-contract", problems):
            return False

    if not ensure_radar_tensors(root, prefix):
        return False
    if not assert_preprocessed_sessions(root, prefix):
        return False
    n_remaining = len(active_session_dirs(root, prefix))
    if n_remaining < MIN_VALID_SESSIONS_PER_GROUP_AFTER_SKIP:
        print(f"Only {n_remaining} usable session(s) remain in {root.name}; skipping this group.")
        return False
    return True

# -----------------------------
# Ghost/depth fusion + RD/RA patch dataset stages
# -----------------------------

GHOST_LABEL_CSV_NAME = "labeled_radar_points_v4_fused.csv"

def summarize_label_file(root: Path, prefix: str, label_csv_name: str):
    try:
        import pandas as pd
    except Exception:
        return
    rows = 0
    accum_counts = {}
    reason_counts = {}
    cols_seen = set()
    for session_dir in sorted(root.glob(f"{prefix}*")):
        csv_path = session_dir / label_csv_name
        if not has_data_rows(csv_path):
            continue
        df = pd.read_csv(csv_path, low_memory=False)
        rows += len(df)
        cols_seen.update(df.columns)
        if "accumulatable_label" in df.columns:
            for k, v in df["accumulatable_label"].astype(str).str.lower().value_counts().to_dict().items():
                accum_counts[k] = accum_counts.get(k, 0) + int(v)
        if "depth_teacher_reason" in df.columns:
            for k, v in df["depth_teacher_reason"].astype(str).value_counts().head(12).to_dict().items():
                reason_counts[k] = reason_counts.get(k, 0) + int(v)
    ghost_cols = sorted(c for c in cols_seen if c in {
        "ghost_score", "ghost_score_combined", "static_confidence",
        "ego_doppler_residual_mps", "depth_teacher_acc_label",
        "depth_teacher_weight", "accumulatable_label", "return_type_label",
    })
    print(f"fused label source: root={root} label={label_csv_name} rows={rows}")
    print("ghost/evidence columns:", ghost_cols)
    print("accumulatable counts:", accum_counts)
    print("depth teacher reasons:", reason_counts)

def materialize_ghost_teacher_processing_root(processing_root: Path, prefix: str, dataset_tag: str) -> tuple[Path, str]:
    sessions = sorted(p for p in processing_root.glob(f"{prefix}*") if p.is_dir())
    if not sessions:
        raise RuntimeError(f"No sessions found for ghost-teacher labeling: {processing_root} {prefix}")

    print(f"Ghost/depth teacher stage for {dataset_tag}: {len(sessions)} session(s)")
    run([
        PY, SPATIOTEMPORAL_TOOLS, "fuse-labels",
        "--processing-root", processing_root,
        "--in-place",
        "--session-prefix", prefix,
        "--output-csv-name", GHOST_LABEL_CSV_NAME,
    ])
    # Ensure the fused file kept the real prepatch feature contract; do not let downstream
    # tools silently replace missing feature columns with zeros/blanks.
    assert_root_label_schema(
        processing_root, prefix, GHOST_LABEL_CSV_NAME,
        NATIVE_RADAR_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS | SPATIOTEMPORAL_GHOST_FEATURE_COLS,
        context=f"{dataset_tag} fused pre-depth contract",
    )
    audit_directness_and_ghost_features_for_root(
        processing_root, prefix, GHOST_LABEL_CSV_NAME,
        context=f"{dataset_tag}-spatiotemporal-ghost-audit",
        require_spatiotemporal=True,
        require_teacher=False,
    )

    ghost_stage_root = DATASET_WORK_ROOT / f"ghost_teacher_{dataset_tag}"
    depth_root = ghost_stage_root / "depth_correspondence"
    fused_root = ghost_stage_root / "processing"
    shutil.rmtree(depth_root, ignore_errors=True)
    shutil.rmtree(fused_root, ignore_errors=True)

    depth_cmd: list[object] = [
        PY, DEPTH_CORRESPONDENCE_TOOLS, "annotate-root",
        "--processing-root", processing_root,
        "--out-root", depth_root,
        "--session-glob", f"{prefix}*",
        "--max-time-diff-ms", MAX_TIME_DIFF_MS,
        "--foreground-occlusion-m", DEPTH_OCCLUSION_M,
        "--depth-patch-r", DEPTH_PATCH_R,
    ]
    if EXTRINSICS_JSON.exists():
        depth_cmd.extend(["--extrinsics-json", EXTRINSICS_JSON])
    run(depth_cmd)

    depth_summary_csv = depth_root / "depth_correspondence_summary.csv"
    if depth_summary_csv.exists():
        try:
            import pandas as pd
            depth_summary = pd.read_csv(depth_summary_csv)
            keep = [c for c in [
                "session", "status", "depth_teacher_eligible", "failure_reason",
                "projection_valid_rate", "depth_available_rate", "depth_match_rate",
                "ok_points", "points", "median_abs_depth_residual_m",
                "time_match_p95_ms", "calibration_error",
                "extrinsics_source", "intrinsics_source",
            ] if c in depth_summary.columns]
            print("\nDepth correspondence / teacher QA:")
            print(depth_summary[keep].to_string(index=False))
        except Exception as exc:
            print("Could not summarize depth correspondence QA:", exc)
    else:
        print("Depth correspondence summary missing:", depth_summary_csv)

    fusion_help = command_help_text([PY, GHOST_TEACHER_FUSION, "fuse-root"])
    fusion_cmd: list[object] = [
        PY, GHOST_TEACHER_FUSION, "fuse-root",
        "--processing-root", processing_root,
        "--depth-root", depth_root,
        "--out-root", fused_root,
        "--session-glob", f"{prefix}*",
        "--label-csv-name", GHOST_LABEL_CSV_NAME,
    ]
    if COPY_GHOST_ASSETS and "--copy-assets" in fusion_help:
        fusion_cmd.append("--copy-assets")
    run(fusion_cmd)
    assert_root_label_schema(
        fused_root, prefix, GHOST_LABEL_CSV_NAME,
        NATIVE_RADAR_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS | DEPTH_TEACHER_EXPECTED_COLS | SPATIOTEMPORAL_GHOST_FEATURE_COLS | GHOST_TEACHER_FEATURE_COLS,
        context=f"{dataset_tag} ghost/depth teacher contract",
    )
    audit_directness_and_ghost_features_for_root(
        fused_root, prefix, GHOST_LABEL_CSV_NAME,
        context=f"{dataset_tag}-ghost-depth-teacher-audit",
        require_spatiotemporal=True,
        require_teacher=True,
    )
    summarize_label_file(fused_root, prefix, GHOST_LABEL_CSV_NAME)
    return fused_root, GHOST_LABEL_CSV_NAME

def build_patches_cmd(points_csv: Path, processing_root: Path, patch_dir: Path, prefix: str) -> list[object]:
    cmd: list[object] = [
        PY, DATASET_TOOLS, "build-patches",
        "--point-csv", points_csv,
        "--sidecar-root", processing_root,
        "--output-dir", patch_dir,
        "--frame-number-offset", str(RD_PATCH_FRAME_NUMBER_OFFSET),
        "--drop-invalid",
        "--require-all-valid",
        "--allowed-session-prefix", prefix,
        "--extract-rd-scalars",
    ]
    help_text = command_help_text([PY, DATASET_TOOLS, "build-patches"])
    # Make RA explicit when supported. Older repo variants may expose only some of these flags.
    for flag, value in [
        ("--ra-patch-source", "true_ra"),
        ("--ra-source", "true_ra"),
        ("--ra-patch-az-mode", "tx0_only"),
        ("--ra-patch-az-fft-size", "64"),
        ("--ra-patch-row-edge-mode", "zero_pad"),
    ]:
        append_supported_flag(cmd, help_text, flag, value)
    for flag in ["--ra-patch-log-scale", "--ra-patch-normalize-by-frame-max"]:
        append_supported_flag(cmd, help_text, flag)
    return cmd

def assert_ra_patch_dataset(csv_path: Path, context: str):
    with open(csv_path, "r", newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        fields = set(reader.fieldnames or [])
        missing = sorted({"rd_patch_shard", "rd_patch_index", "rd_patch_valid",
                          "ra_patch_shard", "ra_patch_index", "ra_patch_valid"} - fields)
        if missing:
            raise RuntimeError(f"{context} is missing patch columns: {missing}")
        rows = 0
        rd_valid = 0
        ra_valid = 0
        bin_ok = 0
        for row in reader:
            rows += 1
            rd_valid += int(str(row.get("rd_patch_valid", "")).strip().lower() in {"1", "true", "t", "yes", "y"})
            ra_valid += int(str(row.get("ra_patch_valid", "")).strip().lower() in {"1", "true", "t", "yes", "y"})
            bin_ok += int(row.get("range_bin", "") not in {"", "nan", "None"} and row.get("doppler_bin", "") not in {"", "nan", "None"})
    if rows == 0 or rd_valid == 0 or ra_valid == 0:
        raise RuntimeError(f"{context} has invalid patches: rows={rows}, rd_valid={rd_valid}, ra_valid={ra_valid}")
    if bin_ok == 0:
        raise RuntimeError(f"{context} has no usable range_bin/doppler_bin provenance.")
    print(f"{context}: rows={rows}, valid RD={rd_valid}, valid RA={ra_valid}, bin rows={bin_ok}")

def build_source_dataset(processing_root: Path, prefix: str, dataset_tag: str, holdout_last: int = PER_BATCH_HOLDOUT_LAST) -> Path:
    label_root, label_csv_name = materialize_ghost_teacher_processing_root(processing_root, prefix, dataset_tag)

    n_sessions = len([p for p in Path(label_root).glob(f"{prefix}*") if p.is_dir()])
    if n_sessions < 2:
        raise RuntimeError(
            f"{dataset_tag}: build-points needs at least 2 sessions with the current dataset_tools.py split implementation; "
            f"found {n_sessions}. Increase RAW_PREPROCESS_BATCH_SIZE or merge this small group with another batch."
        )
    effective_holdout_last = max(1, min(int(holdout_last), n_sessions - 1))
    if effective_holdout_last != int(holdout_last):
        print(f"  [SPLIT] using temporary per-batch holdout_last={effective_holdout_last} for {n_sessions} sessions")

    points_dir = DATASET_WORK_ROOT / f"branch1_{dataset_tag}_directness_true_ra_v1" / "points"
    patch_dir = DATASET_WORK_ROOT / f"branch1_{dataset_tag}_directness_true_ra_v1" / "rdra_patch_dataset"
    four_dir = DATASET_WORK_ROOT / f"branch1_4class_depth_directness_{dataset_tag}_true_ra_v1" / "rdra_patch_dataset"
    points_dir.mkdir(parents=True, exist_ok=True)
    patch_dir.mkdir(parents=True, exist_ok=True)
    four_dir.mkdir(parents=True, exist_ok=True)

    run([
        PY, DATASET_TOOLS, "build-points",
        "--processing-root", label_root,
        "--output-dir", points_dir,
        "--holdout-last", str(int(effective_holdout_last)),
        "--session-prefix", prefix,
        "--labeled-csv-name", label_csv_name,
    ])

    run(build_patches_cmd(points_dir / "points.csv", label_root, patch_dir, prefix))
    raw_patch_csv = patch_dir / "points_with_rd_patch_index.csv"
    assert_ra_patch_dataset(raw_patch_csv, f"{dataset_tag} raw patch dataset")
    audit_ra_patch_shards(patch_dir, raw_patch_csv, context=f"{dataset_tag} raw patch dataset")

    label_help = command_help_text([PY, FOURCLASS_TOOLS, "label-dataset"])
    label_cmd: list[object] = [
        PY, FOURCLASS_TOOLS, "label-dataset",
        "--source-csv", patch_dir / "points_with_rd_patch_index.csv",
        "--source-root", patch_dir,
        "--output-dir", four_dir,
        "--allowed-session-prefix", prefix,
        "--depth-teacher-csv", points_dir / "points.csv",
    ]
    if NO_HUMAN_SESSION_PREFIXES and "--no-human-session-prefix" in label_help:
        for no_human_prefix in NO_HUMAN_SESSION_PREFIXES:
            label_cmd.extend(["--no-human-session-prefix", str(no_human_prefix)])
    if NO_HUMAN_VAL_SESSIONS and "--no-human-val-session" in label_help:
        for sess_name in sorted(NO_HUMAN_VAL_SESSIONS):
            label_cmd.extend(["--no-human-val-session", str(sess_name)])
    append_supported_flag(label_cmd, label_help, "--use-accumulatable-no")
    run(label_cmd)
    final_csv = four_dir / "points_with_rd_patch_index.csv"
    assert_ra_patch_dataset(final_csv, f"{dataset_tag} 4-class patch dataset")
    audit_ra_patch_shards(four_dir, final_csv, context=f"{dataset_tag} 4-class patch dataset")
    assert_csv_has_columns(
        final_csv,
        FINAL_PATCH_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS | SPATIOTEMPORAL_GHOST_FEATURE_COLS | GHOST_TEACHER_FEATURE_COLS | {"point_label_weight"},
        context=f"{dataset_tag} final Branch 1 dataset contract",
    )
    return four_dir

# -----------------------------
# Final combined dataset
# -----------------------------

def rewrite_patch_paths(df, patch_tag: str):
    df = df.copy()
    df["rd_patch_shard"] = df["rd_patch_shard"].astype(str).str.replace("patches/", f"patches_{patch_tag}/", regex=False)
    if "ra_patch_shard" in df.columns:
        df["ra_patch_shard"] = df["ra_patch_shard"].astype(str).str.replace("patches/", f"patches_{patch_tag}/", regex=False)
    return df

def _choose_last_n_sessions(sessions: list[str], requested: int, fraction_if_small: float, *, reserve: int = 0) -> set[str]:
    sessions = sorted(str(s) for s in sessions)
    n = len(sessions)
    if n <= int(reserve):
        return set()
    if requested > 0 and n > requested + reserve:
        k = requested
    elif fraction_if_small > 0 and n > reserve + 1:
        k = max(1, int(round(n * float(fraction_if_small))))
    else:
        k = 0
    k = max(0, min(int(k), n - int(reserve)))
    return set(sessions[-k:]) if k > 0 else set()

def assign_train_val_external_splits(df, group_col: str = "_raw_group_tag"):
    """
    Assign three evaluation partitions without leaking external holdout sessions
    into training:

      eval_split=train              -> model fitting
      eval_split=internal_val       -> early stopping / model selection
      eval_split=external_holdout   -> final post-training evaluation only

    The canonical training CSV later maps train/internal_val back onto
    split=train/val and excludes external_holdout rows entirely.
    """
    df = df.copy()
    if "session" not in df.columns:
        print("No session column; leaving all rows as train.")
        df["eval_split"] = "train"
        df["split"] = "train"
        return df, {"train_sessions": [], "internal_val_sessions": [], "external_holdout_sessions": []}

    if group_col not in df.columns:
        group_col = "_source_tag" if "_source_tag" in df.columns else None

    df["eval_split"] = "train"
    session_series = df["session"].astype(str)
    external_sessions: set[str] = set()
    internal_val_sessions: set[str] = set()

    grouped = df.groupby(group_col, dropna=False) if group_col else [("all", df)]
    for group_name, part in grouped:
        sessions = sorted(part["session"].astype(str).unique())
        if not sessions:
            continue

        # Reserve at least enough train/val sessions for internal model selection.
        ext = _choose_last_n_sessions(
            sessions,
            EXTERNAL_HOLDOUT_SESSIONS_PER_GROUP,
            EXTERNAL_HOLDOUT_FRACTION_IF_SMALL,
            reserve=MIN_TRAINVAL_SESSIONS_PER_GROUP_AFTER_EXTERNAL,
        )
        remaining = [s for s in sessions if s not in ext]
        val = _choose_last_n_sessions(
            remaining,
            INTERNAL_VAL_SESSIONS_PER_GROUP,
            INTERNAL_VAL_FRACTION_IF_SMALL,
            reserve=1,
        )

        external_sessions.update(ext)
        internal_val_sessions.update(val)
        print(
            f"[SPLIT] group={group_name}: "
            f"train={len([s for s in remaining if s not in val])} "
            f"internal_val={len(val)} external_holdout={len(ext)}"
        )

    if internal_val_sessions:
        df.loc[session_series.isin(internal_val_sessions), "eval_split"] = "internal_val"
    if external_sessions:
        df.loc[session_series.isin(external_sessions), "eval_split"] = "external_holdout"

    # This is a convenience label for full/all CSVs only. The canonical training
    # CSV is materialized below after external rows are removed.
    df["split"] = df["eval_split"].map({
        "train": "train",
        "internal_val": "val",
        "external_holdout": "external",
    }).fillna("train")

    train_sessions = sorted(set(session_series.unique()) - internal_val_sessions - external_sessions)
    split_info = {
        "train_sessions": train_sessions,
        "internal_val_sessions": sorted(internal_val_sessions),
        "external_holdout_sessions": sorted(external_sessions),
    }
    print("Final eval_split counts:", df["eval_split"].value_counts().to_dict())
    print("Internal validation sessions:", split_info["internal_val_sessions"][:20], "..." if len(split_info["internal_val_sessions"]) > 20 else "")
    print("External holdout sessions:", split_info["external_holdout_sessions"][:20], "..." if len(split_info["external_holdout_sessions"]) > 20 else "")
    return df, split_info

def materialize_batch_csv_for_final_dataset(patch_tag: str, ds_dir: Path, *, source_tag: str, source_group: str) -> Path:
    """
    Uploads patch shards for one built source dataset immediately, then stores only
    its rewritten CSV locally. This avoids retaining all patch shards on Colab SSD.
    """
    import pandas as pd

    csv_path = ds_dir / "points_with_rd_patch_index.csv"
    patches_dir = ds_dir / "patches"
    if not csv_path.exists() or not patches_dir.exists():
        raise FileNotFoundError(f"Missing dataset outputs for {patch_tag}: {csv_path}, {patches_dir}")

    gcs_patch_dst = f"{FULL_DATASET_ROOT_URI.rstrip('/')}/patches_{patch_tag}"
    print(f"\nUploading patch shards for {patch_tag} -> {gcs_patch_dst}")
    gcloud_storage_rsync(patches_dir, gcs_patch_dst, delete_unmatched=True)

    df = pd.read_csv(csv_path, low_memory=False)
    df = rewrite_patch_paths(df, patch_tag)
    df["_source_tag"] = patch_tag
    df["_raw_group_tag"] = str(source_tag)
    df["_raw_group"] = str(source_group)

    BATCH_CSV_ROOT.mkdir(parents=True, exist_ok=True)
    out_csv = BATCH_CSV_ROOT / f"{patch_tag}_points_with_rd_patch_index.csv"
    df.to_csv(out_csv, index=False)
    print(f"Stored rewritten batch CSV: {out_csv} rows={len(df)}")
    return out_csv

def combine_and_upload_dataset(batch_csvs: list[Path]) -> Path:
    import pandas as pd

    FULL_DATASET_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    frames = []
    manifest_sources = {}
    for csv_path in batch_csvs:
        csv_path = Path(csv_path)
        if not csv_path.exists():
            raise FileNotFoundError(csv_path)
        df = pd.read_csv(csv_path, low_memory=False)
        frames.append(df)
        manifest_sources[csv_path.stem] = str(csv_path)

    if not frames:
        raise RuntimeError("No batch CSVs produced.")

    # Strict alignment: every batch should already have the same Branch 1 schema.
    # Do not manufacture missing columns during concat; that hides preprocessing bugs.
    all_cols = list(frames[0].columns)
    if STRICT_SCHEMA_ALIGNMENT:
        ref = set(all_cols)
        mismatches = []
        for i, df in enumerate(frames):
            cols = set(df.columns)
            missing = sorted(ref - cols)
            extra = sorted(cols - ref)
            if missing or extra:
                mismatches.append((i, missing, extra))
        if mismatches:
            msg = "\n".join(
                f"  batch[{i}]: missing={missing[:20]} extra={extra[:20]}"
                for i, missing, extra in mismatches[:10]
            )
            raise RuntimeError(
                "Batch CSV schemas differ. Refusing to fill blanks/defaults at concat time.\n" + msg
            )
        frames = [df[all_cols] for df in frames]
    else:
        for df in frames:
            for col in df.columns:
                if col not in all_cols:
                    all_cols.append(col)
        for i, df in enumerate(frames):
            missing = [c for c in all_cols if c not in df.columns]
            if missing:
                raise RuntimeError(f"Batch {i} missing columns under non-strict mode: {missing}")
            frames[i] = df[all_cols]

    combined_all = pd.concat(frames, ignore_index=True)
    missing_final = sorted((FINAL_PATCH_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS) - set(combined_all.columns))
    if missing_final:
        raise RuntimeError(f"Combined dataset is missing required Branch 1 columns: {missing_final}")

    combined_all, split_info = assign_train_val_external_splits(combined_all)
    if "point_label_weight" in combined_all.columns:
        combined_all["point_label_weight"] = pd.to_numeric(
            combined_all["point_label_weight"], errors="coerce"
        ).fillna(1.0).clip(0.0, 1.0)

    helper_cols = [c for c in ["_source_tag", "_raw_group_tag", "_raw_group"] if c in combined_all.columns]

    # Canonical training dataset: excludes external_holdout rows and keeps the
    # split contract expected by train_hybrid_rd_patch_kpconv.py: train/val only.
    combined = combined_all[combined_all["eval_split"] != "external_holdout"].copy()
    combined["split"] = combined["eval_split"].map({"train": "train", "internal_val": "val"}).fillna("train")
    if helper_cols:
        combined = combined.drop(columns=helper_cols)

    if set(combined["split"].astype(str)) - {"train", "val"}:
        raise RuntimeError(f"Canonical training CSV has invalid split values: {sorted(set(combined['split'].astype(str)))}")
    if "val" not in set(combined["split"].astype(str)):
        raise RuntimeError("Canonical training CSV has no internal validation rows. Increase INTERNAL_VAL_* settings or batch size.")

    combined_csv = FULL_DATASET_LOCAL_ROOT / "points_with_rd_patch_index.csv"
    combined.to_csv(combined_csv, index=False)

    external = combined_all[combined_all["eval_split"] == "external_holdout"].copy()
    external_csv = FULL_DATASET_LOCAL_ROOT / "external_holdout_points_with_rd_patch_index.csv"
    if len(external):
        external["split"] = "external"
        if helper_cols:
            external = external.drop(columns=helper_cols)
        external.to_csv(external_csv, index=False)
    else:
        external_csv = None

    all_with_external_csv = FULL_DATASET_LOCAL_ROOT / "all_points_with_external_holdout_index.csv"
    if WRITE_ALL_WITH_EXTERNAL_HOLDOUT_CSV:
        all_export = combined_all.copy()
        if helper_cols:
            all_export = all_export.drop(columns=helper_cols)
        all_export.to_csv(all_with_external_csv, index=False)
    else:
        all_with_external_csv = None

    skipped_log_path = _write_skip_log(FULL_DATASET_LOCAL_ROOT)
    manifest = {
        "kind": "branch1_4class_depth_directness_true_ra_dataset_from_raw_session_extracts",
        "created_unix": int(time.time()),
        "raw_extracts_root": RAW_EXTRACTS_ROOT_URI,
        "output_root": FULL_DATASET_ROOT_URI,
        "label_col": "bucket_4class",
        "sample_weight_col": "point_label_weight",
        "source_csvs": manifest_sources,
        "n_rows": int(len(combined)),
        "n_sessions": int(combined["session"].nunique()) if "session" in combined.columns else None,
        "external_holdout_rows": int(len(external)) if "external" in locals() else 0,
        "external_holdout_sessions": split_info.get("external_holdout_sessions", []),
        "internal_val_sessions": split_info.get("internal_val_sessions", []),
        "train_sessions": split_info.get("train_sessions", []),
        "split_counts": {k: int(v) for k, v in combined["split"].value_counts().to_dict().items()} if "split" in combined.columns else {},
        "eval_split_counts_all": {k: int(v) for k, v in combined_all["eval_split"].value_counts().to_dict().items()} if "eval_split" in combined_all.columns else {},
        "class_counts": combined["bucket_4class"].value_counts(dropna=False).to_dict() if "bucket_4class" in combined.columns else {},
        "skipped_sessions_count": int(len(SKIPPED_SESSION_LOG)),
        "skipped_sessions_csv": str(skipped_log_path) if skipped_log_path else "",
    }
    manifest_path = FULL_DATASET_LOCAL_ROOT / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print("\nCombined dataset summary:")
    print(json.dumps(manifest, indent=2, default=str))

    print(f"\nUploading combined CSV and manifest -> {FULL_DATASET_ROOT_URI}")
    gcloud_storage_cp(combined_csv, f"{FULL_DATASET_ROOT_URI.rstrip()}/points_with_rd_patch_index.csv")
    if external_csv and Path(external_csv).exists():
        gcloud_storage_cp(external_csv, f"{FULL_DATASET_ROOT_URI.rstrip()}/external_holdout_points_with_rd_patch_index.csv")
    if all_with_external_csv and Path(all_with_external_csv).exists():
        gcloud_storage_cp(all_with_external_csv, f"{FULL_DATASET_ROOT_URI.rstrip()}/all_points_with_external_holdout_index.csv")
    gcloud_storage_cp(manifest_path, f"{FULL_DATASET_ROOT_URI.rstrip()}/manifest.json")
    if skipped_log_path and skipped_log_path.exists():
        gcloud_storage_cp(skipped_log_path, f"{FULL_DATASET_ROOT_URI.rstrip()}/skipped_sessions.csv")

    # Best-effort final validation through the gcsfuse mount. If the mount is not
    # active, source datasets were still validated before their patches were uploaded.
    mounted_csv = FULL_DATASET_MOUNT_ROOT / "points_with_rd_patch_index.csv"
    if mounted_csv.exists():
        validate_help = command_help_text([PY, DATASET_TOOLS, "validate"])
        validate_cmd = [
            PY, DATASET_TOOLS, "validate",
            "--point-csv", mounted_csv,
            "--dataset-root", FULL_DATASET_MOUNT_ROOT,
            "--session-prefix", "session_2026-",
        ]
        if "--allow-train-only" in validate_help and set(combined["split"].astype(str)) == {"train"}:
            validate_cmd.append("--allow-train-only")
        run(validate_cmd)
    else:
        print("\nSkipping final mounted validation because gcsfuse path is not visible yet:")
        print(" ", mounted_csv)
        print("Each per-batch source dataset was validated before patch upload.")

    print("\nFull Branch 1 dataset built:")
    print(" ", FULL_DATASET_ROOT_URI)
    print("Use as dataset-root:")
    print(" ", f"{FULL_DATASET_ROOT_URI}")
    print("Use as training dataset CSV:")
    print(" ", f"{FULL_DATASET_ROOT_URI}/points_with_rd_patch_index.csv")
    print("Use as external holdout CSV:")
    print(" ", f"{FULL_DATASET_ROOT_URI}/external_holdout_points_with_rd_patch_index.csv")
    return FULL_DATASET_LOCAL_ROOT

# -----------------------------
# Split CPU/GPU preprocessing orchestration
# -----------------------------

# These GCS prefixes intentionally separate the CPU-bound raw/radar preparation,
# GPU-bound OneFormer label-map export, and CPU-bound dataset construction.
# They are independent of the final training dataset prefix.
ONEFORMER_INPUT_SUBDIR = INTERMEDIATE_ONEFORMER_INPUT_SUBDIR
ONEFORMER_OUTPUT_SUBDIR = INTERMEDIATE_ONEFORMER_OUTPUT_SUBDIR
ONEFORMER_INPUT_ROOT_URI = f"{GCS_ROOT_URI}/{ONEFORMER_INPUT_SUBDIR}"
ONEFORMER_OUTPUT_ROOT_URI = f"{GCS_ROOT_URI}/{ONEFORMER_OUTPUT_SUBDIR}"

CPU_PRE_ONEFORMER_LOCAL_ROOT = WORK_ROOT / "_branch1_cpu_pre_oneformer"
GPU_ONEFORMER_LOCAL_ROOT = WORK_ROOT / "_branch1_gpu_oneformer"
CPU_POST_ONEFORMER_LOCAL_ROOT = WORK_ROOT / "_branch1_cpu_post_oneformer"


def _reset_skipped_session_log() -> None:
    try:
        SKIPPED_SESSION_LOG.clear()
    except NameError:
        pass


def _batch_id(batch_idx: int) -> str:
    return f"batch_{int(batch_idx):04d}"


def _records_to_batches(max_batches: int | None = None) -> list[list[dict]]:
    records = raw_session_records()
    batches = list(chunked(records, RAW_PREPROCESS_BATCH_SIZE))
    if max_batches is not None:
        batches = batches[:max_batches]
    print(f"\nTotal raw sessions selected: {len(records)}")
    print(f"Selected {len(batches)} batch(es), batch size={RAW_PREPROCESS_BATCH_SIZE}")
    return batches


def _group_records(records: list[dict]) -> dict[tuple[str, str, str], list[dict]]:
    groups: dict[tuple[str, str, str], list[dict]] = {}
    for rec in records:
        groups.setdefault((rec["tag"], rec["group"], rec["prefix"]), []).append(rec)
    return groups


def _flatten_accidental_nested_group(group_root: Path, group: str) -> None:
    nested = group_root / group
    if nested.exists() and nested.is_dir():
        for child in nested.iterdir():
            target = group_root / child.name
            if target.exists():
                shutil.rmtree(target)
            shutil.move(str(child), str(target))
        shutil.rmtree(nested, ignore_errors=True)


def _write_json_artifact(local_path: Path, payload: dict) -> None:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str), encoding="utf-8")


def _upload_stage_manifest(local_root: Path, dst_uri: str, name: str, payload: dict) -> None:
    path = local_root / name
    _write_json_artifact(path, payload)
    gcloud_storage_cp(path, f"{dst_uri.rstrip('/')}/{name}")


def _download_stage_group(src_uri: str, dst_root: Path) -> bool:
    """Return True when the remote prefix exists and was downloaded."""
    matches = gcloud_storage_ls(f"{src_uri.rstrip('/')}/*", allow_empty=True)
    if not matches:
        print(f"[STAGE PULL] no remote objects under {src_uri}; skipping")
        return False
    shutil.rmtree(dst_root, ignore_errors=True)
    dst_root.mkdir(parents=True, exist_ok=True)
    gcloud_storage_rsync(src_uri, dst_root)
    return True


def _post_oneformer_preprocess_group_root(root: Path, prefix: str) -> bool:
    """Continue preprocessing after session_dir/seg/*.npy already exists."""
    stages = [
        ("sync/seg-meta", lambda: run_sync_and_seg_meta_stage(root, prefix)),
        ("autolabel", lambda: run_depth_gated_autolabel_for_root(root, prefix)),
    ]
    for stage_name, fn in stages:
        if not active_session_dirs(root, prefix):
            print(f"[{stage_name}] no sessions remain; skipping group {root.name}")
            return False
        ok = fn()
        if not ok or not active_session_dirs(root, prefix):
            print(f"[{stage_name}] no usable sessions remain in {root.name}; skipping this group.")
            return False

    try:
        enrich_autolabel_outputs_for_root(root, prefix, "labeled_radar_points_v4.csv")
        assert_root_label_schema(
            root, prefix, "labeled_radar_points_v4.csv",
            NATIVE_RADAR_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS,
            context="post-autolabel feature contract",
        )
        if not audit_directness_and_ghost_features_for_root(
            root, prefix, "labeled_radar_points_v4.csv",
            context="post-autolabel-directness-audit",
            require_spatiotemporal=False,
            require_teacher=False,
        ):
            return False
    except Exception as exc:
        problems = []
        for session_dir in active_session_dirs(root, prefix):
            try:
                assert_csv_has_columns(
                    session_dir / "labeled_radar_points_v4.csv",
                    NATIVE_RADAR_REQUIRED_COLS | COMPUTED_PREPATCH_REQUIRED_COLS,
                    context=f"{session_dir.name} post-autolabel feature contract",
                )
            except Exception as sub_exc:
                problems.append((session_dir.name, str(sub_exc)))
        if not problems:
            if SKIP_CORRUPT_SESSIONS:
                print("[post-autolabel feature contract] root check failed but per-session checks passed; continuing.")
                print("  root error:", exc)
            else:
                raise
        elif not _handle_stage_problems(root, prefix, "post-autolabel-feature-contract", problems):
            return False

    if not ensure_radar_tensors(root, prefix):
        return False
    if not assert_preprocessed_sessions(root, prefix):
        return False
    n_remaining = len(active_session_dirs(root, prefix))
    if n_remaining < MIN_VALID_SESSIONS_PER_GROUP_AFTER_SKIP:
        print(f"Only {n_remaining} usable session(s) remain in {root.name}; skipping this group.")
        return False
    return True


# -----------------------------------------------------------------------------
# Stage 1: CPU runtime — raw download + radar CSV/meta + upload for OneFormer
# -----------------------------------------------------------------------------

def run_cpu_prepare_for_oneformer(max_batches: int | None = None) -> Path:
    """CPU/IO stage. Downloads raw sessions, runs radar CSV + meta, uploads
    per-batch OneFormer input folders to GCS.

    Use this on a CPU runtime. It intentionally does NOT run OneFormer.
    """
    _reset_skipped_session_log()
    batches = _records_to_batches(max_batches)
    shutil.rmtree(CPU_PRE_ONEFORMER_LOCAL_ROOT, ignore_errors=True)
    CPU_PRE_ONEFORMER_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    stage_manifest = {
        "stage": "cpu_prepare_for_oneformer",
        "created_unix": int(time.time()),
        "oneformer_input_root_uri": ONEFORMER_INPUT_ROOT_URI,
        "raw_extracts_root_uri": RAW_EXTRACTS_ROOT_URI,
        "catalog_uri": GCS_CATALOG_URI,
        "batches": [],
    }

    for batch_idx, batch_records in enumerate(batches, start=1):
        bid = _batch_id(batch_idx)
        print(f"\n{'='*80}\nCPU Stage 1 — {bid}: raw/radar/meta preparation\n{'='*80}")
        batch_root = CPU_PRE_ONEFORMER_LOCAL_ROOT / bid
        shutil.rmtree(batch_root, ignore_errors=True)
        batch_root.mkdir(parents=True, exist_ok=True)

        for (tag, group, prefix), items in _group_records(batch_records).items():
            group_root = batch_root / group
            group_root.mkdir(parents=True, exist_ok=True)
            print(f"\nDownloading {len(items)} raw {tag}/{group} session(s) -> {group_root}")
            for xfer in chunked(items, SESSION_DOWNLOAD_BATCH_SIZE):
                gcloud_storage_cp([r["uri"] for r in xfer], group_root, recursive=True)
            _flatten_accidental_nested_group(group_root, group)
            ensure_color_timestamp_aliases(group_root, prefix)

            ok = run_radar_csv_and_meta_stage(group_root, prefix)
            remaining = active_session_dirs(group_root, prefix)
            if not ok or not remaining:
                print(f"[CPU PRE] skipping {tag}/{group}: no usable sessions after radar/meta stage.")
                continue

            dst = f"{ONEFORMER_INPUT_ROOT_URI.rstrip('/')}/{bid}/{group}"
            print(f"\nUploading OneFormer input group -> {dst}")
            gcloud_storage_rsync(group_root, dst, delete_unmatched=False)
            stage_manifest["batches"].append({
                "batch_id": bid,
                "tag": tag,
                "group": group,
                "prefix": prefix,
                "n_input_records": len(items),
                "n_uploaded_sessions": len(remaining),
                "input_uri": dst,
                "sessions": [p.name for p in remaining],
            })

        print("\nSSD status:")
        run(["df", "-h", "/content"], check=False)

    skipped_path = _write_skip_log(CPU_PRE_ONEFORMER_LOCAL_ROOT)
    if skipped_path and skipped_path.exists():
        gcloud_storage_cp(skipped_path, f"{ONEFORMER_INPUT_ROOT_URI.rstrip('/')}/skipped_sessions_cpu_prepare.csv")
    _upload_stage_manifest(CPU_PRE_ONEFORMER_LOCAL_ROOT, ONEFORMER_INPUT_ROOT_URI, "cpu_prepare_manifest.json", stage_manifest)
    print("\nCPU pre-OneFormer stage complete:")
    print(" ", ONEFORMER_INPUT_ROOT_URI)
    return CPU_PRE_ONEFORMER_LOCAL_ROOT


# -----------------------------------------------------------------------------
# Stage 2: GPU runtime — pull prepared sessions + bulk OneFormer + upload maps
# -----------------------------------------------------------------------------

def run_gpu_oneformer_labelmap_export(max_batches: int | None = None) -> Path:
    """GPU stage. Pulls CPU-prepared session folders, runs OneFormer once per
    batch/group, and uploads session folders with seg/*.npy + seg metadata.

    Use this on a GPU runtime. It intentionally does NOT build the dataset.
    """
    _reset_skipped_session_log()
    batches = _records_to_batches(max_batches)
    shutil.rmtree(GPU_ONEFORMER_LOCAL_ROOT, ignore_errors=True)
    GPU_ONEFORMER_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    stage_manifest = {
        "stage": "gpu_oneformer_labelmap_export",
        "created_unix": int(time.time()),
        "oneformer_input_root_uri": ONEFORMER_INPUT_ROOT_URI,
        "oneformer_output_root_uri": ONEFORMER_OUTPUT_ROOT_URI,
        "batches": [],
    }

    for batch_idx, batch_records in enumerate(batches, start=1):
        bid = _batch_id(batch_idx)
        print(f"\n{'='*80}\nGPU Stage 2 — {bid}: bulk OneFormer label-map export\n{'='*80}")
        batch_root = GPU_ONEFORMER_LOCAL_ROOT / bid
        shutil.rmtree(batch_root, ignore_errors=True)
        batch_root.mkdir(parents=True, exist_ok=True)

        for (tag, group, prefix), items in _group_records(batch_records).items():
            dst = f"{ONEFORMER_OUTPUT_ROOT_URI.rstrip('/')}/{bid}/{group}"
            # Idempotency: skip groups whose OneFormer outputs already exist on GCS.
            if not FORCE_REUPLOAD_ONEFORMER_OUTPUTS and _remote_prefix_has_objects(dst):
                print(f"[GPU ONEFORMER] {bid}/{group}: outputs already present at {dst}; "
                      f"skipping (set FORCE_REUPLOAD_ONEFORMER_OUTPUTS=True to rebuild).")
                continue

            src = f"{ONEFORMER_INPUT_ROOT_URI.rstrip('/')}/{bid}/{group}"
            group_root = batch_root / group
            if not _download_stage_group(src, group_root):
                continue

            ok = run_oneformer_for_root(group_root, prefix)
            remaining = active_session_dirs(group_root, prefix)
            if not ok or not remaining:
                print(f"[GPU ONEFORMER] skipping {tag}/{group}: no usable sessions after OneFormer.")
                continue

            print(f"\nUploading OneFormer output group -> {dst}")
            # delete_unmatched=False: never delete remote objects written by a prior run.
            gcloud_storage_rsync(group_root, dst, delete_unmatched=False)
            stage_manifest["batches"].append({
                "batch_id": bid,
                "tag": tag,
                "group": group,
                "prefix": prefix,
                "n_uploaded_sessions": len(remaining),
                "output_uri": dst,
                "sessions": [p.name for p in remaining],
            })

        print("\nSSD status:")
        run(["df", "-h", "/content"], check=False)

    skipped_path = _write_skip_log(GPU_ONEFORMER_LOCAL_ROOT)
    if skipped_path and skipped_path.exists():
        gcloud_storage_cp(skipped_path, f"{ONEFORMER_OUTPUT_ROOT_URI.rstrip('/')}/skipped_sessions_gpu_oneformer.csv")
    _upload_stage_manifest(GPU_ONEFORMER_LOCAL_ROOT, ONEFORMER_OUTPUT_ROOT_URI, "gpu_oneformer_manifest.json", stage_manifest)
    print("\nGPU OneFormer stage complete:")
    print(" ", ONEFORMER_OUTPUT_ROOT_URI)
    return GPU_ONEFORMER_LOCAL_ROOT


# -----------------------------------------------------------------------------
# Stage 3: CPU runtime — pull maps + autolabel/depth/tensors/patches/dataset
# -----------------------------------------------------------------------------

def run_cpu_post_oneformer_dataset_build(max_batches: int | None = None) -> Path:
    """CPU/IO stage. Pulls OneFormer-output session folders and performs all
    remaining non-GPU operations: sync, autolabel, depth correspondence,
    ghost/directness fusion, tensor export, RD/true-RA patches, and final split.
    """
    _reset_skipped_session_log()
    existing = gcloud_storage_ls(f"{FULL_DATASET_ROOT_URI}/points_with_rd_patch_index.csv", allow_empty=True)
    if existing and not FORCE_REBUILD_FULL_DATASET:
        raise FileExistsError(
            f"Dataset already exists at {FULL_DATASET_ROOT_URI}.\n"
            "Set FORCE_REBUILD_FULL_DATASET=True in this cell if you really want to rebuild it."
        )

    batches = _records_to_batches(max_batches)
    shutil.rmtree(CPU_POST_ONEFORMER_LOCAL_ROOT, ignore_errors=True)
    shutil.rmtree(BATCH_CSV_ROOT, ignore_errors=True)
    CPU_POST_ONEFORMER_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    BATCH_CSV_ROOT.mkdir(parents=True, exist_ok=True)

    all_batch_csvs: list[Path] = []
    try:
        for batch_idx, batch_records in enumerate(batches, start=1):
            bid = _batch_id(batch_idx)
            print(f"\n{'='*80}\nCPU Stage 3 — {bid}: post-OneFormer dataset build\n{'='*80}")
            batch_root = CPU_POST_ONEFORMER_LOCAL_ROOT / bid
            shutil.rmtree(batch_root, ignore_errors=True)
            batch_root.mkdir(parents=True, exist_ok=True)
            shutil.rmtree(DATASET_WORK_ROOT, ignore_errors=True)
            DATASET_WORK_ROOT.mkdir(parents=True, exist_ok=True)

            for (tag, group, prefix), items in _group_records(batch_records).items():
                src = f"{ONEFORMER_OUTPUT_ROOT_URI.rstrip('/')}/{bid}/{group}"
                group_root = batch_root / group
                if not _download_stage_group(src, group_root):
                    continue

                ok = _post_oneformer_preprocess_group_root(group_root, prefix)
                remaining = active_session_dirs(group_root, prefix)
                if not ok or not remaining:
                    print(f"[CPU POST] skipping {tag}/{group}: no usable sessions after post-OneFormer preprocessing.")
                    continue
                if len(remaining) < 2:
                    for session_dir in remaining:
                        _drop_session_dir(session_dir, "batch-min-session-count", "fewer than 2 usable sessions remain for temporary split")
                    print(f"[CPU POST] skipping {tag}/{group}: fewer than 2 usable sessions remain.")
                    continue

                # Curated admission in CPU Stage 3/4 is now the only downstream handoff.
                # Do not mirror processed sessions into the retired processed_training_ready_sessions tree.

                patch_tag = f"b{batch_idx:04d}_{tag}"
                try:
                    ds_dir = build_source_dataset(group_root, prefix, patch_tag)
                    all_batch_csvs.append(materialize_batch_csv_for_final_dataset(
                        patch_tag, ds_dir, source_tag=tag, source_group=group
                    ))
                except Exception as exc:
                    if SKIP_CORRUPT_SESSIONS:
                        print(f"[CPU POST] dataset build failed for {tag}/{group}; skipping this group.")
                        print("  error:", exc)
                        for session_dir in active_session_dirs(group_root, prefix):
                            _record_skipped_session(session_dir, "dataset-build-group", str(exc))
                        continue
                    raise

            print("\nSSD status:")
            run(["df", "-h", "/content"], check=False)

        final_local = combine_and_upload_dataset(all_batch_csvs)
        return final_local
    finally:
        print("\nDone. Final local dataset, if built:", FULL_DATASET_LOCAL_ROOT)


# Debug-only convenience wrapper. Prefer the three staged functions above when
# separating CPU and GPU Colab runtimes.
def run_full_raw_preprocess_and_dataset_build(max_batches: int | None = None):
    """Legacy helper retained for compatibility; this notebook does not use it."""
    run_cpu_prepare_for_oneformer(max_batches=max_batches)
    run_gpu_oneformer_labelmap_export(max_batches=max_batches)
    return run_cpu_post_oneformer_dataset_build(max_batches=max_batches)


print("Resolved code root:", CODE_ROOT)
print("Resolved BRANCH1:", BRANCH1)
print("Radar tensor exporter:", RADAR_TENSOR_EXPORT)
print("Radar tensor workers:", RADAR_TENSOR_WORKERS, "npz=", RADAR_TENSOR_NPZ_COMPRESSION, "strip=", RADAR_TENSOR_SIDECAR_STRIP_MODE)
print("RD patch frame offset:", RD_PATCH_FRAME_NUMBER_OFFSET)
print("Raw extracts:", RAW_EXTRACTS_ROOT_URI)
print("OneFormer input staging:", ONEFORMER_INPUT_ROOT_URI)
print("OneFormer frame batch size:", ONEFORMER_FRAME_BATCH_SIZE, "fp16=", ONEFORMER_USE_FP16, "timing=", ONEFORMER_TIMING)
print("OneFormer output staging:", ONEFORMER_OUTPUT_ROOT_URI)
print("\nProcessing-only Colab workflow:")
print("  1) run_audit_and_write_manifest(max_sessions=None)")
print("  2) CPU runtime: run_cpu_prepare_for_oneformer(max_batches=None)")
print("  3) GPU runtime: run_gpu_oneformer_labelmap_export(max_batches=None)")
print("  4) CPU runtime: run_cpu_finalize_processed_uploads(max_batches=None)")
print("\nThis notebook intentionally does not launch model training.")

In [ ]:
# @title defs: audit processed sessions, recover raw sessions, and override processing records

from __future__ import annotations

import csv
import hashlib
import tarfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any

# -----------------------------------------------------------------------------
# Audit/staging configuration
# -----------------------------------------------------------------------------

# Input roots now follow the raw -> intermediate -> curated layout.
AUDIT_PROCESSED_ROOT_URI = f"{GCS_ROOT_URI}/intermediate/oneformer/outputs"
AUDIT_RAW_ROOT_URI_CANDIDATES = [
    f"{GCS_ROOT_URI}/raw/sessions"
]
AUDIT_RAW_ROOT_URI = AUDIT_RAW_ROOT_URI_CANDIDATES[0]
CURATED_SESSION_ROOT_URI = f"{GCS_ROOT_URI}/curated/sessions"
CURATED_GOLD_ROOT_URI = f"{CURATED_SESSION_ROOT_URI}/gold"
CURATED_SILVER_ROOT_URI = f"{CURATED_SESSION_ROOT_URI}/silver"
CURATED_QUARANTINE_ROOT_URI = f"{CURATED_SESSION_ROOT_URI}/quarantine"

# Audit artifacts / stage intermediates.
AUDIT_OUTPUT_ROOT_URI = f"{GCS_ROOT_URI}/intermediate/audits"
AUDIT_LOCAL_ROOT = WORK_ROOT / "rigorous_preprocess_audit"
AUDIT_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
AUDIT_REPORT_CSV = AUDIT_LOCAL_ROOT / "processed_raw_audit.csv"
STAGED_MANIFEST_CSV = AUDIT_LOCAL_ROOT / "staged_sessions.csv"
CURATION_MANIFEST_CSV = AUDIT_LOCAL_ROOT / "curated_session_manifest.csv"

# Audit behavior.
INCLUDE_RAW_NOT_IN_PROCESSED = True
SESSION_PREFIX = RAW_SESSION_PREFIX
RECURSIVE_SESSION_DISCOVERY = True
MAX_AUDIT_SESSIONS = None     # set small for debugging

# Empty-manifest behavior.
# "noop" means CPU Stage 1 exits cleanly when audit finds nothing to process.
# "error" restores strict behavior.
EMPTY_STAGE_MANIFEST_BEHAVIOR = "noop"
AUTO_RUN_AUDIT_IF_STAGE_EMPTY = True

MIN_RADAR_BIN_BYTES = 1024
MIN_COLOR_BYTES = 1024
MIN_DEPTH_BYTES = 16
MIN_LABEL_CSV_ROWS = 2        # header + at least one data row

# Final upload behavior.
UPLOAD_COMPRESSED_ARCHIVES = True
UPLOAD_EXPANDED_PROCESSED_READY = False
DELETE_UNMATCHED_ON_EXPANDED_SYNC = False

# CPU Stage 3 parallelism.
# Compression + upload are independent per session, so these can safely run concurrently.
# If Colab/network throttles or the runtime gets unstable, drop FINALIZE_UPLOAD_WORKERS to 4 or 2.
FINALIZE_UPLOAD_WORKERS = int(globals().get("FINALIZE_UPLOAD_WORKERS", 8))
FINALIZE_TAR_COMPRESSLEVEL = int(globals().get("FINALIZE_TAR_COMPRESSLEVEL", 3))

print("Audit processed root:", AUDIT_PROCESSED_ROOT_URI)
print("Audit raw roots     :", AUDIT_RAW_ROOT_URI_CANDIDATES)
print("Curated gold root  :", CURATED_GOLD_ROOT_URI)
print("Curated silver root:", CURATED_SILVER_ROOT_URI)
print("Curated quarantine :", CURATED_QUARANTINE_ROOT_URI)
print("Audit artifacts     :", AUDIT_OUTPUT_ROOT_URI)
print("Catalog URI         :", GCS_CATALOG_URI)


# -----------------------------------------------------------------------------
# Low-level remote inspection helpers
# -----------------------------------------------------------------------------

def _as_uri_prefix(uri: str) -> str:
    return uri.rstrip("/") + "/"


def _remote_ls(uri_pattern: str, *, recursive: bool = False, allow_empty: bool = True) -> list[str]:
    cmd = ["gcloud", "--quiet", "--verbosity=error", "storage", "ls"]
    if recursive:
        cmd.append("--recursive")
    cmd.append(uri_pattern)
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if proc.returncode != 0:
        if allow_empty:
            return []
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
    return [x.strip() for x in (proc.stdout or "").splitlines() if x.strip()]


def _remote_ls_long(uri_pattern: str, *, recursive: bool = False, allow_empty: bool = True) -> dict[str, int]:
    """Return gs://object -> size_bytes. Directory markers are ignored."""
    cmd = ["gcloud", "--quiet", "--verbosity=error", "storage", "ls", "--long"]
    if recursive:
        cmd.append("--recursive")
    cmd.append(uri_pattern)
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if proc.returncode != 0:
        if allow_empty:
            return {}
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
    out: dict[str, int] = {}
    for raw in (proc.stdout or "").splitlines():
        line = raw.strip()
        if not line or line.startswith("TOTAL:") or line.endswith("/"):
            continue
        parts = line.split()
        # Typical: SIZE  YYYY-MM-DDTHH:MM:SSZ  gs://...
        uri = next((p for p in reversed(parts) if p.startswith("gs://")), "")
        if not uri:
            continue
        size = 0
        for p in parts:
            try:
                size = int(p)
                break
            except ValueError:
                continue
        out[uri] = size
    return out


def _remote_file_size(uri: str) -> int:
    listing = _remote_ls_long(uri, recursive=False, allow_empty=True)
    return int(listing.get(uri, 0))


def _session_name_from_uri(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1]


def _group_name_from_uri(root_uri: str, child_uri: str) -> str:
    rel = child_uri.rstrip("/")[len(root_uri.rstrip("/") + "/"):]
    return rel.split("/", 1)[0]


def _safe_name(text: str) -> str:
    out = "".join(ch if ch.isalnum() else "_" for ch in str(text)).strip("_")
    return out or "unknown"


def _csv_has_data_remote(uri: str, *, min_rows: int = MIN_LABEL_CSV_ROWS) -> bool:
    """Cheap remote row-count test. Downloads only the target CSV to a tiny local temp file."""
    size = _remote_file_size(uri)
    if size <= 0:
        return False
    tmp = AUDIT_LOCAL_ROOT / "_tmp_csv_checks" / hashlib.md5(uri.encode()).hexdigest()
    tmp.parent.mkdir(parents=True, exist_ok=True)
    try:
        gcloud_storage_cp(uri, tmp)
        rows = 0
        with tmp.open("r", encoding="utf-8", errors="replace", newline="") as fh:
            for rows, _ in enumerate(fh, start=1):
                if rows >= min_rows:
                    return True
        return rows >= min_rows
    finally:
        tmp.unlink(missing_ok=True)


def _has_any(paths: dict[str, int], predicate, *, min_bytes: int = 1) -> bool:
    return any(size >= min_bytes and predicate(uri) for uri, size in paths.items())


def _count_any(paths: dict[str, int], predicate, *, min_bytes: int = 1) -> int:
    return sum(1 for uri, size in paths.items() if size >= min_bytes and predicate(uri))


def _uri_basename(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1].lower()


def _is_standard_color_timestamp_csv(uri: str) -> bool:
    name = _uri_basename(uri)
    return name.endswith(".csv") and "color_timestamps" in name


def _is_session_named_timestamp_csv(uri: str, session: str) -> bool:
    """Some captures store color timestamps as <session>.csv, not <session>_color_timestamps.csv."""
    return _uri_basename(uri) == f"{session.lower()}.csv"


def _has_standard_color_timestamps(paths: dict[str, int], *, min_bytes: int = MIN_DEPTH_BYTES) -> bool:
    return _has_any(paths, _is_standard_color_timestamp_csv, min_bytes=min_bytes)


def _has_session_named_color_timestamps(paths: dict[str, int], session: str, *, min_bytes: int = MIN_DEPTH_BYTES) -> bool:
    return _has_any(paths, lambda u: _is_session_named_timestamp_csv(u, session), min_bytes=min_bytes)


def _color_timestamp_source(paths: dict[str, int], session: str, *, min_bytes: int = MIN_DEPTH_BYTES) -> str:
    if _has_standard_color_timestamps(paths, min_bytes=min_bytes):
        return "standard_color_timestamps_csv"
    if _has_session_named_color_timestamps(paths, session, min_bytes=min_bytes):
        return "session_named_csv"
    return "missing"


# -----------------------------------------------------------------------------
# Processed/raw classification
# -----------------------------------------------------------------------------

def _session_refs_from_recursive_listing(
    root_uri: str,
    *,
    group_field: str,
    uri_field: str,
    glob_hint: str = "*",
) -> list[dict[str, str]]:
    """Find session_* directories at any nesting depth below root_uri.

    GCS has no real directories, so this derives session roots from object paths.
    It intentionally uses the first path component that starts with SESSION_PREFIX.
    Examples it catches:
      root/dataset4/session_.../file
      root/data4-22/data4-22/session_.../file
      root/some/messy/nested/path/session_.../file
    """
    root = root_uri.rstrip("/")
    if glob_hint and glob_hint != "*":
        search_root = f"{root}/{glob_hint.strip('/')}"
    else:
        search_root = root

    print(f"Recursive session discovery under: {search_root}", flush=True)
    objects = _remote_ls(f"{search_root.rstrip('/')}/**", recursive=True, allow_empty=True)
    seen: dict[tuple[str, str], dict[str, str]] = {}

    for obj_uri in objects:
        obj_uri = obj_uri.strip().rstrip("/")
        if not obj_uri.startswith(search_root.rstrip("/") + "/"):
            continue
        rel = obj_uri[len(search_root.rstrip("/") + "/"):]
        parts = [p for p in rel.split("/") if p]
        for idx, part in enumerate(parts):
            if not part.startswith(SESSION_PREFIX):
                continue
            # Do not treat a lone file such as session_YYYY-MM-DD.csv or
            # session_YYYY-MM-DD.tar.gz as a session root. In normal object
            # paths, the real session folder appears before the file leaf.
            if idx == len(parts) - 1 and re.search(r"\.(csv|json|mp4|avi|mov|bin|npz|npy|png|jpg|jpeg|tar|tgz|gz)$", part, flags=re.IGNORECASE):
                continue
            session = part
            # parent relative to search_root, preserving messy nested folders for traceability.
            parent_rel = "/".join(parts[:idx])
            if glob_hint and glob_hint != "*":
                # Restore the user-provided restricted folder into the reported group path.
                group = f"{glob_hint.strip('/')}/{parent_rel}".strip("/")
                session_uri = f"{root}/{group}/{session}"
            else:
                group = parent_rel or "root"
                session_uri = f"{search_root.rstrip('/')}/{parent_rel + '/' if parent_rel else ''}{session}"
            key = (group, session)
            if key not in seen:
                seen[key] = {
                    "session": session,
                    group_field: group,
                    uri_field: session_uri.rstrip("/"),
                }
            break

    refs = sorted(seen.values(), key=lambda r: (r[group_field], r["session"]))
    print(f"Recursive discovery found {len(refs)} session root(s).", flush=True)
    return refs


def list_processed_session_refs() -> list[dict[str, str]]:
    if RECURSIVE_SESSION_DISCOVERY:
        return _session_refs_from_recursive_listing(
            AUDIT_PROCESSED_ROOT_URI,
            group_field="processed_group",
            uri_field="processed_uri",
        )

    refs: list[dict[str, str]] = []
    group_uris = _remote_ls(f"{AUDIT_PROCESSED_ROOT_URI.rstrip('/')}/*/", allow_empty=True)
    for group_uri in group_uris:
        group = _group_name_from_uri(AUDIT_PROCESSED_ROOT_URI, group_uri)
        session_uris = _remote_ls(f"{group_uri.rstrip('/')}/{SESSION_PREFIX}*/", allow_empty=True)
        for sess_uri in session_uris:
            refs.append({
                "session": _session_name_from_uri(sess_uri),
                "processed_group": group,
                "processed_uri": sess_uri.rstrip("/"),
            })
    refs = sorted(refs, key=lambda r: (r["processed_group"], r["session"]))
    return refs


def list_raw_session_refs_for_root(root_uri: str) -> list[dict[str, str]]:
    """Discover raw session roots from the live catalog rather than GCS globs."""
    refs: list[dict[str, str]] = []
    rows = raw_catalog_rows()
    for row in rows.itertuples(index=False):
        raw_uri = str(row.raw_path).rstrip("/")
        session_id = str(row.session_id)
        if not raw_uri or not session_id.startswith(SESSION_PREFIX):
            continue
        refs.append({
            "session": session_id,
            "dataset_id": str(row.dataset_id),
            "raw_group": str(row.dataset_id),
            "raw_uri": raw_uri,
            "raw_root_uri": root_uri.rstrip("/"),
            "legacy_dataset_name": str(row.legacy_dataset_name),
            "capture_date": str(row.capture_date),
            "catalog_label_status": str(row.label_status),
            "catalog_quality_tier": str(row.quality_tier),
        })
    return refs


def list_raw_session_refs() -> list[dict[str, str]]:
    refs = list_raw_session_refs_for_root(AUDIT_RAW_ROOT_URI)
    print(f"Catalog-backed raw session refs: {len(refs)}", flush=True)
    return sorted(refs, key=lambda r: (r.get("dataset_id", ""), r["session"]))


def classify_processed_session(ref: dict[str, str]) -> dict[str, Any]:
    uri = ref["processed_uri"].rstrip("/")
    session = ref["session"]
    objs = _remote_ls_long(f"{uri}/**", recursive=True, allow_empty=True)
    lower = {u.lower(): size for u, size in objs.items()}

    label_candidates = [
        f"{uri}/labeled_radar_points_v4_fused.csv",
        f"{uri}/labeled_radar_points_v4.csv",
    ]
    label_ok = any(_csv_has_data_remote(x) for x in label_candidates if _remote_file_size(x) > 0)

    radar_tensor_ok = _has_any(lower, lambda u: u.endswith("_radar_tensors.npz") or u.endswith("radar_tensors.npz"), min_bytes=MIN_RADAR_BIN_BYTES)
    sync_ok = _has_any(lower, lambda u: f"synchronized_{session.lower()}.csv" in u or "/synchronized_" in u, min_bytes=MIN_COLOR_BYTES)
    color_ok = _has_any(lower, lambda u: u.endswith(".mp4") and "color" in u, min_bytes=MIN_COLOR_BYTES)
    radar_csv_ok = _has_any(lower, lambda u: u.endswith(f"/{session.lower()}.csv"), min_bytes=MIN_COLOR_BYTES)
    seg_ok = _has_any(lower, lambda u: "/seg/" in u and u.endswith(".npy"), min_bytes=MIN_DEPTH_BYTES)
    rd_ok = _has_any(lower, lambda u: "/hybrid_rd/" in u or "_rd" in u, min_bytes=MIN_DEPTH_BYTES)

    reasons = []
    for ok, reason in [
        (label_ok, "missing_or_empty_label_csv"),
        (radar_tensor_ok, "missing_radar_tensors_npz"),
        (sync_ok, "missing_synchronized_csv"),
        (color_ok, "missing_color_mp4"),
        (radar_csv_ok, "missing_native_radar_csv"),
    ]:
        if not ok:
            reasons.append(reason)

    status = "good" if not reasons else "bad"
    return {
        **ref,
        "processed_status": status,
        "processed_reason": "ok" if status == "good" else ";".join(reasons),
        "processed_object_count": len(objs),
        "processed_label_ok": label_ok,
        "processed_radar_tensor_ok": radar_tensor_ok,
        "processed_sync_ok": sync_ok,
        "processed_color_ok": color_ok,
        "processed_radar_csv_ok": radar_csv_ok,
        "processed_seg_ok": seg_ok,
        "processed_rd_ok": rd_ok,
    }


def classify_raw_session(ref: dict[str, str]) -> dict[str, Any]:
    uri = ref["raw_uri"].rstrip("/")
    session = ref["session"]
    dataset_id = str(ref.get("dataset_id") or ref.get("raw_group") or "")
    objs = _remote_ls_long(f"{uri}/**", recursive=True, allow_empty=True)
    lower = {u.lower(): size for u, size in objs.items()}

    radar_bin_count = _count_any(
        lower,
        lambda u: u.endswith(".bin") and not u.endswith(".idx.bin"),
        min_bytes=MIN_RADAR_BIN_BYTES,
    )
    color_count = _count_any(
        lower,
        lambda u: u.endswith(".mp4") and ("color" in u or "rgb" in u),
        min_bytes=MIN_COLOR_BYTES,
    )
    depth_count = _count_any(
        lower,
        lambda u: (
            ("depth" in u and (u.endswith(".npy") or u.endswith(".npz") or u.endswith(".png") or u.endswith(".mp4") or u.endswith(".csv")))
        ),
        min_bytes=MIN_DEPTH_BYTES,
    )
    timestamp_color_standard = _has_standard_color_timestamps(lower, min_bytes=MIN_DEPTH_BYTES)
    timestamp_color_session_named = _has_session_named_color_timestamps(lower, session, min_bytes=MIN_DEPTH_BYTES)
    timestamp_color_source = _color_timestamp_source(lower, session, min_bytes=MIN_DEPTH_BYTES)
    timestamp_color = timestamp_color_standard or timestamp_color_session_named
    timestamp_depth = _has_any(lower, lambda u: "depth_timestamps" in u and u.endswith(".csv"), min_bytes=MIN_DEPTH_BYTES)
    timestamp_radar = _has_any(lower, lambda u: "radar_timestamps" in u and u.endswith(".csv"), min_bytes=MIN_DEPTH_BYTES)
    meta_ok = _has_any(lower, lambda u: u.endswith("meta_data.json") or u.endswith("metadata.json"), min_bytes=MIN_DEPTH_BYTES)

    radar_ok = radar_bin_count > 0
    color_ok = color_count > 0
    depth_ok = depth_count > 0 and timestamp_color

    if radar_ok and color_ok and depth_ok:
        raw_material_class = "gold_standard"
        decision = "stage"
        reason = "radar_bin+color+depth present"
    elif radar_ok and color_ok:
        raw_material_class = "nonstandard"
        decision = "stage"
        reason = "radar_bin+color present; depth missing or incomplete"
    else:
        raw_material_class = "unusable"
        decision = "skip"
        missing = []
        if not radar_ok:
            missing.append("radar_bin")
        if not color_ok:
            missing.append("color_mp4")
        reason = "missing_or_invalid_" + "+".join(missing)

    return {
        **ref,
        "dataset_id": dataset_id,
        "raw_status": "present",
        "raw_object_count": len(objs),
        "raw_radar_bin_count": radar_bin_count,
        "raw_color_mp4_count": color_count,
        "raw_depth_asset_count": depth_count,
        "raw_color_timestamps": timestamp_color,
        "raw_color_timestamps_standard": timestamp_color_standard,
        "raw_color_timestamps_session_csv": timestamp_color_session_named,
        "raw_color_timestamps_source": timestamp_color_source,
        "raw_depth_timestamps": timestamp_depth,
        "raw_radar_timestamps": timestamp_radar,
        "raw_ts_completeness": int(timestamp_color_standard) + int(timestamp_depth) + int(timestamp_radar),
        "raw_meta_ok": meta_ok,
        "raw_material_class": raw_material_class,
        "stage_decision": decision,
        "raw_reason": reason,
    }



def _unique_refs_by_uri(refs: list[dict[str, str]], uri_field: str) -> list[dict[str, str]]:
    """Remove exact duplicate discovered roots while preserving order."""
    out: list[dict[str, str]] = []
    seen: set[str] = set()
    for ref in refs:
        uri = str(ref.get(uri_field, "")).rstrip("/")
        if uri in seen:
            continue
        seen.add(uri)
        out.append(ref)
    return out


def _index_refs_by_session(refs: list[dict[str, str]], uri_field: str) -> dict[str, list[dict[str, str]]]:
    """Group discovered refs by canonical session name and drop exact URI duplicates."""
    indexed: dict[str, list[dict[str, str]]] = {}
    for ref in _unique_refs_by_uri(refs, uri_field):
        indexed.setdefault(str(ref["session"]), []).append(ref)
    return indexed


def _quality_rank(quality: str) -> int:
    return {
        "gold_standard": 0,
        "nonstandard": 1,
        "unusable": 2,
        "": 3,
    }.get(str(quality), 9)


def _processed_score(row: dict[str, Any]) -> tuple[int, int, int, int, str]:
    """Lower is better. Prefer training-ready processed sessions, then richer bad ones."""
    good_rank = 0 if row.get("processed_status") == "good" else 1
    missing_count = 0 if row.get("processed_status") == "good" else len(str(row.get("processed_reason", "")).split(";"))
    asset_score = -sum(
        int(bool(row.get(k)))
        for k in [
            "processed_label_ok",
            "processed_radar_tensor_ok",
            "processed_sync_ok",
            "processed_color_ok",
            "processed_radar_csv_ok",
            "processed_seg_ok",
            "processed_rd_ok",
        ]
    )
    object_score = -int(row.get("processed_object_count", 0) or 0)
    return (good_rank, missing_count, asset_score, object_score, str(row.get("processed_group", "")))


def _raw_score(row: dict[str, Any]) -> tuple[int, int, int, int, str]:
    """Lower is better. Prefer gold, then nonstandard, then richer raw folders."""
    return (
        _quality_rank(str(row.get("raw_material_class", ""))),
        -int(row.get("raw_depth_asset_count", 0) or 0),
        -int(row.get("raw_color_mp4_count", 0) or 0),
        -int(row.get("raw_radar_bin_count", 0) or 0),
        str(row.get("raw_group", "")),
    )


def _duplicate_summary(rows: list[dict[str, Any]], uri_field: str, group_field: str) -> dict[str, Any]:
    uris = [str(r.get(uri_field, "")).rstrip("/") for r in rows if str(r.get(uri_field, "")).strip()]
    groups = [str(r.get(group_field, "")) for r in rows if str(r.get(group_field, "")).strip()]
    return {
        f"{uri_field}_duplicate_count": max(0, len(uris) - 1),
        f"{uri_field}_duplicates": " | ".join(uris[1:]),
        f"{group_field}_duplicates": " | ".join(groups[1:]),
    }


def find_raw_for_session(session: str, raw_index: dict[str, list[dict[str, str]]]) -> dict[str, str] | None:
    candidates = raw_index.get(session, [])
    if candidates:
        return candidates[0]

    # Slow fallback for very messy buckets: derive the session root from any object
    # nested below a matching session directory, not just root/*/session_name/.
    hits = _remote_ls(f"{AUDIT_RAW_ROOT_URI.rstrip('/')}/**", recursive=True, allow_empty=True)
    marker = f"/{session}/"
    for obj_uri in hits:
        if marker not in obj_uri:
            continue
        session_uri = obj_uri.split(marker, 1)[0] + f"/{session}"
        rel_parent = session_uri.rstrip("/")[len(AUDIT_RAW_ROOT_URI.rstrip("/") + "/"):].rsplit("/", 1)[0]
        return {
            "session": session,
            "raw_group": rel_parent or "root",
            "raw_uri": session_uri.rstrip("/"),
        }
    return None


def choose_best_raw_for_session(
    session: str,
    raw_index: dict[str, list[dict[str, str]]],
) -> tuple[dict[str, Any] | None, list[dict[str, Any]]]:
    candidates = raw_index.get(session, [])
    if not candidates:
        fallback = find_raw_for_session(session, raw_index)
        candidates = [fallback] if fallback else []
    if not candidates:
        return None, []
    rows = [classify_raw_session(ref) for ref in _unique_refs_by_uri(candidates, "raw_uri")]
    rows = sorted(rows, key=_raw_score)
    best = rows[0]
    best.update(_duplicate_summary(rows, "raw_uri", "raw_group"))
    return best, rows[1:]


def choose_best_processed_for_session(
    session: str,
    processed_candidates: list[dict[str, str]],
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    rows = [classify_processed_session(ref) for ref in _unique_refs_by_uri(processed_candidates, "processed_uri")]
    rows = sorted(rows, key=_processed_score)
    best = rows[0]
    best.update(_duplicate_summary(rows, "processed_uri", "processed_group"))
    return best, rows[1:]


def build_processing_audit(max_sessions: int | None = MAX_AUDIT_SESSIONS) -> pd.DataFrame:
    processed_refs = list_processed_session_refs()
    raw_refs = list_raw_session_refs() if INCLUDE_RAW_NOT_IN_PROCESSED else []

    processed_index = _index_refs_by_session(processed_refs, "processed_uri")
    raw_index = _index_refs_by_session(raw_refs, "raw_uri")

    processed_sessions = sorted(processed_index)
    if max_sessions is not None:
        processed_sessions = processed_sessions[: int(max_sessions)]

    rows: list[dict[str, Any]] = []
    processed_seen: set[str] = set()

    duplicate_processed_sessions = {
        session: refs for session, refs in processed_index.items() if len(_unique_refs_by_uri(refs, "processed_uri")) > 1
    }
    duplicate_raw_sessions = {
        session: refs for session, refs in raw_index.items() if len(_unique_refs_by_uri(refs, "raw_uri")) > 1
    }
    print(f"Found processed session refs: {len(processed_refs)}")
    print(f"Unique processed sessions: {len(processed_index)}")
    print(f"Processed duplicate session names: {len(duplicate_processed_sessions)}")
    print(f"Found raw session refs: {len(raw_refs)}")
    print(f"Unique raw sessions: {len(raw_index)}")
    print(f"Raw duplicate session names: {len(duplicate_raw_sessions)}")

    for idx, session in enumerate(processed_sessions, start=1):
        prefs = processed_index[session]
        print(f"[processed audit {idx}/{len(processed_sessions)}] {session} candidates={len(prefs)}")
        processed_seen.add(session)
        prow, processed_dups = choose_best_processed_for_session(session, prefs)

        if processed_dups:
            print(
                f"  duplicate processed candidates skipped for {session}: "
                f"{[r.get('processed_group', '') for r in processed_dups]}",
                flush=True,
            )

        if prow["processed_status"] == "good":
            rows.append({
                **prow,
                "raw_status": "",
                "raw_material_class": "",
                "stage_decision": "skip",
                "output_group": "",
                "raw_reason": "processed session already training-ready; duplicate candidates skipped if present",
            })
            continue

        rrow, raw_dups = choose_best_raw_for_session(session, raw_index)
        if rrow is None:
            rows.append({
                **prow,
                "raw_status": "missing",
                "raw_material_class": "unusable",
                "stage_decision": "skip",
                "output_group": "",
                "raw_reason": "processed session bad but matching raw session was not found",
            })
            continue

        if raw_dups:
            print(
                f"  duplicate raw candidates skipped for {session}: "
                f"{[r.get('raw_group', '') for r in raw_dups]}",
                flush=True,
            )
        rows.append({**prow, **rrow})

    if INCLUDE_RAW_NOT_IN_PROCESSED:
        raw_only_sessions = sorted(session for session in raw_index if session not in processed_seen)
        if max_sessions is not None:
            raw_only_sessions = raw_only_sessions[: int(max_sessions)]
        print(f"Found raw-only / unprocessed unique sessions: {len(raw_only_sessions)}")
        for idx, session in enumerate(raw_only_sessions, start=1):
            candidates = raw_index[session]
            print(f"[raw-only audit {idx}/{len(raw_only_sessions)}] {session} candidates={len(candidates)}")
            rrow, raw_dups = choose_best_raw_for_session(session, raw_index)
            if rrow is None:
                continue
            if raw_dups:
                print(
                    f"  duplicate raw-only candidates skipped for {session}: "
                    f"{[r.get('raw_group', '') for r in raw_dups]}",
                    flush=True,
                )
            rows.append({
                "session": session,
                "processed_group": "",
                "processed_uri": "",
                "processed_status": "missing",
                "processed_reason": "no processed session found",
                "processed_uri_duplicate_count": 0,
                "processed_uri_duplicates": "",
                "processed_group_duplicates": "",
                **rrow,
            })

    df = pd.DataFrame(rows)

    def _audit_series(frame: pd.DataFrame, col: str, default):
        """Return an existing column or a same-length default Series.

        This avoids AttributeError from pd.to_numeric(df.get(col, 0)).fillna(...)
        when the audit rows do not include an optional count column.
        """
        if col in frame.columns:
            return frame[col]
        return pd.Series(default, index=frame.index)

    if not df.empty:
        # Final safety net: one audit/staging row per session. Keep the best stageable row.
        df["_stage_rank"] = _audit_series(df, "stage_decision", "skip").map({"stage": 0, "skip": 1}).fillna(9)
        df["_quality_rank"] = _audit_series(df, "quality_class", "unusable").map({
            "gold_standard": 0,
            "nonstandard": 1,
            "unusable": 2,
            "already_good": 3,
        }).fillna(9)
        df["_radar_rank"] = -pd.to_numeric(_audit_series(df, "raw_radar_bin_count", 0), errors="coerce").fillna(0)
        df["_color_rank"] = -pd.to_numeric(_audit_series(df, "raw_color_mp4_count", 0), errors="coerce").fillna(0)
        df["_depth_rank"] = -pd.to_numeric(_audit_series(df, "raw_depth_asset_count", 0), errors="coerce").fillna(0)
        # Prefer copies with explicit timestamp CSVs (dataset2/dataset3: ts_completeness=3)
        # over partial copies from other raw-extract folders (ts_completeness=0).
        df["_ts_completeness_rank"] = -pd.to_numeric(_audit_series(df, "raw_ts_completeness", 0), errors="coerce").fillna(0)
        df = (
            df.sort_values(["session", "_stage_rank", "_quality_rank", "_ts_completeness_rank", "_depth_rank", "_color_rank", "_radar_rank"])
              .drop_duplicates("session", keep="first")
              .sort_values(["_stage_rank", "_quality_rank", "session"])
              .drop(columns=["_stage_rank", "_quality_rank", "_radar_rank", "_color_rank", "_depth_rank", "_ts_completeness_rank"])
              .reset_index(drop=True)
        )
    return df



STAGED_MANIFEST_COLUMNS = [
    "tag", "group", "prefix", "name", "uri",
    "dataset_id", "session", "raw_group", "raw_root_uri", "raw_uri", "raw_material_class",
    "destination_root_uri", "raw_reason",
    "raw_radar_bin_count", "raw_color_mp4_count", "raw_depth_asset_count",
    "raw_color_timestamps", "raw_color_timestamps_standard",
    "raw_color_timestamps_session_csv", "raw_color_timestamps_source",
    "raw_depth_timestamps", "raw_radar_timestamps", "raw_ts_completeness",
    "processed_group", "processed_status", "processed_reason",
    "raw_uri_duplicate_count", "raw_uri_duplicates", "raw_group_duplicates",
    "processed_uri_duplicate_count", "processed_uri_duplicates", "processed_group_duplicates",
]


def print_audit_stage_diagnostics(df: pd.DataFrame) -> None:
    print("\nAudit/staging diagnostics:")
    if df is None or df.empty:
        print("  audit dataframe is empty: no processed or raw session refs were discovered")
        print("  check AUDIT_RAW_ROOT_URI_CANDIDATES:", AUDIT_RAW_ROOT_URI_CANDIDATES)
        print("  check AUDIT_PROCESSED_ROOT_URI:", AUDIT_PROCESSED_ROOT_URI)
        print("  check RECURSIVE_SESSION_DISCOVERY:", RECURSIVE_SESSION_DISCOVERY)
        return
    for col in ["stage_decision", "raw_material_class", "raw_status", "processed_status", "raw_reason", "processed_reason"]:
        if col in df.columns:
            print(f"  {col}:", df[col].fillna("").astype(str).value_counts(dropna=False).head(20).to_dict())
    if "stage_decision" in df.columns:
        staged = df[df["stage_decision"].astype(str).eq("stage")]
        print("  staged rows:", len(staged))
        if staged.empty:
            print("  Nothing was staged. This can be valid if all processed sessions are already training-ready,")
            print("  or if raw candidates are missing radar/color material required for preprocessing.")
            cols = [c for c in ["session", "dataset_id", "processed_status", "raw_material_class", "raw_status", "raw_reason", "processed_reason", "raw_root_uri", "raw_uri"] if c in df.columns]
            print("  First non-staged examples:")
            try:
                print(df[cols].head(20).to_string(index=False))
            except Exception:
                print(df.head(20).to_string(index=False))


def write_audit_and_stage_manifest(df: pd.DataFrame) -> tuple[Path, Path]:
    AUDIT_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    df.to_csv(AUDIT_REPORT_CSV, index=False)

    staged = df[df["stage_decision"].eq("stage")].copy() if "stage_decision" in df.columns else pd.DataFrame()
    if not staged.empty:
        staged = staged.sort_values(["session", "raw_material_class"]).drop_duplicates("session", keep="first").reset_index(drop=True)
        staged["dataset_id"] = staged["dataset_id"].fillna(staged["raw_group"]).astype(str)
        staged["tag"] = staged["dataset_id"]
        staged["group"] = staged["dataset_id"]
        staged["prefix"] = SESSION_PREFIX
        staged["name"] = staged["session"]
        staged["uri"] = staged["raw_uri"]
        staged["destination_root_uri"] = ONEFORMER_INPUT_ROOT_URI
        for col in STAGED_MANIFEST_COLUMNS:
            if col not in staged.columns:
                staged[col] = ""
        staged = staged[STAGED_MANIFEST_COLUMNS]
    else:
        staged = pd.DataFrame(columns=STAGED_MANIFEST_COLUMNS)

    staged.to_csv(STAGED_MANIFEST_CSV, index=False)

    print("Audit rows:", len(df))
    print("Unique sessions in audit:", df["session"].nunique() if not df.empty and "session" in df.columns else 0)
    print("Stage rows:", len(staged))
    print("Unique sessions staged:", staged["session"].nunique() if not staged.empty else 0)
    print("Raw material counts:", df["raw_material_class"].value_counts(dropna=False).to_dict() if not df.empty and "raw_material_class" in df.columns else {})
    print("Stage dataset counts:", staged["group"].value_counts(dropna=False).to_dict() if not staged.empty else {})
    if staged.empty:
        print_audit_stage_diagnostics(df)

    gcloud_storage_cp(AUDIT_REPORT_CSV, f"{AUDIT_OUTPUT_ROOT_URI.rstrip('/')}/{AUDIT_REPORT_CSV.name}")
    gcloud_storage_cp(STAGED_MANIFEST_CSV, f"{AUDIT_OUTPUT_ROOT_URI.rstrip('/')}/{STAGED_MANIFEST_CSV.name}")
    return AUDIT_REPORT_CSV, STAGED_MANIFEST_CSV


def run_audit_and_write_manifest(max_sessions: int | None = MAX_AUDIT_SESSIONS) -> pd.DataFrame:
    df = build_processing_audit(max_sessions=max_sessions)
    write_audit_and_stage_manifest(df)
    return df


# -----------------------------------------------------------------------------
# Override the processing engine record source with the audit manifest.
# -----------------------------------------------------------------------------

def raw_session_records() -> list[dict]:
    if not STAGED_MANIFEST_CSV.exists() or FORCE_REAUDIT:
        if AUTO_RUN_AUDIT_IF_STAGE_EMPTY:
            print(f"Missing staged manifest: {STAGED_MANIFEST_CSV}. Running audit now...")
            run_audit_and_write_manifest()
        else:
            raise FileNotFoundError(
                f"Missing staged manifest: {STAGED_MANIFEST_CSV}. "
                "Run run_audit_and_write_manifest() first."
            )

    try:
        df = pd.read_csv(STAGED_MANIFEST_CSV)
    except pd.errors.EmptyDataError:
        print(f"Staged manifest exists but is zero-byte/has no header: {STAGED_MANIFEST_CSV}")
        df = pd.DataFrame(columns=STAGED_MANIFEST_COLUMNS)
    if df.empty:
        if AUTO_RUN_AUDIT_IF_STAGE_EMPTY:
            print(f"Staged manifest is empty: {STAGED_MANIFEST_CSV}. Re-running audit once...")
            audit_df = run_audit_and_write_manifest()
            df = pd.read_csv(STAGED_MANIFEST_CSV)
            if df.empty:
                print_audit_stage_diagnostics(audit_df)
        if df.empty:
            msg = (
                "Staged manifest contains zero sessions. CPU Stage 1 has nothing to prepare. "
                "This usually means the audit found no bad/missing processed sessions with usable raw material."
            )
            if str(EMPTY_STAGE_MANIFEST_BEHAVIOR).lower() == "error":
                raise RuntimeError(msg)
            print("NO-OP:", msg)
            return []

    records: list[dict] = []
    for _, row in df.iterrows():
        records.append({
            "tag": str(row["tag"]),
            "group": str(row["group"]),
            "prefix": str(row["prefix"]),
            "name": str(row["name"]),
            "uri": str(row["uri"]),
            "quality_class": str(row.get("quality_class", row["tag"])),
            "raw_group": str(row.get("raw_group", "")),
            "raw_root_uri": str(row.get("raw_root_uri", "")),
            "destination_root_uri": str(row.get("destination_root_uri", "")),
        })
    return sorted(records, key=lambda r: (r["group"], r["name"]))


# -----------------------------------------------------------------------------
# Processing-only finalization: post-OneFormer CPU work + compressed uploads.
# -----------------------------------------------------------------------------

def _label_csv_for_session(session_dir: Path) -> Path | None:
    for name in ["labeled_radar_points_v4_fused.csv", "labeled_radar_points_v4.csv"]:
        path = session_dir / name
        if has_data_rows(path):
            return path
    return None


def _coverage_mask_from_labels(df: pd.DataFrame):
    invalid = {"", "nan", "none", "unknown", "unlabeled"}
    # Prefer the semantic-teacher field. bucket_4class is materialized later
    # for every point, so using it first would make a failed autolabel look 100% covered.
    for col in ["cam_semantic_bucket", "bucket", "bucket_3class", "bucket_4class", "point_label_source"]:
        if col in df.columns:
            values = df[col].fillna("").astype(str).str.strip().str.lower()
            return col, ~values.isin(invalid)
    return "", pd.Series(False, index=df.index)


def _session_autolabel_coverage(session_dir: Path) -> tuple[float, str, Path | None]:
    label_csv = _label_csv_for_session(session_dir)
    if label_csv is None:
        return 0.0, "missing_label_csv", None
    df = pd.read_csv(label_csv, low_memory=False)
    if df.empty:
        return 0.0, "empty_label_csv", label_csv
    col, mask = _coverage_mask_from_labels(df)
    if not col:
        return 0.0, "missing_coverage_columns", label_csv
    return float(mask.mean()), col, label_csv


def _append_note(existing: str, extra: str) -> str:
    existing = str(existing or "").strip()
    extra = str(extra or "").strip()
    if not existing:
        return extra
    if not extra:
        return existing
    return existing if extra in existing else f"{existing} | {extra}"


def _catalog_note_for_session(catalog: pd.DataFrame, session_id: str, dataset_id: str) -> str:
    rows = catalog[(catalog["session_id"].astype(str) == str(session_id)) & (catalog["dataset_id"].astype(str) == str(dataset_id))]
    if rows.empty:
        return ""
    return str(rows.iloc[0].get("notes", "") or "")


def _update_catalog_session(
    catalog: pd.DataFrame,
    *,
    session_id: str,
    dataset_id: str,
    quality_tier: str,
    processed_path: str,
    notes: str,
) -> pd.DataFrame:
    merged_notes = _append_note(_catalog_note_for_session(catalog, session_id, dataset_id), notes)
    return admit_session(
        catalog,
        session_id=session_id,
        dataset_id=dataset_id,
        quality_tier=quality_tier,
        label_status="curated",
        processed_path=processed_path,
        notes=merged_notes,
    )


def _quarantine_note(batch_id: str, dataset_id: str, reason: str) -> str:
    return f"nb1_quarantine batch={batch_id} dataset_id={dataset_id} reason={reason}"


def _make_session_archive(session_dir: Path, archive_root: Path) -> Path:
    archive_root.mkdir(parents=True, exist_ok=True)
    archive_path = archive_root / f"{session_dir.name}.tar.gz"
    if archive_path.exists():
        archive_path.unlink()
    # Keep the session directory itself as the top-level member. Lower compression
    # level is deliberate here: these are large preprocessing artifacts, and wall-clock
    # throughput matters more than squeezing the last few percent of archive size.
    with tarfile.open(archive_path, "w:gz", compresslevel=int(FINALIZE_TAR_COMPRESSLEVEL)) as tar:
        tar.add(session_dir, arcname=session_dir.name)
    return archive_path


def _upload_processed_session(session_dir: Path, dataset_id: str, batch_id: str) -> dict[str, Any]:
    coverage, coverage_basis, label_csv = _session_autolabel_coverage(session_dir)
    unresolved_flags = [x for x in SKIPPED_SESSION_LOG if str(x.get("session")) == session_dir.name]
    if unresolved_flags:
        quality_tier = "quarantine"
        notes = _quarantine_note(batch_id, dataset_id, f"unresolved_skip_flags:{len(unresolved_flags)}")
        destination_root = f"{CURATED_QUARANTINE_ROOT_URI.rstrip('/')}/{dataset_id}"
    elif coverage >= float(MIN_AUTOLABEL_COVERAGE):
        quality_tier = "gold"
        notes = f"nb1_curated batch={batch_id} dataset_id={dataset_id} coverage={coverage:.4f} basis={coverage_basis}"
        destination_root = f"{CURATED_GOLD_ROOT_URI.rstrip('/')}/{dataset_id}"
    else:
        quality_tier = "silver"
        notes = (
            f"nb1_curated batch={batch_id} dataset_id={dataset_id} coverage={coverage:.4f} "
            f"basis={coverage_basis} threshold={MIN_AUTOLABEL_COVERAGE:.2f}"
        )
        destination_root = f"{CURATED_SILVER_ROOT_URI.rstrip('/')}/{dataset_id}"

    archive_uri = f"{destination_root.rstrip('/')}/{session_dir.name}.tar.gz"
    if not FORCE_REUPLOAD_PROCESSED_SESSIONS and _remote_file_size(archive_uri) > 0:
        print(f"[UPLOAD SKIP] {quality_tier}/{dataset_id}/{session_dir.name}: archive already exists at {archive_uri}", flush=True)
        return {
            "session": session_dir.name,
            "dataset_id": dataset_id,
            "quality_tier": quality_tier,
            "batch_id": batch_id,
            "archive_path": "",
            "archive_uri": archive_uri,
            "autolabel_coverage": coverage,
            "coverage_basis": coverage_basis,
            "label_csv": str(label_csv) if label_csv else "",
            "catalog_label_status": "curated",
            "notes": notes,
            "skipped_existing": True,
            "created_unix": int(time.time()),
        }

    local_archives = WORK_ROOT / "rigorous_preprocess_archives" / quality_tier / dataset_id
    archive_path = _make_session_archive(session_dir, local_archives)
    if UPLOAD_COMPRESSED_ARCHIVES:
        gcloud_storage_cp(archive_path, archive_uri)

    manifest = {
        "session": session_dir.name,
        "dataset_id": dataset_id,
        "quality_tier": quality_tier,
        "batch_id": batch_id,
        "archive_path": str(archive_path),
        "archive_uri": archive_uri if UPLOAD_COMPRESSED_ARCHIVES else "",
        "autolabel_coverage": coverage,
        "coverage_basis": coverage_basis,
        "label_csv": str(label_csv) if label_csv else "",
        "catalog_label_status": "curated",
        "notes": notes,
        "skipped_existing": False,
        "created_unix": int(time.time()),
    }
    manifest_path = archive_path.with_suffix("").with_suffix(".manifest.json")
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    gcloud_storage_cp(manifest_path, f"{destination_root.rstrip('/')}/{manifest_path.name}")
    return manifest


def _upload_processed_sessions_parallel(
    session_jobs: list[tuple[Path, str, str]],
) -> list[dict[str, Any]]:
    """Compress/upload processed session folders concurrently.

    Each job is independent: one session directory -> one .tar.gz + one manifest.
    This makes CPU Stage 3 much faster on Colab because tar/gzip and gcloud uploads
    no longer run strictly one session at a time.
    """
    if not session_jobs:
        return []

    workers = max(1, min(int(FINALIZE_UPLOAD_WORKERS), len(session_jobs)))
    print(
        f"[UPLOAD POOL] sessions={len(session_jobs)} workers={workers} "
        f"compresslevel={FINALIZE_TAR_COMPRESSLEVEL}",
        flush=True,
    )

    if workers == 1:
        out: list[dict[str, Any]] = []
        for session_dir, dataset_id, batch_id in session_jobs:
            try:
                manifest = _upload_processed_session(session_dir, dataset_id, batch_id)
                out.append(manifest)
                print(f"[UPLOAD OK] {dataset_id}/{session_dir.name} -> {manifest.get('quality_tier', 'unknown')}", flush=True)
            except Exception as exc:
                _record_skipped_session(session_dir, "processed-upload-failed", repr(exc))
                print(f"[UPLOAD FAILED] {dataset_id}/{session_dir.name}: {exc}", flush=True)
        return out

    out: list[dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        future_to_job = {
            pool.submit(_upload_processed_session, session_dir, dataset_id, batch_id): (
                session_dir, dataset_id, batch_id
            )
            for session_dir, dataset_id, batch_id in session_jobs
        }
        for fut in as_completed(future_to_job):
            session_dir, dataset_id, batch_id = future_to_job[fut]
            try:
                manifest = fut.result()
                out.append(manifest)
                print(f"[UPLOAD OK] {dataset_id}/{session_dir.name} -> {manifest.get('quality_tier', 'unknown')}", flush=True)
            except Exception as exc:
                _record_skipped_session(session_dir, "processed-upload-failed", repr(exc))
                print(f"[UPLOAD FAILED] {dataset_id}/{session_dir.name}: {exc}", flush=True)
    return out


def _sync_curation_catalog(upload_manifests: list[dict[str, Any]]) -> None:
    """Commit Stage 4 admissions once per CPU run, after parallel uploads finish.

    The archive is uploaded first; only then is its GCS URI made authoritative
    in the catalog. This prevents a catalog row from advertising a missing archive.
    """
    if not upload_manifests:
        return
    sync_catalog_from_gcs(CATALOG_LOCAL_PATH, GCS_CATALOG_URI)
    catalog = load_catalog(CATALOG_LOCAL_PATH)
    for manifest in upload_manifests:
        archive_uri = str(manifest.get("archive_uri", ""))
        if not archive_uri:
            raise RuntimeError(f"Cannot admit {manifest.get('session', '<unknown>')}: archive URI is empty")
        catalog = _update_catalog_session(
            catalog,
            session_id=str(manifest["session"]),
            dataset_id=str(manifest["dataset_id"]),
            quality_tier=str(manifest["quality_tier"]),
            processed_path=archive_uri,
            notes=str(manifest.get("notes", "")),
        )
    catalog.to_csv(CATALOG_LOCAL_PATH, index=False)
    sync_catalog_to_gcs(CATALOG_LOCAL_PATH, GCS_CATALOG_URI)
    print(f"Updated live session catalog for {len(upload_manifests)} curated session(s).")


def run_cpu_finalize_processed_uploads(max_batches: int | None = None) -> Path:
    """CPU Stage 3: pull OneFormer outputs, finish preprocessing, compress per-session outputs."""
    _reset_skipped_session_log()
    batches = _records_to_batches(max_batches)
    shutil.rmtree(CPU_POST_ONEFORMER_LOCAL_ROOT, ignore_errors=True)
    CPU_POST_ONEFORMER_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    upload_manifests: list[dict[str, Any]] = []
    for batch_idx, batch_records in enumerate(batches, start=1):
        bid = _batch_id(batch_idx)
        print(f"\n{'='*80}\nCPU Stage 3 — {bid}: finalize processed sessions and upload archives\n{'='*80}")
        batch_root = CPU_POST_ONEFORMER_LOCAL_ROOT / bid
        shutil.rmtree(batch_root, ignore_errors=True)
        batch_root.mkdir(parents=True, exist_ok=True)

        def process_group(args):
            (tag, group, prefix), items = args

            # Tier assignment depends on the completed session's measured coverage,
            # so only _upload_processed_session can choose its destination. Its
            # per-session archive check is the rerun guard; never resurrect the
            # former gold/nonstandard output-root shortcut here.
            src = f"{ONEFORMER_OUTPUT_ROOT_URI.rstrip('/')}/{bid}/{group}"
            group_root = batch_root / group
            if not _download_stage_group(src, group_root):
                return []

            ok = _post_oneformer_preprocess_group_root(group_root, prefix)
            remaining = active_session_dirs(group_root, prefix)
            if not ok or not remaining:
                print(f"[CPU FINALIZE] skipping {tag}/{group}: no usable sessions after post-OneFormer preprocessing.")
                return []

            session_jobs = [(session_dir, str(group), bid) for session_dir in remaining]
            return _upload_processed_sessions_parallel(session_jobs)

        # Process groups concurrently
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            results = executor.map(process_group, _group_records(batch_records).items())
            for res in results:
                if res:
                    upload_manifests.extend(res)

    _sync_curation_catalog(upload_manifests)

    manifest_df = pd.DataFrame(upload_manifests)
    out_csv = AUDIT_LOCAL_ROOT / "processed_upload_manifest.csv"
    manifest_df.to_csv(out_csv, index=False)
    gcloud_storage_cp(out_csv, f"{AUDIT_OUTPUT_ROOT_URI.rstrip('/')}/{out_csv.name}")

    skipped_path = _write_skip_log(AUDIT_LOCAL_ROOT)
    if skipped_path and skipped_path.exists():
        gcloud_storage_cp(skipped_path, f"{AUDIT_OUTPUT_ROOT_URI.rstrip('/')}/skipped_sessions_finalize.csv")

    print("Uploaded processed sessions:", len(upload_manifests))
    print("Upload manifest:", out_csv)
    return out_csv

In [ ]:
# @title defs: multithreaded audit building
import concurrent.futures
import pandas as pd
from typing import Any

# Redefine the audit builder to use multithreading for faster network I/O
def build_processing_audit(max_sessions: int | None = MAX_AUDIT_SESSIONS) -> pd.DataFrame:
    processed_refs = list_processed_session_refs()
    raw_refs = list_raw_session_refs() if INCLUDE_RAW_NOT_IN_PROCESSED else []

    processed_index = _index_refs_by_session(processed_refs, "processed_uri")
    raw_index = _index_refs_by_session(raw_refs, "raw_uri")

    processed_sessions = sorted(processed_index)
    if max_sessions is not None:
        processed_sessions = processed_sessions[: int(max_sessions)]

    rows: list[dict[str, Any]] = []
    processed_seen: set[str] = set(processed_sessions)

    duplicate_processed_sessions = {
        session: refs for session, refs in processed_index.items() if len(_unique_refs_by_uri(refs, "processed_uri")) > 1
    }
    duplicate_raw_sessions = {
        session: refs for session, refs in raw_index.items() if len(_unique_refs_by_uri(refs, "raw_uri")) > 1
    }
    print(f"Found processed session refs: {len(processed_refs)}")
    print(f"Unique processed sessions: {len(processed_index)}")
    print(f"Processed duplicate session names: {len(duplicate_processed_sessions)}")
    print(f"Found raw session refs: {len(raw_refs)}")
    print(f"Unique raw sessions: {len(raw_index)}")
    print(f"Raw duplicate session names: {len(duplicate_raw_sessions)}")

    def process_processed_session(args):
        idx, session = args
        prefs = processed_index[session]
        print(f"[processed audit {idx}/{len(processed_sessions)}] {session} candidates={len(prefs)}")
        prow, processed_dups = choose_best_processed_for_session(session, prefs)

        if processed_dups:
            print(
                f"  duplicate processed candidates skipped for {session}: "
                f"{[r.get('processed_group', '') for r in processed_dups]}",
                flush=True,
            )

        if prow["processed_status"] == "good":
            return {
                **prow,
                "raw_status": "",
                "quality_class": "already_good",
                "stage_decision": "skip",
                "output_group": "",
                "raw_reason": "processed session already training-ready; duplicate candidates skipped if present",
            }

        rrow, raw_dups = choose_best_raw_for_session(session, raw_index)
        if rrow is None:
            return {
                **prow,
                "raw_status": "missing",
                "quality_class": "unusable",
                "stage_decision": "skip",
                "output_group": "",
                "raw_reason": "processed session bad but matching raw session was not found",
            }

        if raw_dups:
            print(
                f"  duplicate raw candidates skipped for {session}: "
                f"{[r.get('raw_group', '') for r in raw_dups]}",
                flush=True,
            )
        return {**prow, **rrow}

    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        rows.extend(executor.map(process_processed_session, enumerate(processed_sessions, start=1)))

    if INCLUDE_RAW_NOT_IN_PROCESSED:
        raw_only_sessions = sorted(session for session in raw_index if session not in processed_seen)
        if max_sessions is not None:
            raw_only_sessions = raw_only_sessions[: int(max_sessions)]
        print(f"Found raw-only / unprocessed unique sessions: {len(raw_only_sessions)}")

        def process_raw_only_session(args):
            idx, session = args
            candidates = raw_index[session]
            print(f"[raw-only audit {idx}/{len(raw_only_sessions)}] {session} candidates={len(candidates)}")
            rrow, raw_dups = choose_best_raw_for_session(session, raw_index)
            if rrow is None:
                return None
            if raw_dups:
                print(
                    f"  duplicate raw-only candidates skipped for {session}: "
                    f"{[r.get('raw_group', '') for r in raw_dups]}",
                    flush=True,
                )
            return {
                "session": session,
                "processed_group": "",
                "processed_uri": "",
                "processed_status": "missing",
                "processed_reason": "no processed session found",
                "processed_uri_duplicate_count": 0,
                "processed_uri_duplicates": "",
                "processed_group_duplicates": "",
                **rrow,
            }

        with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
            raw_results = executor.map(process_raw_only_session, enumerate(raw_only_sessions, start=1))
            rows.extend(r for r in raw_results if r is not None)

    df = pd.DataFrame(rows)

    def _audit_series(frame: pd.DataFrame, col: str, default):
        """Return an existing column or a same-length default Series.

        This avoids AttributeError from pd.to_numeric(df.get(col, 0)).fillna(...)
        when the audit rows do not include an optional count column.
        """
        if col in frame.columns:
            return frame[col]
        return pd.Series(default, index=frame.index)

    if not df.empty:
        df["_stage_rank"] = _audit_series(df, "stage_decision", "skip").map({"stage": 0, "skip": 1}).fillna(9)
        df["_quality_rank"] = _audit_series(df, "quality_class", "unusable").map({
            "gold_standard": 0,
            "nonstandard": 1,
            "unusable": 2,
            "already_good": 3,
        }).fillna(9)
        df["_radar_rank"] = -pd.to_numeric(_audit_series(df, "raw_radar_bin_count", 0), errors="coerce").fillna(0)
        df["_color_rank"] = -pd.to_numeric(_audit_series(df, "raw_color_mp4_count", 0), errors="coerce").fillna(0)
        df["_depth_rank"] = -pd.to_numeric(_audit_series(df, "raw_depth_asset_count", 0), errors="coerce").fillna(0)
        df["_ts_completeness_rank"] = -pd.to_numeric(_audit_series(df, "raw_ts_completeness", 0), errors="coerce").fillna(0)
        df = (
            df.sort_values(["session", "_stage_rank", "_quality_rank", "_ts_completeness_rank", "_depth_rank", "_color_rank", "_radar_rank"])
              .drop_duplicates("session", keep="first")
              .sort_values(["_stage_rank", "_quality_rank", "session"])
              .drop(columns=["_stage_rank", "_quality_rank", "_radar_rank", "_color_rank", "_depth_rank", "_ts_completeness_rank"])
              .reset_index(drop=True)
        )
    return df

In [ ]:
# @title defs: multithreaded oneformer processing
import concurrent.futures
import time
import shutil
from pathlib import Path

def run_cpu_prepare_for_oneformer(max_batches: int | None = None) -> Path:
    """CPU/IO stage. Downloads raw sessions, runs radar CSV + meta, uploads
    per-batch OneFormer input folders to GCS.

    Use this on a CPU runtime. It intentionally does NOT run OneFormer.
    """
    _reset_skipped_session_log()
    batches = _records_to_batches(max_batches)
    shutil.rmtree(CPU_PRE_ONEFORMER_LOCAL_ROOT, ignore_errors=True)
    CPU_PRE_ONEFORMER_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    stage_manifest = {
        "stage": "cpu_prepare_for_oneformer",
        "created_unix": int(time.time()),
        "oneformer_input_root_uri": ONEFORMER_INPUT_ROOT_URI,
        "raw_extracts_root_uri": RAW_EXTRACTS_ROOT_URI,
        "audit_raw_root_uri_candidates": globals().get("AUDIT_RAW_ROOT_URI_CANDIDATES", [RAW_EXTRACTS_ROOT_URI]),
        "batches": [],
    }

    if not batches:
        print("\nCPU Stage 1 has no staged sessions to prepare. Exiting cleanly.")
        print("Staged manifest:", STAGED_MANIFEST_CSV)
        _upload_stage_manifest(CPU_PRE_ONEFORMER_LOCAL_ROOT, ONEFORMER_INPUT_ROOT_URI, "cpu_prepare_manifest.json", stage_manifest)
        return CPU_PRE_ONEFORMER_LOCAL_ROOT

    for batch_idx, batch_records in enumerate(batches, start=1):
        bid = _batch_id(batch_idx)
        print(f"\n{'='*80}\nCPU Stage 1 — {bid}: raw/radar/meta preparation\n{'='*80}")
        batch_root = CPU_PRE_ONEFORMER_LOCAL_ROOT / bid
        shutil.rmtree(batch_root, ignore_errors=True)
        batch_root.mkdir(parents=True, exist_ok=True)

        def process_group(args):
            (tag, group, prefix), items = args
            dst = f"{ONEFORMER_INPUT_ROOT_URI.rstrip('/')}/{bid}/{group}"

            # Idempotency: skip groups whose OneFormer inputs already exist on GCS.
            if not FORCE_REUPLOAD_ONEFORMER_INPUTS and _remote_prefix_has_objects(dst):
                print(f"[CPU PRE] {bid}/{group}: OneFormer inputs already present at {dst}; "
                      f"skipping (set FORCE_REUPLOAD_ONEFORMER_INPUTS=True to rebuild).")
                return {
                    "batch_id": bid, "tag": tag, "group": group, "prefix": prefix,
                    "n_input_records": len(items), "n_uploaded_sessions": 0,
                    "input_uri": dst, "sessions": [], "skipped_existing": True,
                }

            group_root = batch_root / group
            group_root.mkdir(parents=True, exist_ok=True)
            print(f"\nDownloading {len(items)} raw {tag}/{group} session(s) -> {group_root}")
            for xfer in chunked(items, SESSION_DOWNLOAD_BATCH_SIZE):
                gcloud_storage_cp([r["uri"] for r in xfer], group_root, recursive=True)
            _flatten_accidental_nested_group(group_root, group)
            ensure_color_timestamp_aliases(group_root, prefix)

            ok = run_radar_csv_and_meta_stage(group_root, prefix)
            remaining = active_session_dirs(group_root, prefix)
            if not ok or not remaining:
                print(f"[CPU PRE] skipping {tag}/{group}: no usable sessions after radar/meta stage.")
                return None

            print(f"\nUploading OneFormer input group -> {dst}")
            # delete_unmatched=False: never delete remote objects written by a prior run.
            gcloud_storage_rsync(group_root, dst, delete_unmatched=False)
            return {
                "batch_id": bid,
                "tag": tag,
                "group": group,
                "prefix": prefix,
                "n_input_records": len(items),
                "n_uploaded_sessions": len(remaining),
                "input_uri": dst,
                "sessions": [p.name for p in remaining],
            }

        # Use 4 workers to balance I/O throughput against Colab's SSD bandwidth
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            results = executor.map(process_group, _group_records(batch_records).items())
            for res in results:
                if res:
                    stage_manifest["batches"].append(res)

        print("\nSSD status:")
        run(["df", "-h", "/content"], check=False)

    skipped_path = _write_skip_log(CPU_PRE_ONEFORMER_LOCAL_ROOT)
    if skipped_path and skipped_path.exists():
        gcloud_storage_cp(skipped_path, f"{ONEFORMER_INPUT_ROOT_URI.rstrip('/')}/skipped_sessions_cpu_prepare.csv")
    _upload_stage_manifest(CPU_PRE_ONEFORMER_LOCAL_ROOT, ONEFORMER_INPUT_ROOT_URI, "cpu_prepare_manifest.json", stage_manifest)
    print("\nCPU pre-OneFormer stage complete:")
    print(" ", ONEFORMER_INPUT_ROOT_URI)
    return CPU_PRE_ONEFORMER_LOCAL_ROOT

## CPU Stage 1 — prepare staged raw sessions for OneFormer

Run on a CPU runtime. Uses `staged_sessions.csv`, downloads only stageable raw sessions, and uploads OneFormer inputs to the rigorous preprocessing stage prefix.

In [ ]:
# @title runtime: prep staged sessions
PREPROCESS_MAX_BATCHES = None

# This stage now exits cleanly iIf the audit produced zero staged rows.
# To inspect why nothing was staged, reun the audit cell above and check
# `audit_df["stage_decision"].value_counts()` plus `audit_df["raw_reason"].value_counts()`.
run_cpu_prepare_for_oneformer(max_batches=PREPROCESS_MAX_BATCHES)

## GPU Stage 2 — OneFormer label-map export

Run on an A100/high-RAM GPU runtime. Increase `ONEFORMER_FRAME_BATCH_SIZE` only if VRAM allows.

In [ ]:
# @title defs: pull oneformer large
!pip -q install -U huggingface_hub transformers safetensors accelerate

from huggingface_hub import snapshot_download
from pathlib import Path
import shutil

target = BRANCH1 / "processing" / "noah_scripts" / "OneFormer" / "oneformer_large"

# Remove any incomplete previous copy
if target.exists():
    shutil.rmtree(target)

snapshot_download(
    repo_id="shi-labs/oneformer_ade20k_swin_large",
    local_dir=str(target),
    local_dir_use_symlinks=False,
)

print("Downloaded files:")
for p in sorted(target.iterdir()):
    print(" ", p.name)

In [ ]:
# @title runtime: run gpu oneformer labelmap export
import sys
!{sys.executable} -m pip install opencv-python

ONEFORMER_MODEL = "large"
ONEFORMER_MAX_W = 1280
ONEFORMER_EVERY_N = 1
ONEFORMER_SAVE_PNG = False
ONEFORMER_FRAME_BATCH_SIZE = 64
ONEFORMER_USE_FP16 = True
ONEFORMER_TIMING = True

PREPROCESS_MAX_BATCHES = None
run_gpu_oneformer_labelmap_export(max_batches=PREPROCESS_MAX_BATCHES) # fixing the batch oneformer export

## CPU Stage 3 — finalize, recompress, and upload processed sessions

Run on a CPU runtime after GPU Stage 2. This finishes preprocessing and uploads compressed session archives only; it does not train a model.

In [ ]:
# @title runtime: finalize, recompress, curate, upload
PREPROCESS_MAX_BATCHES = 3

# Parallel CPU Stage 3/4 controls.
# Increase on high-CPU Colab runtimes; reduce if GCS throttles, RAM spikes, or uploads become flaky.
FINALIZE_UPLOAD_WORKERS = 32
FINALIZE_TAR_COMPRESSLEVEL = 2  # faster than gzip default; use 1 for max speed, 6-9 for smaller archives
MIN_AUTOLABEL_COVERAGE = 0.90

# Radar tensor export controls. The exporter is CPU/RAM/IO bound, not GPU-bound.
# With 64 GB CPU RAM, start at 4 workers; try 6 if memory stays comfortable.
RADAR_TENSOR_WORKERS = 4
RADAR_TENSOR_NPZ_COMPRESSION = "stored"
RADAR_TENSOR_SIDECAR_STRIP_MODE = "rewrite"
RADAR_TENSOR_SIDECAR_STRIP_COMPRESSION = "compressed"
RADAR_TENSOR_OVERWRITE = False
RADAR_TENSOR_QUIET = True

# Sessions that clear preprocessing are admitted into:
#   gs://.../curated/sessions/gold/<dataset_id>/<session_id>.tar.gz
#   gs://.../curated/sessions/silver/<dataset_id>/<session_id>.tar.gz
# Failures and unresolved flags are recorded in the live catalog as quarantine.
run_cpu_finalize_processed_uploads(max_batches=PREPROCESS_MAX_BATCHES)


## Optional: inspect manifests

In [ ]:
import pandas as pd
print("Audit CSV:", AUDIT_REPORT_CSV)
print("Staged CSV:", STAGED_MANIFEST_CSV)
print("Upload manifest:", AUDIT_LOCAL_ROOT / "processed_upload_manifest.csv")

display(pd.read_csv(AUDIT_REPORT_CSV).head(20))
if (AUDIT_LOCAL_ROOT / "processed_upload_manifest.csv").exists():
    display(pd.read_csv(AUDIT_LOCAL_ROOT / "processed_upload_manifest.csv").head(20))


In [ ]:
# @title optional: copies color timestamps from raw session locations
import os
from google.cloud import storage

def sync_session_files(project_id, bucket_name):
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)

    source_prefix = "CapstoneData/raw/sessions/dataset2"
    dest_base = f"CapstoneData/intermediate/oneformer/inputs/"

    # 1. Get all source session folders
    blobs = bucket.list_blobs(prefix=source_prefix, delimiter='/')
    list(blobs) # Consume iterator to populate prefixes
    source_sessions = [prefix.split('/')[-2] for prefix in blobs.prefixes]

    print(f"Found {len(source_sessions)} sessions to process.")

    for session_id in source_sessions:
        # Define the three files required per session
        files_to_copy = [
            f"{session_id}_color_timestamps.csv",
            f"{session_id}_color.mp4"
        ]

        # 2. Locate the correct destination batch folder for this session
        batch_blobs = bucket.list_blobs(prefix=dest_base, delimiter='/')
        list(batch_blobs)

        found_dest_batch_path = None
        current_batch_name = "Unknown"

        for batch_prefix in batch_blobs.prefixes:
            check_dest_path = f"{batch_prefix}dataset2/{session_id}/"
            # Verify the directory exists in this batch
            dest_blobs = bucket.list_blobs(prefix=check_dest_path, max_results=1)
            if any(dest_blobs):
                print(f"Found destination folder: {check_dest_path}")
                found_dest_batch_path = check_dest_path
                current_batch_name = batch_prefix.split('/')[-3]
                break

        if not found_dest_batch_path:
            print(f"Warning: Destination folder NOT FOUND for session {session_id}")
            continue

        print(f"Processing session {session_id} in {current_batch_name}:")

        # 3. Process the three specific files
        for target_filename in files_to_copy:
            source_blob_path = f"{source_prefix}{session_id}/{target_filename}"
            source_blob = bucket.blob(source_blob_path)

            if not source_blob.exists():
                print(f"  - Missing Source: {target_filename} {session_id}")
                continue

            dest_blob_path = f"{found_dest_batch_path}{target_filename}"
            dest_blob = bucket.blob(dest_blob_path)

            # No-Clobber logic: Skip if already at destination
            if dest_blob.exists():
                print(f"  - Skipped: {target_filename} (Already exists)")
            else:
                bucket.copy_blob(source_blob, bucket, dest_blob_path)
                print(f"  - Copied:  {target_filename}")

# Configuration
PROJECT_ID = "fluent-webbing-496616-u8"
BUCKET_NAME = "miamioh-resa-data"

sync_session_files(PROJECT_ID, BUCKET_NAME)


In [ ]:
# @title optional ADC pointcloud csv extractor
# ============================================================================
# STANDALONE BACKFILL CELL + ADC CONVERSION FALLBACK
# Generates {session}_radar_timestamps.csv and {session}_color_timestamps.csv
# for every session under DATASET_URI (recursive).
#
# If {session}.csv is missing, this cell downloads the raw {session}.bin,
# runs adc_to_pointcloud_v6.py locally, uploads the generated {session}.csv,
# then derives _radar_timestamps.csv from that generated point-cloud CSV.
#
#   _radar_timestamps.csv : recovered from {session}.csv
#       timestamp_us per radar_frame_num -> no ADC re-run if CSV already exists.
#   _color_timestamps.csv : per-frame presentation times read from
#       {session}_color.mp4, anchored to meta_data.json -> video.start_ms.
#
# Output format matches the recorder exactly: columns [frame_index, timestamp_ms].
# Idempotent: existing timestamp CSVs are skipped unless FORCE_REGEN_TIMESTAMPS=True.
# Assumes gcloud auth / GCS setup was done in an earlier cell.
# ============================================================================

import subprocess, sys, json, tempfile, shutil
from pathlib import Path
import pandas as pd

DATASET_URI = "gs://miamioh-resa-data/CapstoneData/raw/sessions/dataset3"

# ---- USER-FILL CONSTANTS -----------------------------------------------------
# Local path in the Colab/runtime repo checkout.
ADC_TO_POINTCLOUD_SCRIPT = Path("/content/work/code/branch1/processing/adc_to_pointcloud_v6.py")

# May be a local path OR a gs:// URI. Replace with your cfg path.
MMWAVE_CFG = "/content/work/code/config/profile_objdet.cfg"
# Example GCS form:
# MMWAVE_CFG = "gs://miamioh-resa-data/CapstoneData/code_v2/RESA_mmWave/config/profile_objdet.cfg"

# Keep this False for timestamp-only backfill. Set True only if you also want
# adc_to_pointcloud_v6.py to write hybrid_rd sidecars locally.
WRITE_RD_SIDECARS = True

# Extra CLI args forwarded to adc_to_pointcloud_v6.py.
ADC_EXTRA_ARGS = [
    # "--min-snr", "3.0",
    # "--pfa", "0.01",
    # "--clutter-mode", "off",
    # "--frame-number-offset", "1",
]




MAX_WORKERS = 16          # Start with 4. Try 6 or 8 if stable.
PRINT_EACH_SESSION = True


# ---- REGEN FLAGS -------------------------------------------------------------
FORCE_REGEN_TIMESTAMPS = False   # True -> overwrite timestamp CSVs
FORCE_ADC_REGEN_CSV = False      # True -> rerun ADC even if {session}.csv exists
UPLOAD_RD_SIDECARS = False       # True -> upload hybrid_rd/ when WRITE_RD_SIDECARS=True

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)
    import cv2


def _sh(args, *, check=False, label=None):
    """Run a command quietly; return CompletedProcess unless check=True fails."""
    r = subprocess.run(args, capture_output=True, text=True)
    if check and r.returncode != 0:
        cmd = " ".join(map(str, args))
        msg = (r.stderr or r.stdout or "").strip()
        raise RuntimeError(f"{label or 'command'} failed [{r.returncode}]: {cmd}\n{msg[:2000]}")
    return r

def _ls_recursive(uri):
    r = _sh(["gcloud", "storage", "ls", "--recursive", uri.rstrip("/") + "/**"], check=True, label="gcloud ls")
    return [l.strip() for l in r.stdout.splitlines() if l.strip().startswith("gs://")]

def _cp(src, dst):
    r = _sh(["gcloud", "storage", "cp", str(src), str(dst)])
    return r.returncode == 0, (r.stderr or r.stdout or "").strip()

def _cp_required(src, dst, what="copy"):
    ok, msg = _cp(src, dst)
    if not ok:
        raise RuntimeError(f"failed to {what}: {src} -> {dst}\n{msg[:1000]}")

def _cp_recursive_required(src, dst, what="recursive copy"):
    r = _sh(["gcloud", "storage", "cp", "--recursive", str(src), str(dst)])
    if r.returncode != 0:
        msg = (r.stderr or r.stdout or "").strip()
        raise RuntimeError(f"failed to {what}: {src} -> {dst}\n{msg[:1000]}")

def _gcs_basename(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1]

def _find_session_bin(root: str, name: str, objects: list[str]) -> str | None:
    preferred = f"{root}/{name}.bin"
    if preferred in objects:
        return preferred

    bins = [
        o for o in objects
        if o.startswith(root.rstrip("/") + "/")
        and o.lower().endswith(".bin")
        and "/" not in o[len(root.rstrip("/") + "/"):]
    ]

    if len(bins) == 1:
        return bins[0]

    return None

def _find_color_mp4(root: str, objects: list[str]) -> str | None:
    immediate = [
        o for o in objects
        if o.startswith(root.rstrip("/") + "/")
        and o.lower().endswith("_color.mp4")
        and "/" not in o[len(root.rstrip("/") + "/"):]
    ]
    return immediate[0] if immediate else None

def _materialize_cfg(cfg_ref: str, tmp_root: Path) -> Path:
    """Return a local cfg path. Supports local paths and gs:// URIs."""
    if str(cfg_ref).startswith("gs://"):
        local_cfg = tmp_root / _gcs_basename(str(cfg_ref))
        _cp_required(cfg_ref, local_cfg, what="download cfg")
        return local_cfg

    local_cfg = Path(cfg_ref)
    if not local_cfg.exists():
        raise FileNotFoundError(
            f"MMWAVE_CFG does not exist: {local_cfg}\n"
            "Set MMWAVE_CFG to your local cfg path or a gs:// URI."
        )

    return local_cfg

def _assert_adc_script():
    if not ADC_TO_POINTCLOUD_SCRIPT.exists():
        raise FileNotFoundError(
            f"ADC_TO_POINTCLOUD_SCRIPT does not exist: {ADC_TO_POINTCLOUD_SCRIPT}\n"
            "Set ADC_TO_POINTCLOUD_SCRIPT to branch1/processing/adc_to_pointcloud_v6.py."
        )


# ---- 1. discover session folders recursively --------------------------------
all_objs = _ls_recursive(DATASET_URI)
existing = set(all_objs)

session_roots = {}  # root_uri -> session_name
for obj in all_objs:
    parts = obj.split("/")
    for k, p in enumerate(parts):
        if p.startswith("session_") and "." not in p:  # folder, not a file
            session_roots["/".join(parts[:k + 1])] = p
            break

print(f"Discovered {len(session_roots)} session(s) under {DATASET_URI}\n")


# ---- 2. timestamp generation helpers ----------------------------------------
def _radar_timestamps(radar_csv: Path) -> pd.DataFrame:
    """
    frame_index <- recorder-compatible frame index.
    timestamp_ms <- timestamp_us / 1000.

    Notes:
      - adc_to_pointcloud_v6.py writes both frame_num and radar_frame_num.
      - By default, radar_frame_num = raw frame_index + 1.
      - Recorder timestamp CSVs expect frame_index, so we undo that +1 offset
        when radar_frame_num appears to be one-based.
    """
    head = pd.read_csv(radar_csv, nrows=0).columns
    fcol = "radar_frame_num" if "radar_frame_num" in head else "frame_num"
    tcol = "timestamp_us" if "timestamp_us" in head else "timestamp_ms"

    df = pd.read_csv(radar_csv, usecols=[fcol, tcol])
    if df.empty:
        return pd.DataFrame({
            "frame_index": pd.Series(dtype="int64"),
            "timestamp_ms": pd.Series(dtype="float64"),
        })

    g = df.groupby(fcol, as_index=False)[tcol].first().sort_values(fcol)
    ts_ms = g[tcol].astype(float) * (1e-3 if tcol == "timestamp_us" else 1.0)

    if fcol == "radar_frame_num" and g[fcol].min() >= 1:
        frame_index = g[fcol].astype(int) - 1
    else:
        frame_index = g[fcol].astype(int)

    return pd.DataFrame({
        "frame_index": frame_index,
        "timestamp_ms": ts_ms.round(3),
    })

def _color_timestamps(color_mp4: Path, start_ms: float, meta: dict) -> pd.DataFrame:
    """Per-frame mp4 presentation times, anchored to video.start_ms."""
    pos = []
    cap = cv2.VideoCapture(str(color_mp4))

    if cap.isOpened():
        while True:
            t = cap.get(cv2.CAP_PROP_POS_MSEC)
            if not cap.grab():
                break
            pos.append(t)

    cap.release()

    if not pos:
        v = meta.get("video", {})
        fps = v.get("actual_fps") or v.get("fps") or 30.0
        n = int(meta.get("depth", {}).get("num_frames", 0) or v.get("num_frames", 0) or 0)
        pos = [i * 1000.0 / float(fps) for i in range(n)]

    return pd.DataFrame({
        "frame_index": range(len(pos)),
        "timestamp_ms": [round(float(start_ms) + float(p), 3) for p in pos],
    })


# ---- 3. ADC conversion fallback ---------------------------------------------
def _run_adc_to_pointcloud(root: str, name: str, objects: list[str], tmp: Path, cfg_path: Path) -> Path:
    """
    Download raw bin, run adc_to_pointcloud_v6.py on a local session directory,
    upload {session}.csv back to GCS, and return the local generated CSV path.
    """
    _assert_adc_script()

    bin_uri = _find_session_bin(root, name, objects)
    if bin_uri is None:
        raise RuntimeError(f"missing or ambiguous .bin for {name}")

    local_session = tmp / name
    local_session.mkdir(parents=True, exist_ok=True)

    local_bin = local_session / _gcs_basename(bin_uri)
    _cp_required(bin_uri, local_bin, what="download raw ADC .bin")

    cmd = [
        sys.executable,
        str(ADC_TO_POINTCLOUD_SCRIPT),
        "--session", str(local_session),
        "--cfg", str(cfg_path),
        "--force",
        *ADC_EXTRA_ARGS,
    ]

    if WRITE_RD_SIDECARS:
        cmd.append("--write-rd-sidecars")

    r = _sh(cmd)

    if r.returncode != 0:
        msg = "\n".join([
            "adc_to_pointcloud_v6.py failed.",
            "--- stdout tail ---",
            (r.stdout or "")[-2000:],
            "--- stderr tail ---",
            (r.stderr or "")[-2000:],
        ])
        raise RuntimeError(msg)

    local_csv = local_session / f"{name}.csv"

    if not local_csv.exists():
        csv_candidates = sorted(
            p for p in local_session.glob("*.csv")
            if not p.name.endswith("_timestamps.csv")
        )

        if len(csv_candidates) == 1:
            local_csv = csv_candidates[0]
        else:
            raise RuntimeError(
                f"ADC conversion finished but no unique point-cloud CSV was found in {local_session}"
            )

    dst_csv_uri = f"{root}/{name}.csv"
    _cp_required(local_csv, dst_csv_uri, what="upload generated point-cloud CSV")

    if WRITE_RD_SIDECARS and UPLOAD_RD_SIDECARS:
        sidecar_dir = local_session / "hybrid_rd"
        if sidecar_dir.exists():
            _cp_recursive_required(sidecar_dir, f"{root}/hybrid_rd", what="upload RD sidecars")

    return local_csv


# ---- 4. per-session processing ----------------------------------------------
# ---- 4. parallel per-session processing --------------------------------------

existing_snapshot = frozenset(existing)

# Pre-index objects once so every worker does not scan all_objs repeatedly.
objs_by_root = {
    root: [o for o in all_objs if o.startswith(root.rstrip("/") + "/")]
    for root in session_roots
}

# Materialize cfg once. This avoids downloading the cfg once per session if MMWAVE_CFG is gs://.
_shared_cfg_tmp = Path(tempfile.mkdtemp(prefix="adc_cfg_"))
ADC_CFG_FOR_RUN = _materialize_cfg(str(MMWAVE_CFG), _shared_cfg_tmp)

print(f"Using ADC cfg: {ADC_CFG_FOR_RUN}")
print(f"Parallel workers: {MAX_WORKERS}\n")


def _process_one_session(item):
    root, name = item
    session_objs = objs_by_root[root]

    radar_csv_uri = f"{root}/{name}.csv"
    radar_ts_uri = f"{root}/{name}_radar_timestamps.csv"
    color_ts_uri = f"{root}/{name}_color_timestamps.csv"
    meta_uri = f"{root}/meta_data.json"

    need_radar_ts = FORCE_REGEN_TIMESTAMPS or radar_ts_uri not in existing_snapshot
    need_color_ts = FORCE_REGEN_TIMESTAMPS or color_ts_uri not in existing_snapshot
    need_adc_csv = FORCE_ADC_REGEN_CSV or radar_csv_uri not in existing_snapshot

    if not need_radar_ts and not need_color_ts and not need_adc_csv:
        return {
            "status": "skipped",
            "name": name,
            "adc": False,
            "radar_n": "existing",
            "color_n": "existing",
            "message": f"[skip] {name}: point-cloud CSV and both timestamp CSVs already present",
        }

    tmp = Path(tempfile.mkdtemp(prefix=f"{name}_"))

    try:
        local_radar = None
        radar_df = None
        color_df = None
        adc_ran = False

        if need_adc_csv:
            local_radar = _run_adc_to_pointcloud(
                root=root,
                name=name,
                objects=session_objs,
                tmp=tmp,
                cfg_path=ADC_CFG_FOR_RUN,
            )
            adc_ran = True
        else:
            local_radar = tmp / f"{name}.csv"
            _cp_required(radar_csv_uri, local_radar, what="download existing point-cloud CSV")

        if need_radar_ts:
            radar_df = _radar_timestamps(local_radar)

            out = tmp / f"{name}_radar_timestamps.csv"
            radar_df.to_csv(out, index=False)

            _cp_required(out, radar_ts_uri, what="upload radar timestamps")

        if need_color_ts:
            meta = {}

            if meta_uri in existing_snapshot:
                local_meta = tmp / "meta_data.json"
                try:
                    _cp_required(meta_uri, local_meta, what="download meta_data.json")
                    meta = json.loads(local_meta.read_text(encoding="utf-8"))
                except Exception:
                    meta = {}

            start_ms = float(meta.get("video", {}).get("start_ms", 0) or 0)

            if not start_ms:
                if radar_df is None:
                    radar_df = _radar_timestamps(local_radar)

                start_ms = float(radar_df["timestamp_ms"].iloc[0]) if len(radar_df) else 0.0

            color_uri = _find_color_mp4(root, session_objs)

            if color_uri is None:
                raise RuntimeError("no immediate *_color.mp4 found")

            local_color = tmp / _gcs_basename(color_uri)
            _cp_required(color_uri, local_color, what="download color video")

            color_df = _color_timestamps(local_color, start_ms, meta)

            out = tmp / f"{name}_color_timestamps.csv"
            color_df.to_csv(out, index=False)

            _cp_required(out, color_ts_uri, what="upload color timestamps")

        radar_n = len(radar_df) if radar_df is not None else "existing"
        color_n = len(color_df) if color_df is not None else "existing"

        prefix = "[adc+ok]" if adc_ran else "[ok]"
        return {
            "status": "ok",
            "name": name,
            "adc": adc_ran,
            "radar_n": radar_n,
            "color_n": color_n,
            "message": f"{prefix} {name}: radar_ts={radar_n}, color_ts={color_n}",
        }

    except Exception as exc:
        return {
            "status": "failed",
            "name": name,
            "adc": False,
            "radar_n": None,
            "color_n": None,
            "message": f"[FAIL] {name}: {exc}",
            "error": str(exc),
        }

    finally:
        shutil.rmtree(tmp, ignore_errors=True)


# ---- 5. execute in parallel + summary ----------------------------------------

items = sorted(session_roots.items())

processed = []
skipped = []
failed = []
adc_generated = []

try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(_process_one_session, item) for item in items]

        for i, fut in enumerate(concurrent.futures.as_completed(futures), start=1):
            res = fut.result()

            if PRINT_EACH_SESSION:
                print(f"[{i:>3}/{len(futures)}] {res['message']}", flush=True)

            if res["status"] == "ok":
                processed.append(res["name"])
                if res["adc"]:
                    adc_generated.append(res["name"])

            elif res["status"] == "skipped":
                skipped.append(res["name"])

            elif res["status"] == "failed":
                failed.append((res["name"], res.get("error", "unknown error")))

finally:
    shutil.rmtree(_shared_cfg_tmp, ignore_errors=True)


print(f"\n{'=' * 60}")
print(
    f"adc_generated={len(adc_generated)}  "
    f"processed={len(processed)}  "
    f"skipped={len(skipped)}  "
    f"failed={len(failed)}"
)

if failed:
    print("\nfailed sessions:")
    for n, e in failed:
        print(f"  {n}: {e}")